In [ ]:
# Cài đặt thư viện cần thiết
!pip -q install -U "pymilvus[milvus-lite]" "rapidfuzz>=3.0"
!pip -q install --no-deps timm==0.4.12
!pip -q install --no-deps torchscale sentencepiece ftfy
!pip -q install -U open_clip_torch regex tqdm huggingface_hub
!pip -q install -U --no-deps "sentence-transformers[image]>=5.4.0"
!pip -q install -U --no-deps "transformers>=4.46.3" "accelerate" "einops" "safetensors"
!pip -q install tensorboardX torchmetrics fairscale
!pip -q install fastapi uvicorn pyngrok nest_asyncio python-multipart pydantic faiss-cpu rank_bm25 orjson underthesea
!pip install transformers -U
!pip install --upgrade transformers sentence-transformers

import os
from pathlib import Path

# 1. Clone BEiT-3
REPO_DIR = Path("/kaggle/working/unilm")
if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/microsoft/unilm.git /kaggle/working/unilm

CKPT_DIR = Path("/kaggle/working/beit3_ckpt")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
if not (CKPT_DIR / "beit3_base_patch16_384_coco_retrieval.pth").exists():
    !wget -q -nc -O /kaggle/working/beit3_ckpt/beit3_base_patch16_384_coco_retrieval.pth \
    https://github.com/addf400/files/releases/download/beit3/beit3_base_patch16_384_coco_retrieval.pth
if not (CKPT_DIR / "beit3.spm").exists():
    !wget -q -nc -O /kaggle/working/beit3_ckpt/beit3.spm \
    https://github.com/addf400/files/releases/download/beit3/beit3.spm

# 2. Clone Perception
PE_REPO_DIR = Path("/kaggle/working/perception_models")
if not PE_REPO_DIR.exists():
    !git clone --depth 1 https://github.com/facebookresearch/perception_models.git /kaggle/working/perception_models
!pip -q install -e /kaggle/working/perception_models --no-deps

print("Đã chuẩn bị xong môi trường!")

In [ ]:
import sys
from pathlib import Path

# 1. Khôi phục lại file utils.py gốc từ git để lấy lại hàm ClipLoss và các hàm khác
!cd /kaggle/working/unilm && git checkout beit3/utils.py

# 2. Patch file an toàn (Giống hệt cách đã hoạt động trong file aic-emd-beit-3 của bạn)
utils_path = Path("/kaggle/working/unilm/beit3/utils.py")
if utils_path.exists():
    text = utils_path.read_text(encoding="utf-8")
    
    # Fix torch._six
    text = text.replace(
        "from torch._six import inf", 
        "from math import inf"
    )
    
    # Fix tensorboardX 
    text = text.replace(
        "from tensorboardX import SummaryWriter",
        """try:
    from tensorboardX import SummaryWriter
except Exception:
    class SummaryWriter:
        def __init__(self, *args, **kwargs):
            pass
        def add_scalar(self, *args, **kwargs):
            pass
        def add_scalars(self, *args, **kwargs):
            pass
        def add_text(self, *args, **kwargs):
            pass
        def close(self):
            pass"""
    )
    utils_path.write_text(text, encoding="utf-8")

# 3. Dọn dẹp cache module để Python buộc phải đọc lại file utils.py vừa được sửa
for module_name in list(sys.modules.keys()):
    if module_name in ["utils", "modeling_finetune", "modeling_utils"]:
        del sys.modules[module_name]

print("Đã khôi phục và patch file utils.py thành công! Hàm ClipLoss đã quay trở lại.")

In [ ]:
import sys
import gc
import math
import heapq
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

# IMPORT TRƯỚC ĐỂ TRÁNH XUNG ĐỘT (NAMESPACE COLLISION)
from sentence_transformers import SentenceTransformer
import datasets 
import open_clip

# Thêm đường dẫn mã nguồn vào sys.path
BEIT3_DIR = Path("/kaggle/working/unilm/beit3")
PE_REPO_DIR = Path("/kaggle/working/perception_models")

if str(BEIT3_DIR) not in sys.path:
    sys.path.append(str(BEIT3_DIR))
if str(PE_REPO_DIR) not in sys.path:
    sys.path.append(str(PE_REPO_DIR))

# Tiếp tục import của BEiT-3 và Perception
import timm
import modeling_finetune
import sentencepiece as spm
import core.vision_encoder.pe as pe
import core.vision_encoder.transforms as pe_transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Sử dụng thiết bị: {device}\n")

# ================================
# 1. LOAD BGE-VL-LARGE
# ================================
print("Loading BGE-VL-Large...")
bge_model = SentenceTransformer("BAAI/BGE-VL-large", trust_remote_code=True, device=device)

# ================================
# 2. LOAD METACLIP
# ================================
print("Loading MetaCLIP...")
metaclip_model, _, metaclip_preprocess = open_clip.create_model_and_transforms(
    "ViT-L-14-quickgelu", pretrained="metaclip_fullcc", device=device
)
metaclip_tokenizer = open_clip.get_tokenizer("ViT-L-14-quickgelu")
metaclip_model.eval()

# ================================
# 3. LOAD BEIT-3
# ================================
print("Loading BEiT-3...")
beit3_model = timm.create_model("beit3_base_patch16_384_retrieval", pretrained=False)
ckpt = torch.load("/kaggle/working/beit3_ckpt/beit3_base_patch16_384_coco_retrieval.pth", map_location="cpu", weights_only=False)
new_state_dict = {k[len("module."):]: v for k, v in ckpt["model"].items() if k.startswith("module.")}
beit3_model.load_state_dict(new_state_dict, strict=False)
beit3_model = beit3_model.to(device).eval()

sp = spm.SentencePieceProcessor()
sp.load("/kaggle/working/beit3_ckpt/beit3.spm")
BOS_ID, PAD_ID, EOS_ID = max(sp.piece_to_id("<s>"), 0), max(sp.piece_to_id("<pad>"), 1), max(sp.piece_to_id("</s>"), 2)

# ================================
# 4. LOAD PERCEPTION
# ================================
print("Loading Perception...")
perception_model = pe.CLIP.from_config("PE-Core-L14-336", pretrained=True).to(device).eval()
perception_tokenizer = pe_transforms.get_text_tokenizer(perception_model.context_length)

print("\nĐã load thành công toàn bộ 4 models!")

In [ ]:
@torch.no_grad()
def encode_bge_text(query):
    q_emb = bge_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False
    )[0]
    return q_emb.astype("float32")

@torch.no_grad()
def encode_metaclip_text(query):
    tokens = metaclip_tokenizer([query]).to(device)
    features = F.normalize(metaclip_model.encode_text(tokens), dim=-1)
    return features[0].detach().cpu().numpy().astype("float32")

@torch.no_grad()
def encode_beit3_text(query, max_len=64):
    ids = sp.encode(query, out_type=int)
    ids = [BOS_ID] + ids[:max_len - 2] + [EOS_ID]
    ids = ids + [PAD_ID] * (max_len - len(ids))
    
    input_ids = torch.tensor([ids], dtype=torch.long).to(device)
    output = beit3_model(
        image=None, text_description=input_ids, 
        padding_mask=input_ids.eq(PAD_ID).to(device), only_infer=True
    )
    text_emb = output[1] if isinstance(output, tuple) else output.get("language_cls", output)
    return F.normalize(text_emb, dim=-1)[0].detach().cpu().numpy().astype("float32")

@torch.inference_mode()
def encode_perception_text(query):
    text = perception_tokenizer([query]).to(device)
    if hasattr(perception_model, "encode_text"):
        features = perception_model.encode_text(text)
        if isinstance(features, tuple): features = features[0]
    else:
        h = w = perception_model.image_size if isinstance(perception_model.image_size, int) else perception_model.image_size[0]
        _, features, _ = perception_model(torch.zeros((1, 3, h, w), device=device), text)
        
    return F.normalize(features, dim=-1)[0].detach().cpu().numpy().astype("float32")

print("Đã định nghĩa xong các hàm Encode Text!")

In [ ]:
# ============================================================
# MILVUS LITE DIRECT EMBEDDED VECTOR SEARCH — NO gRPC
#
# IMPORTANT:
#   The previous version used:
#       pymilvus.MilvusClient(uri="/path/to/local.db")
#   For a local Milvus Lite database this goes through the local
#   Milvus-compatible gRPC adapter (127.0.0.1). A long-lived notebook
#   client can therefore hit gRPC keepalive GOAWAY / too_many_pings.
#
#   This cell opens the SAME Milvus Lite data directory with the
#   direct Python engine:
#       milvus_lite.MilvusLite
#   so visual vector search does NOT use localhost gRPC at all.
# ============================================================

import gc
import json
import math
import os
import re
import shutil
import threading
import time
from pathlib import Path
from typing import Any, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from milvus_lite import MilvusLite
except Exception as exc:
    raise RuntimeError(
        "Cannot import milvus_lite.MilvusLite. "
        "Run Cell 0 again so pymilvus[milvus-lite] is upgraded."
    ) from exc


# ------------------------------------------------------------
# 0. CLOSE AN OLD LOCAL pymilvus CLIENT IF THIS CELL IS RERUN
# ------------------------------------------------------------

_old_milvus_client = globals().get("_MILVUS_CLIENT")
if _old_milvus_client is not None:
    try:
        _old_milvus_client.close()
        print("[MILVUS] Closed old pymilvus local gRPC client.")
    except Exception:
        pass

# Remove the old handle so later cells cannot accidentally use it.
_MILVUS_CLIENT = None
gc.collect()


# ------------------------------------------------------------
# 1. DATABASE CONFIG
# ------------------------------------------------------------

MILVUS_DB_SOURCE = Path(
    os.getenv(
        "AIC_MILVUS_DB_PATH",
        "/kaggle/input/notebooks/thnhtrnhvn/aic-emd-db/"
        "aic_embeddings_milvus.db",
    )
)

MILVUS_DB_WORKING = Path(
    os.getenv(
        "AIC_MILVUS_WORKING_PATH",
        "/kaggle/working/aic_embeddings_milvus.db",
    )
)

# auto: copy only when source filesystem is read-only
# 1/true: always copy to working
# 0/false: open source directly
MILVUS_COPY_MODE = os.getenv(
    "AIC_MILVUS_COPY_TO_WORKING",
    "auto",
).strip().lower()

MILVUS_VERBOSE_TIMING = os.getenv(
    "AIC_MILVUS_VERBOSE_TIMING",
    "1",
).strip().lower() not in {"0", "false", "no"}

MILVUS_COLLECTION_BY_PREFIX = {
    "bge_image_emb_chunk": (
        "embeddings_bge_vl_large__bge_image_emb__d768"
    ),
    "metaclip_image_emb_chunk": (
        "embeddings_metaclip_openclip__metaclip_image_emb__d768"
    ),
    "beit3_image_emb_chunk": (
        "embeddings_beit3_base_coco_retrieval__beit3_image_emb__d768"
    ),
    "perception_image_emb_chunk": (
        "embeddings_perception_pe_core_l14_336__"
        "perception_image_emb__d1024"
    ),
}

# Direct embedded-engine singletons.
_MILVUS_DB: Optional[MilvusLite] = None
_MILVUS_DB_RUNTIME_PATH: Optional[Path] = None
_MILVUS_COLLECTIONS: set[str] = set()
_MILVUS_COLLECTION_HANDLES: dict[str, Any] = {}
_MILVUS_LOADED_COLLECTIONS: set[str] = set()

# Protect only handle lifecycle. The expensive read-only search calls are
# intentionally not locked so different collections can overlap.
_MILVUS_HANDLE_LOCK = threading.RLock()

MILVUS_PRELOAD_COLLECTIONS = os.getenv(
    "AIC_MILVUS_PRELOAD_COLLECTIONS",
    "1",
).strip().lower() not in {"0", "false", "no"}


def _is_read_only_filesystem(path: Path) -> bool:
    """Best-effort read-only mount detection without modifying the DB."""
    try:
        readonly_flag = getattr(os, "ST_RDONLY", 1)
        return bool(os.statvfs(path).f_flag & readonly_flag)
    except Exception:
        return False


def prepare_milvus_db_path() -> Path:
    """
    Return a writable Milvus Lite data directory.

    Kaggle /kaggle/input is read-only, while Milvus Lite uses a lock and
    may perform local recovery. In that case copy once to /kaggle/working.
    On a rented GPU with writable storage, open the source directly.
    """
    source = MILVUS_DB_SOURCE

    if not source.exists():
        raise FileNotFoundError(
            "Milvus database not found at:\n"
            f"  {source}\n"
            "Add aic-emd-db first or set AIC_MILVUS_DB_PATH."
        )

    if MILVUS_COPY_MODE in {"1", "true", "yes", "always"}:
        should_copy = True
    elif MILVUS_COPY_MODE in {"0", "false", "no", "never"}:
        should_copy = False
    else:
        should_copy = _is_read_only_filesystem(source)

    if not should_copy:
        return source

    target = MILVUS_DB_WORKING
    stamp = target.parent / f".{target.name}.copy_complete"

    source_signature = json.dumps(
        {
            "source": str(source),
            "is_dir": source.is_dir(),
        },
        sort_keys=True,
    )

    if target.exists() and stamp.is_file():
        try:
            if stamp.read_text(encoding="utf-8") == source_signature:
                print(f"[MILVUS] Using existing writable copy: {target}")
                return target
        except Exception:
            pass

    print("[MILVUS] Kaggle input is read-only; copying DB once:")
    print(f"          {source}")
    print(f"       -> {target}")

    target.parent.mkdir(parents=True, exist_ok=True)

    if target.exists():
        if target.is_dir():
            shutil.rmtree(target)
        else:
            target.unlink()

    started = time.perf_counter()
    if source.is_dir():
        shutil.copytree(source, target)
    else:
        shutil.copy2(source, target)

    stamp.write_text(source_signature, encoding="utf-8")

    print(
        f"[MILVUS] DB copy finished in "
        f"{time.perf_counter() - started:.2f}s"
    )
    return target


def get_milvus_db() -> MilvusLite:
    """
    Open Milvus Lite through the direct embedded Python engine.

    No MilvusClient, no localhost socket, no gRPC keepalive.
    """
    global _MILVUS_DB
    global _MILVUS_DB_RUNTIME_PATH
    global _MILVUS_COLLECTIONS

    if _MILVUS_DB is not None:
        return _MILVUS_DB

    runtime_path = prepare_milvus_db_path()
    started = time.perf_counter()

    _MILVUS_DB = MilvusLite(str(runtime_path))
    _MILVUS_DB_RUNTIME_PATH = runtime_path
    _MILVUS_COLLECTIONS = set(_MILVUS_DB.list_collections())

    required = set(MILVUS_COLLECTION_BY_PREFIX.values())
    missing = sorted(required - _MILVUS_COLLECTIONS)

    if missing:
        try:
            _MILVUS_DB.close()
        except Exception:
            pass
        _MILVUS_DB = None
        raise RuntimeError(
            "Milvus DB is missing required collections:\n  - "
            + "\n  - ".join(missing)
            + "\nAvailable collections:\n  - "
            + "\n  - ".join(sorted(_MILVUS_COLLECTIONS))
        )

    print(
        f"[MILVUS DIRECT] Opened in "
        f"{time.perf_counter() - started:.3f}s"
    )
    print(f"[MILVUS DIRECT] DB: {runtime_path}")
    print(f"[MILVUS DIRECT] Collections: {len(_MILVUS_COLLECTIONS)}")
    print("[MILVUS DIRECT] Transport: embedded Python engine (NO gRPC)")

    return _MILVUS_DB


# Backward-compatible name used nowhere in the new OCR adapter.
# It intentionally returns the direct engine, not pymilvus.MilvusClient.
def get_milvus_client():
    return get_milvus_db()


def get_milvus_collection(collection_name: str):
    """
    Return a cached direct-engine collection handle.

    Creation/load is protected because visual retrieval may use several
    worker threads. collection.search() itself is not locked.
    """
    db = get_milvus_db()

    if collection_name not in _MILVUS_COLLECTIONS:
        raise KeyError(f"Milvus collection not found: {collection_name}")

    with _MILVUS_HANDLE_LOCK:
        collection = _MILVUS_COLLECTION_HANDLES.get(collection_name)
        if collection is None:
            collection = db.get_collection(collection_name)
            _MILVUS_COLLECTION_HANDLES[collection_name] = collection

        if collection_name not in _MILVUS_LOADED_COLLECTIONS:
            try:
                collection.load()
            except Exception:
                # Some direct-engine lifecycle states are already queryable.
                pass
            _MILVUS_LOADED_COLLECTIONS.add(collection_name)

    return collection


def preload_milvus_collections() -> dict[str, float]:
    """Load all collection handles once; no vector scan is executed."""
    timings: dict[str, float] = {}

    for collection_name in MILVUS_COLLECTION_BY_PREFIX.values():
        started = time.perf_counter()
        get_milvus_collection(collection_name)
        timings[collection_name] = (
            time.perf_counter() - started
        ) * 1000.0

    if MILVUS_VERBOSE_TIMING:
        print(
            f"[MILVUS PRELOAD] {len(timings)} collection(s) ready "
            f"in {sum(timings.values()):.1f} ms"
        )

    return timings


def close_milvus_db():
    """Optional cleanup helper when restarting/replacing the DB."""
    global _MILVUS_DB
    global _MILVUS_DB_RUNTIME_PATH
    global _MILVUS_COLLECTIONS

    if _MILVUS_DB is not None:
        try:
            _MILVUS_DB.close()
        except Exception:
            pass

    _MILVUS_DB = None
    _MILVUS_DB_RUNTIME_PATH = None
    _MILVUS_COLLECTIONS = set()
    _MILVUS_COLLECTION_HANDLES.clear()
    _MILVUS_LOADED_COLLECTIONS.clear()
    gc.collect()


def get_milvus_status() -> dict[str, Any]:
    db = get_milvus_db()
    return {
        "db_source": str(MILVUS_DB_SOURCE),
        "db_runtime": str(_MILVUS_DB_RUNTIME_PATH),
        "transport": "embedded_python_no_grpc",
        "collections": sorted(db.list_collections()),
        "visual_collection_map": dict(MILVUS_COLLECTION_BY_PREFIX),
    }


def _milvus_hit_to_dict(hit: Any) -> dict[str, Any]:
    if isinstance(hit, dict):
        return dict(hit)
    if hasattr(hit, "to_dict"):
        value = hit.to_dict()
        if isinstance(value, dict):
            return value
    try:
        return dict(hit)
    except Exception:
        return {"raw": hit}


def _decode_metadata(value: Any) -> dict[str, Any]:
    if isinstance(value, dict):
        return dict(value)

    if isinstance(value, (bytes, bytearray, memoryview)):
        try:
            value = bytes(value).decode("utf-8")
        except Exception:
            return {}

    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            return parsed if isinstance(parsed, dict) else {}
        except Exception:
            return {}

    return {}


def search_embeddings(
    q_emb,
    emb_dirs=None,
    file_prefix: str | None = None,
    top_k: int = 100,
):
    """
    Direct Milvus Lite vector search.

    Signature remains compatible with the old .npy search function.
    `emb_dirs` is ignored for vector retrieval.
    """
    del emb_dirs

    if not file_prefix:
        raise ValueError("file_prefix is required for Milvus routing.")

    collection_name = MILVUS_COLLECTION_BY_PREFIX.get(file_prefix)
    if not collection_name:
        raise KeyError(
            f"No Milvus collection mapping for prefix: {file_prefix}"
        )

    query_vector = np.asarray(q_emb, dtype=np.float32).reshape(-1)
    if not np.isfinite(query_vector).all():
        raise ValueError("Query embedding contains NaN/Inf.")

    collection = get_milvus_collection(collection_name)
    started = time.perf_counter()

    # Direct embedded-engine API. This bypasses the local gRPC adapter.
    raw_results = collection.search(
        [query_vector.tolist()],
        top_k=max(1, int(top_k)),
        metric_type="COSINE",
        anns_field="embedding",
        output_fields=[
            "owner",
            "source_dataset",
            "source_root",
            "chunk_file",
            "row_in_chunk",
            "metadata",
        ],
    )

    elapsed = time.perf_counter() - started
    hits = raw_results[0] if raw_results else []

    rows: list[dict[str, Any]] = []

    for rank, raw_hit in enumerate(hits, start=1):
        hit = _milvus_hit_to_dict(raw_hit)
        entity = hit.get("entity")
        if not isinstance(entity, dict):
            entity = {}

        metadata = _decode_metadata(
            entity.get("metadata", hit.get("metadata"))
        )

        row = dict(metadata)

        for field in (
            "owner",
            "source_dataset",
            "source_root",
            "chunk_file",
            "row_in_chunk",
        ):
            value = entity.get(field, hit.get(field))
            if value is not None:
                row.setdefault(field, value)

        distance = hit.get("distance", hit.get("score"))
        if distance is None and "distance" in entity:
            distance = entity["distance"]

        row["rank"] = rank
        row["score"] = float(distance) if distance is not None else 0.0
        row["milvus_id"] = hit.get("id", entity.get("id"))
        row["milvus_collection"] = collection_name

        if row.get("image_path") is not None:
            row["image_path"] = str(row["image_path"]).replace(
                "https://www.kaggle.com/kaggle/input/",
                "/kaggle/input/",
                1,
            )

        rows.append(row)

    if MILVUS_VERBOSE_TIMING:
        print(
            f"[MILVUS DIRECT SEARCH] {file_prefix}: "
            f"{len(rows)} hits in {elapsed:.3f}s"
        )

    dataframe = pd.DataFrame(rows)

    if not dataframe.empty and "image_path" not in dataframe.columns:
        raise RuntimeError(
            f"Collection {collection_name} returned metadata without "
            "image_path."
        )

    # Keep timing in DataFrame.attrs so RRF schema stays unchanged.
    dataframe.attrs["milvus_search_seconds"] = float(elapsed)
    dataframe.attrs["milvus_collection"] = collection_name
    dataframe.attrs["milvus_prefix"] = file_prefix
    return dataframe


def compute_rrf(list_of_dfs, k=60):
    rrf_scores, metadata_dict = {}, {}
    for df in list_of_dfs:
        if df.empty: continue
        for _, row in df.iterrows():
            path, rank = row['image_path'], row['rank']
            if path not in rrf_scores:
                rrf_scores[path] = 0.0
                metadata_dict[path] = row.to_dict()
            rrf_scores[path] += 1.0 / (k + rank)

    final_df = pd.DataFrame([ {**metadata_dict[p], 'rrf_score': s} for p, s in rrf_scores.items() ])
    if final_df.empty: return final_df
    
    final_df = final_df.sort_values('rrf_score', ascending=False).reset_index(drop=True)
    final_df['final_rank'] = final_df.index + 1
    final_df['score'] = final_df['rrf_score'] # Đồng bộ với hàm lọc trùng
    return final_df

FRAME_PATTERN = re.compile(r"(?P<video_id>L\d{2}_V\d{3})_t(?P<timestamp>\d+(?:\.\d+)?)_f(?P<frame_idx>\d+)_s(?P<shot_id>\d+)")

def fix_results_metadata(results):
    if results.empty: return results
    fixed = results.copy()
    corrected = []
    for _, row in fixed.iterrows():
        # Đảm bảo đường dẫn map đúng vào input của Kaggle hiện tại
        local_path = str(row["image_path"]).replace("https://www.kaggle.com/kaggle/input/", "/kaggle/input/")
        fname = Path(local_path).name
        match = FRAME_PATTERN.search(fname)
        
        corrected.append({
            "local_image_path": local_path, "filename": fname,
            "video_id_fixed": match.group("video_id") if match else row.get("video_id"),
            "video_title": match.group("video_id") if match else row.get("video_id"),
            "timestamp_fixed": float(match.group("timestamp")) if match else row.get("timestamp"),
            "time_fixed": f"{int(float(match.group('timestamp')) // 60):02d}:{float(match.group('timestamp')) % 60:04.1f}" if match else "",
            "frame_idx_fixed": int(match.group("frame_idx")) if match else row.get("frame_idx"),
            "shot_id_fixed": int(match.group("shot_id")) if match else row.get("shot_id"),
            "image_exists": Path(local_path).exists(),
        })
    for col in corrected[0].keys(): fixed[col] = [r[col] for r in corrected]
    return fixed

def diversify_results(results, min_gap_seconds=5.0, max_per_video=5, max_results=20):
    if results.empty: return results
    selected, selected_times = [], {}
    results = results.sort_values("score", ascending=False).reset_index(drop=True)
    
    for _, row in results.iterrows():
        vid, ts = row["video_id_fixed"], row["timestamp_fixed"]
        if vid not in selected_times: selected_times[vid] = []
        if len(selected_times[vid]) >= max_per_video: continue
            
        if pd.notna(ts) and any(abs(float(ts) - old_t) < min_gap_seconds for old_t in selected_times[vid]): continue
        if pd.notna(ts): selected_times[vid].append(float(ts))
            
        selected.append(row)
        if len(selected) >= max_results: break

    out = pd.DataFrame(selected).reset_index(drop=True)
    out["final_rank"] = range(1, len(out) + 1)
    return out

def visualize_results(results, n=12, cols=4):
    if results.empty: return print("Lỗi: DataFrame rỗng, không có gì để hiển thị.")
    valid_rows = [row for _, row in results.iterrows() if Path(row["local_image_path"]).exists()][:n]
    if not valid_rows: return print("Lỗi: Không tìm thấy file ảnh gốc trên ổ cứng.")
        
    rows = math.ceil(len(valid_rows) / cols)
    plt.figure(figsize=(cols * 4.5, rows * 3.6))
    for i, row in enumerate(valid_rows):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(Image.open(row["local_image_path"]).convert("RGB"))
        plt.axis("off")
        title = f"Rank {int(row['final_rank'])} | RRF={row['rrf_score']:.3f}\n{row['video_title']} | {row['time_fixed']}"
        plt.title(title, fontsize=9)
    plt.tight_layout()
    plt.show()

if MILVUS_PRELOAD_COLLECTIONS:
    try:
        preload_milvus_collections()
    except Exception as error:
        print("[MILVUS PRELOAD WARNING]", repr(error))

print("Đã khởi tạo xong các hàm thuật toán và hiển thị!")

In [ ]:
# ============================================================
# LEGACY EMBEDDING/METADATA DIRECTORIES
# Vector search no longer scans these .npy files. They are kept only
# because OCR/ASR frame-path adapters still reuse metadata_chunk_*.csv.
# ============================================================

BGE_DIR = ["/kaggle/input/notebooks/thnhtrnhvn/aic-emd/embeddings_bge_vl_large",
           "/kaggle/input/notebooks/lamnguynx/aic-emd/embeddings_bge_vl_large",
           "/kaggle/input/notebooks/callmettr/aic-emd-bge-l21/embeddings_bge_vl_large"
          ]

METACLIP_DIR = ["/kaggle/input/notebooks/thnhtrnhvn/aic-emd-metaclip2/embeddings_metaclip_openclip",
                "/kaggle/input/notebooks/lamnguynx/aic-emd-metaclip2/embeddings_metaclip_openclip",
                "/kaggle/input/notebooks/l4mznguyenx/aic-emd-metaclip2-l21/embeddings_metaclip_openclip"
               ]

BEIT3_DIR = ["/kaggle/input/notebooks/thnhtrnhvn/aic-emd-beit-3/embeddings_beit3_base_coco_retrieval",
             "/kaggle/input/notebooks/lamnguynx/aic-emd-beit-3/embeddings_beit3_base_coco_retrieval",
             "/kaggle/input/notebooks/callmettr/aic-emd-beit-3-l21/embeddings_beit3_base_coco_retrieval"
            ]

PERCEPTION_DIR = ["/kaggle/input/notebooks/lamnguynx/aic-2026-emd-perception/embeddings_perception_pe_core_l14_336",
                  "/kaggle/input/notebooks/thnhtrnhvn/aic-2026-emd-perception/embeddings_perception_pe_core_l14_336",
                  "/kaggle/input/notebooks/l4mznguyenx/aic-2026-emd-perception-l21/embeddings_perception_pe_core_l14_336"
                 ]

# Connect once now so a missing DB/collection fails early instead of at the
# first API request.
print("[MILVUS] Validating visual database before search...")
_milvus_status = get_milvus_status()
print("[MILVUS] Visual DB ready:", _milvus_status["db_runtime"])

# Bật True chỉ khi muốn test thủ công trước khi mở FastAPI.
RUN_INITIAL_SMOKE_TEST = False

if RUN_INITIAL_SMOKE_TEST:
    query = (
        "three people, two women and one man, sitting together "
        "and playing a round hollow metal instrument; a person "
        "in a white shirt sits between two people in black shirts; "
        "a multi-tiered bookshelf filled with colorful books "
        "appears in the background"
    )
    TOP_K_SEARCH = 100

    print(f"Xử lý Query: {query!r}")

    model_results = {
        "bge": search_embeddings(
            encode_bge_text(query),
            BGE_DIR,
            "bge_image_emb_chunk",
            top_k=TOP_K_SEARCH,
        ),
        "metaclip": search_embeddings(
            encode_metaclip_text(query),
            METACLIP_DIR,
            "metaclip_image_emb_chunk",
            top_k=TOP_K_SEARCH,
        ),
        "beit3": search_embeddings(
            encode_beit3_text(query),
            BEIT3_DIR,
            "beit3_image_emb_chunk",
            top_k=TOP_K_SEARCH,
        ),
        "perception": search_embeddings(
            encode_perception_text(query),
            PERCEPTION_DIR,
            "perception_image_emb_chunk",
            top_k=TOP_K_SEARCH,
        ),
    }

    rrf_results = compute_rrf(
        list(model_results.values()),
        k=60,
    )

    if not rrf_results.empty:
        fixed_results = fix_results_metadata(
            rrf_results
        )
        final_results = diversify_results(
            fixed_results,
            min_gap_seconds=5.0,
            max_per_video=5,
            max_results=20,
        )
        visualize_results(final_results, n=12)
    else:
        print("Không có kết quả. Kiểm tra lại embedding paths.")
else:
    print("Embedding directories configured; smoke test skipped.")

In [ ]:
# ============================================================
# QUERY RECONSTRUCTION REMOVED
# ============================================================
# The previous LLM-based query reconstruction/rewrite feature has been removed.
# Search now uses the visual/OCR/ASR queries exactly as provided by the client.


In [ ]:
# ============================================================
# OCR SEARCH V2 — USE PREBUILT SQLITE DATASET
# Replaces the old JSONL -> SQLite builder.
# Source logic: ocr-search-test.ipynb
# Run BEFORE Multimodal Retrieval.
# ============================================================

import sys

# rapidfuzz is installed once in Cell 0; do not mutate packages after DB open.

import importlib.util
import json
import os
import re
import sqlite3
import time
import unicodedata
from pathlib import Path
from typing import Any, Optional

import pandas as pd


# ------------------------------------------------------------
# 1. FIXED INPUT PATH
# Dataset contains:
#   - ocr_search.sqlite
#   - ocr_search_manifest.json
# ------------------------------------------------------------

OCR_SEARCH_DATA_ROOT = Path(
    "/kaggle/input/datasets/khanq1504/"
    "final-ocr-search-data-aic-2025"
)

DATABASE_PATH = OCR_SEARCH_DATA_ROOT / "ocr_search.sqlite"
MANIFEST_PATH = OCR_SEARCH_DATA_ROOT / "ocr_search_manifest.json"

RUNTIME_ROOT = Path("/kaggle/working/aic_ocr_search_v2")
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)

OCR_DEFAULT_MODE = "balanced"
OCR_DEFAULT_TOP_K = 100
OCR_DEFAULT_CANDIDATE_LIMIT = 2000
OCR_DEFAULT_DEDUPLICATE_SECONDS = 2.4

OCR_VALID_MODES = {
    "precision",
    "balanced",
    "recall",
}

if not DATABASE_PATH.is_file():
    raise FileNotFoundError(
        "Không tìm thấy OCR database tại: "
        f"{DATABASE_PATH}. Hãy Add Input dataset "
        "Final_OCR_Search_Data_AIC_2025."
    )


# ------------------------------------------------------------
# 2. FAST STARTUP VALIDATION
# ------------------------------------------------------------
# IMPORTANT:
# PRAGMA quick_check is O(N) over the database and can take many minutes
# on a large SQLite file mounted under /kaggle/input. It is therefore
# disabled during normal notebook startup. Enable it only for explicit
# database diagnostics with AIC_OCR_FULL_VALIDATION=1.

OCR_FULL_STARTUP_VALIDATION = os.getenv(
    "AIC_OCR_FULL_VALIDATION",
    "0",
).strip().lower() in {"1", "true", "yes"}


def connect_ocr_search_db(readonly: bool = True) -> sqlite3.Connection:
    if readonly:
        # Kaggle input datasets are immutable for the running notebook.
        # immutable=1 avoids unnecessary file locking/change detection.
        connection = sqlite3.connect(
            f"file:{DATABASE_PATH}?mode=ro&immutable=1",
            uri=True,
            check_same_thread=False,
            timeout=5.0,
        )
    else:
        connection = sqlite3.connect(
            str(DATABASE_PATH),
            check_same_thread=False,
            timeout=5.0,
        )

    connection.row_factory = sqlite3.Row
    connection.execute("PRAGMA query_only = ON")
    connection.execute("PRAGMA temp_store = MEMORY")
    connection.execute("PRAGMA cache_size = -65536")
    return connection


# Manifest is cheap to read and may already contain useful counts.
if MANIFEST_PATH.is_file():
    OCR_SEARCH_MANIFEST = json.loads(
        MANIFEST_PATH.read_text(encoding="utf-8")
    )
else:
    OCR_SEARCH_MANIFEST = {}


def _manifest_int(*keys: str, default: int = -1) -> int:
    for key in keys:
        value = OCR_SEARCH_MANIFEST.get(key)
        try:
            if value is not None:
                return int(value)
        except (TypeError, ValueError):
            pass
    return int(default)


print("[OCR INIT] Opening SQLite in immutable read-only mode...")
_started_validation = time.perf_counter()

with connect_ocr_search_db(readonly=True) as connection:
    # Schema lookup is tiny and does not scan the FTS/data tables.
    ocr_tables = {
        row[0]
        for row in connection.execute(
            "SELECT name FROM sqlite_master "
            "WHERE type IN ('table', 'view')"
        )
    }

    required_tables = {
        "frames",
        "regions",
        "frames_word_fts",
        "metadata",
    }

    missing_tables = required_tables - ocr_tables
    if missing_tables:
        raise RuntimeError(
            "OCR database thiếu bảng bắt buộc: "
            f"{sorted(missing_tables)}"
        )

    # Touch one row only. This catches an unreadable/corrupt schema without
    # doing an O(N) integrity scan during every notebook startup.
    connection.execute("SELECT 1 FROM frames LIMIT 1").fetchone()
    connection.execute("SELECT 1 FROM regions LIMIT 1").fetchone()

    # Read the tiny metadata table once. Prefer stored counts if available.
    _ocr_db_metadata = {}
    for _row in connection.execute("SELECT key, value FROM metadata"):
        try:
            _ocr_db_metadata[str(_row["key"])] = json.loads(_row["value"])
        except Exception:
            _ocr_db_metadata[str(_row["key"])] = _row["value"]

    def _db_meta_int(*keys: str, default: int = -1) -> int:
        for key in keys:
            value = _ocr_db_metadata.get(key)
            try:
                if value is not None:
                    return int(value)
            except (TypeError, ValueError):
                pass
        return int(default)

    ocr_frame_count = _db_meta_int(
        "frame_count", "frames", "row_count",
        default=_manifest_int("frame_count", "frames", "row_count"),
    )
    ocr_region_count = _db_meta_int(
        "region_count", "regions",
        default=_manifest_int("region_count", "regions"),
    )

    if OCR_FULL_STARTUP_VALIDATION:
        print("[OCR INIT] Full validation enabled: running PRAGMA quick_check...")
        quick_check = connection.execute(
            "PRAGMA quick_check"
        ).fetchone()[0]

        # Exact counts are also full scans; only do them in diagnostic mode.
        ocr_frame_count = int(
            connection.execute("SELECT COUNT(*) FROM frames").fetchone()[0]
        )
        ocr_region_count = int(
            connection.execute("SELECT COUNT(*) FROM regions").fetchone()[0]
        )
    else:
        quick_check = "skipped (fast startup)"

_validation_elapsed = time.perf_counter() - _started_validation

print("OCR database:", DATABASE_PATH)
print("OCR validation:", quick_check)
print(
    "OCR frames:",
    f"{ocr_frame_count:,}" if ocr_frame_count >= 0 else "deferred",
)
print(
    "OCR regions:",
    f"{ocr_region_count:,}" if ocr_region_count >= 0 else "deferred",
)
print(f"[OCR INIT] Ready in {_validation_elapsed:.3f}s")


BASE_ENGINE_SOURCE = '\nfrom __future__ import annotations\n\nimport json\nimport re\nimport sqlite3\nimport unicodedata\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Any\n\nfrom rapidfuzz import fuzz\n\n\n_NON_WORD_RE = re.compile(\n    r"[^\\w%]+",\n    flags=re.UNICODE,\n)\n_SPACE_RE = re.compile(r"\\s+")\n_NUMBER_RE = re.compile(\n    r"\\d+(?:[.,:/-]\\d+)*%?"\n)\n\n\ndef normalize_unicode(value: Any) -> str:\n    text = "" if value is None else str(value)\n    text = unicodedata.normalize("NFC", text).lower()\n    text = text.replace("_", " ")\n    text = _NON_WORD_RE.sub(" ", text)\n    return _SPACE_RE.sub(" ", text).strip()\n\n\ndef remove_accents(value: Any) -> str:\n    text = normalize_unicode(value).replace("đ", "d")\n    text = "".join(\n        character\n        for character in unicodedata.normalize("NFD", text)\n        if unicodedata.category(character) != "Mn"\n    )\n    return unicodedata.normalize("NFC", text)\n\n\ndef extract_numbers(value: Any) -> list[str]:\n    return _NUMBER_RE.findall(normalize_unicode(value))\n\n\ndef quote_fts_token(token: str) -> str:\n    return \'"\' + token.replace(\'"\', \'""\') + \'"\'\n\n\ndef build_word_query(\n    text: str,\n    *,\n    column: str,\n    strict: bool,\n) -> str | None:\n    tokens = normalize_unicode(text).split()\n\n    if not tokens:\n        return None\n\n    operator = " AND " if strict else " OR "\n\n    return operator.join(\n        f"{column} : {quote_fts_token(token)}"\n        for token in tokens\n    )\n\n\ndef build_phrase_query(\n    text: str,\n    *,\n    column: str,\n) -> str | None:\n    normalized = normalize_unicode(text)\n\n    if len(normalized.replace(" ", "")) < 3:\n        return None\n\n    return (\n        f"{column} : "\n        f"{quote_fts_token(normalized)}"\n    )\n\n\ndef fuzzy_score(\n    query: str,\n    candidate: str,\n) -> float:\n    if not query or not candidate:\n        return 0.0\n\n    return max(\n        float(fuzz.partial_ratio(query, candidate)),\n        float(fuzz.token_set_ratio(query, candidate)),\n        float(fuzz.WRatio(query, candidate)),\n    )\n\n\ndef chunks(\n    values: list[int],\n    size: int = 800,\n):\n    for start in range(0, len(values), size):\n        yield values[start:start + size]\n\n\nclass _BaseOCRSearchEngine:\n    def __init__(\n        self,\n        database_path: str | Path,\n    ) -> None:\n        self.database_path = Path(database_path)\n\n        if not self.database_path.exists():\n            raise FileNotFoundError(\n                self.database_path\n            )\n\n        self.connection = sqlite3.connect(\n            f"file:{self.database_path}?mode=ro&immutable=1",\n            uri=True,\n            check_same_thread=False,\n            timeout=5.0,\n        )\n        self.connection.row_factory = sqlite3.Row\n        self.connection.execute(\n            "PRAGMA query_only = ON"\n        )\n        self.connection.execute(\n            "PRAGMA temp_store = MEMORY"\n        )\n        self.connection.execute(\n            "PRAGMA cache_size = -100000"\n        )\n\n        rows = self.connection.execute(\n            "SELECT key, value FROM metadata"\n        ).fetchall()\n        self.metadata = {\n            row["key"]: json.loads(row["value"])\n            for row in rows\n        }\n        self.has_trigram = bool(\n            self.metadata.get(\n                "has_trigram",\n                False,\n            )\n        )\n\n    def close(self) -> None:\n        self.connection.close()\n\n    def _run_fts(\n        self,\n        *,\n        table: str,\n        match_query: str | None,\n        limit: int,\n        weight: float,\n        scores: dict[int, float],\n    ) -> None:\n        if not match_query:\n            return\n\n        if table == "frames_word_fts":\n            bm25_expression = (\n                "bm25("\n                "frames_word_fts, "\n                "3.0, 1.0, 2.5, 0.8"\n                ")"\n            )\n        elif table == "frames_trigram_fts":\n            bm25_expression = (\n                "bm25("\n                "frames_trigram_fts, "\n                "3.0, 1.0, 2.5, 0.8"\n                ")"\n            )\n        else:\n            raise ValueError(table)\n\n        sql = f"""\n            SELECT\n                rowid AS frame_id,\n                {bm25_expression} AS rank_value\n            FROM {table}\n            WHERE {table} MATCH ?\n            ORDER BY rank_value\n            LIMIT ?\n        """\n\n        try:\n            rows = self.connection.execute(\n                sql,\n                (match_query, int(limit)),\n            ).fetchall()\n        except sqlite3.OperationalError:\n            return\n\n        for rank, row in enumerate(rows, start=1):\n            scores[int(row["frame_id"])] += (\n                weight / (60.0 + rank)\n            )\n\n    def _retrieve(\n        self,\n        query: str,\n        candidate_limit: int,\n    ) -> dict[int, float]:\n        query_norm = normalize_unicode(query)\n        query_ascii = remove_accents(query)\n        scores: dict[int, float] = defaultdict(float)\n\n        specs = [\n            (\n                "frames_word_fts",\n                build_word_query(\n                    query_norm,\n                    column="primary_norm",\n                    strict=True,\n                ),\n                1.35,\n            ),\n            (\n                "frames_word_fts",\n                build_word_query(\n                    query_norm,\n                    column="recall_norm",\n                    strict=True,\n                ),\n                0.95,\n            ),\n            (\n                "frames_word_fts",\n                build_word_query(\n                    query_ascii,\n                    column="primary_ascii",\n                    strict=True,\n                ),\n                1.20,\n            ),\n            (\n                "frames_word_fts",\n                build_word_query(\n                    query_ascii,\n                    column="recall_ascii",\n                    strict=True,\n                ),\n                0.85,\n            ),\n            (\n                "frames_word_fts",\n                build_word_query(\n                    query_norm,\n                    column="primary_norm",\n                    strict=False,\n                ),\n                0.55,\n            ),\n            (\n                "frames_word_fts",\n                build_word_query(\n                    query_ascii,\n                    column="primary_ascii",\n                    strict=False,\n                ),\n                0.50,\n            ),\n        ]\n\n        for table, fts_query, weight in specs:\n            self._run_fts(\n                table=table,\n                match_query=fts_query,\n                limit=candidate_limit,\n                weight=weight,\n                scores=scores,\n            )\n\n        if self.has_trigram:\n            trigram_specs = [\n                (\n                    build_phrase_query(\n                        query_norm,\n                        column="primary_norm",\n                    ),\n                    1.50,\n                ),\n                (\n                    build_phrase_query(\n                        query_norm,\n                        column="recall_norm",\n                    ),\n                    1.00,\n                ),\n                (\n                    build_phrase_query(\n                        query_ascii,\n                        column="primary_ascii",\n                    ),\n                    1.35,\n                ),\n                (\n                    build_phrase_query(\n                        query_ascii,\n                        column="recall_ascii",\n                    ),\n                    0.90,\n                ),\n            ]\n\n            for fts_query, weight in trigram_specs:\n                self._run_fts(\n                    table="frames_trigram_fts",\n                    match_query=fts_query,\n                    limit=candidate_limit,\n                    weight=weight,\n                    scores=scores,\n                )\n\n        if not scores and query_ascii:\n            pattern = f"%{query_ascii}%"\n            rows = self.connection.execute(\n                """\n                SELECT id\n                FROM frames\n                WHERE primary_ascii LIKE ?\n                   OR recall_ascii LIKE ?\n                LIMIT ?\n                """,\n                (\n                    pattern,\n                    pattern,\n                    candidate_limit,\n                ),\n            ).fetchall()\n\n            for rank, row in enumerate(rows, start=1):\n                scores[int(row["id"])] += (\n                    1.0 / (60.0 + rank)\n                )\n\n        return dict(scores)\n\n    def _fetch_frames(\n        self,\n        frame_ids: list[int],\n    ) -> dict[int, sqlite3.Row]:\n        result = {}\n\n        for part in chunks(frame_ids):\n            placeholders = ",".join(\n                "?" for _ in part\n            )\n            rows = self.connection.execute(\n                f"""\n                SELECT *\n                FROM frames\n                WHERE id IN ({placeholders})\n                """,\n                part,\n            ).fetchall()\n\n            for row in rows:\n                result[int(row["id"])] = row\n\n        return result\n\n    def _fetch_regions(\n        self,\n        frame_ids: list[int],\n    ) -> dict[int, list[sqlite3.Row]]:\n        result = defaultdict(list)\n\n        for part in chunks(frame_ids):\n            placeholders = ",".join(\n                "?" for _ in part\n            )\n            rows = self.connection.execute(\n                f"""\n                SELECT *\n                FROM regions\n                WHERE frame_id IN ({placeholders})\n                ORDER BY frame_id, region_order\n                """,\n                part,\n            ).fetchall()\n\n            for row in rows:\n                result[int(row["frame_id"])].append(\n                    row\n                )\n\n        return dict(result)\n\n    def search(\n        self,\n        query: str,\n        *,\n        top_k: int = 50,\n        candidate_limit: int = 1500,\n        collection_id: str | None = None,\n        video_id: str | None = None,\n        deduplicate_seconds: float = 2.4,\n    ) -> list[dict[str, Any]]:\n        query_norm = normalize_unicode(query)\n        query_ascii = remove_accents(query)\n\n        if not query_norm:\n            return []\n\n        lexical_scores = self._retrieve(\n            query,\n            candidate_limit,\n        )\n\n        if not lexical_scores:\n            return []\n\n        candidate_ids = [\n            frame_id\n            for frame_id, _ in sorted(\n                lexical_scores.items(),\n                key=lambda item: item[1],\n                reverse=True,\n            )[:candidate_limit]\n        ]\n\n        frames = self._fetch_frames(candidate_ids)\n\n        candidate_ids = [\n            frame_id\n            for frame_id in candidate_ids\n            if (\n                frame_id in frames\n                and (\n                    collection_id is None\n                    or frames[frame_id][\n                        "collection_id"\n                    ] == collection_id\n                )\n                and (\n                    video_id is None\n                    or frames[frame_id][\n                        "video_id"\n                    ] == video_id\n                )\n            )\n        ]\n\n        if not candidate_ids:\n            return []\n\n        regions = self._fetch_regions(candidate_ids)\n\n        max_lexical = max(\n            lexical_scores.get(frame_id, 0.0)\n            for frame_id in candidate_ids\n        )\n        max_lexical = max(max_lexical, 1e-12)\n\n        query_numbers = extract_numbers(query)\n        scored = []\n\n        for frame_id in candidate_ids:\n            row = frames[frame_id]\n\n            unicode_value = max(\n                fuzzy_score(\n                    query_norm,\n                    row["primary_norm"],\n                ),\n                0.92\n                * fuzzy_score(\n                    query_norm,\n                    row["recall_norm"],\n                ),\n            )\n            ascii_value = max(\n                fuzzy_score(\n                    query_ascii,\n                    row["primary_ascii"],\n                ),\n                0.92\n                * fuzzy_score(\n                    query_ascii,\n                    row["recall_ascii"],\n                ),\n            )\n\n            best_region = None\n            best_region_value = 0.0\n\n            for region in regions.get(\n                frame_id,\n                [],\n            ):\n                text = region["text"] or ""\n                region_value = max(\n                    fuzzy_score(\n                        query_norm,\n                        normalize_unicode(text),\n                    ),\n                    fuzzy_score(\n                        query_ascii,\n                        remove_accents(text),\n                    ),\n                )\n                region_value *= {\n                    "strong": 1.0,\n                    "weak": 0.96,\n                    "drop": 0.86,\n                }.get(\n                    region["quality"],\n                    0.90,\n                )\n\n                if region_value > best_region_value:\n                    best_region_value = region_value\n                    best_region = region\n\n            lexical_value = (\n                100.0\n                * lexical_scores.get(frame_id, 0.0)\n                / max_lexical\n            )\n\n            combined_ascii = (\n                row["primary_ascii"]\n                + " "\n                + row["recall_ascii"]\n            )\n\n            if query_numbers:\n                matched = sum(\n                    number in combined_ascii\n                    for number in query_numbers\n                )\n                number_value = (\n                    100.0\n                    * matched\n                    / len(query_numbers)\n                )\n            else:\n                number_value = None\n\n            score = (\n                0.42 * unicode_value\n                + 0.20 * ascii_value\n                + 0.23 * best_region_value\n                + 0.15 * lexical_value\n            )\n\n            if number_value is not None:\n                score = (\n                    0.90 * score\n                    + 0.10 * number_value\n                )\n\n            exact_bonus = 0.0\n\n            if query_norm in row["primary_norm"]:\n                exact_bonus = 8.0\n            elif query_norm in row["recall_norm"]:\n                exact_bonus = 5.0\n\n            if query_ascii in row["primary_ascii"]:\n                exact_bonus = max(\n                    exact_bonus,\n                    7.0,\n                )\n            elif query_ascii in row["recall_ascii"]:\n                exact_bonus = max(\n                    exact_bonus,\n                    4.0,\n                )\n\n            result = {\n                "modality": "ocr",\n                "collection_id": row[\n                    "collection_id"\n                ],\n                "dataset_id": row["dataset_id"],\n                "video_id": row["video_id"],\n                "timestamp_sec": row[\n                    "timestamp_sec"\n                ],\n                "frame_index": row[\n                    "frame_index"\n                ],\n                "shot_id": row["shot_id"],\n                "frame": row["frame"],\n                "frame_text_index": row[\n                    "frame_text_index"\n                ],\n                "frame_text_recall": row[\n                    "frame_text_recall"\n                ],\n                "score": round(\n                    min(\n                        100.0,\n                        score + exact_bonus,\n                    ),\n                    4,\n                ),\n                "score_breakdown": {\n                    "unicode": round(\n                        unicode_value,\n                        4,\n                    ),\n                    "ascii": round(\n                        ascii_value,\n                        4,\n                    ),\n                    "region": round(\n                        best_region_value,\n                        4,\n                    ),\n                    "lexical": round(\n                        lexical_value,\n                        4,\n                    ),\n                    "number": (\n                        round(number_value, 4)\n                        if number_value\n                        is not None\n                        else None\n                    ),\n                },\n            }\n\n            if best_region is not None:\n                result["matched_region"] = {\n                    "text": best_region["text"],\n                    "quality": best_region[\n                        "quality"\n                    ],\n                    "polygon": json.loads(\n                        best_region[\n                            "polygon_json"\n                        ]\n                    ),\n                    "det_conf": best_region[\n                        "det_conf"\n                    ],\n                    "rec_conf": best_region[\n                        "rec_conf"\n                    ],\n                    "rotation": best_region[\n                        "rotation"\n                    ],\n                }\n            else:\n                result["matched_region"] = None\n\n            scored.append(result)\n\n        scored.sort(\n            key=lambda item: item["score"],\n            reverse=True,\n        )\n\n        selected = []\n\n        for result in scored:\n            duplicate = False\n\n            for old in selected:\n                if (\n                    result["video_id"]\n                    != old["video_id"]\n                ):\n                    continue\n\n                first = result["timestamp_sec"]\n                second = old["timestamp_sec"]\n\n                if (\n                    first is not None\n                    and second is not None\n                    and abs(first - second)\n                    <= deduplicate_seconds\n                ):\n                    duplicate = True\n                    break\n\n            if not duplicate:\n                selected.append(result)\n\n            if len(selected) >= top_k:\n                break\n\n        for rank, result in enumerate(\n            selected,\n            start=1,\n        ):\n            result["rank"] = rank\n\n        return selected\n\n    def get_frame(\n        self,\n        frame_name: str,\n    ) -> dict[str, Any] | None:\n        frame = self.connection.execute(\n            """\n            SELECT *\n            FROM frames\n            WHERE frame = ?\n            """,\n            (frame_name,),\n        ).fetchone()\n\n        if frame is None:\n            return None\n\n        regions = self.connection.execute(\n            """\n            SELECT *\n            FROM regions\n            WHERE frame_id = ?\n            ORDER BY region_order\n            """,\n            (int(frame["id"]),),\n        ).fetchall()\n\n        return {\n            "frame": dict(frame),\n            "regions": [\n                {\n                    **dict(region),\n                    "polygon": json.loads(\n                        region["polygon_json"]\n                    ),\n                }\n                for region in regions\n            ],\n        }\n\n\n_ALNUM_RE = re.compile(\n    r"[^\\W_]",\n    flags=re.UNICODE,\n)\n\n\ndef textuality_score(\n    value: Any,\n) -> float:\n    raw = (\n        ""\n        if value is None\n        else str(value)\n    )\n    compact = "".join(\n        character\n        for character in raw\n        if not character.isspace()\n    )\n\n    if not compact:\n        return 0.0\n\n    alnum_count = sum(\n        character.isalnum()\n        for character in compact\n    )\n    ratio = (\n        alnum_count\n        / max(\n            1,\n            len(compact),\n        )\n    )\n\n    lowered = "".join(\n        character.lower()\n        for character in compact\n        if character.isalnum()\n    )\n\n    if not lowered:\n        return 0.0\n\n    unique_ratio = (\n        len(set(lowered))\n        / len(lowered)\n    )\n\n    if (\n        len(lowered) >= 5\n        and unique_ratio < 0.18\n    ):\n        ratio *= 0.45\n\n    return max(\n        0.0,\n        min(\n            1.0,\n            ratio,\n        ),\n    )\n\n\ndef token_coverage(\n    normalize_function,\n    query: str,\n    candidate: str,\n) -> tuple[float, float]:\n    query_tokens = [\n        token\n        for token in normalize_function(\n            query\n        ).split()\n        if token\n    ]\n    candidate_tokens = [\n        token\n        for token in normalize_function(\n            candidate\n        ).split()\n        if token\n    ]\n\n    if not query_tokens:\n        return 0.0, 0.0\n\n    candidate_set = set(\n        candidate_tokens\n    )\n    exact_hits = sum(\n        token in candidate_set\n        for token in query_tokens\n    )\n    exact_coverage = (\n        exact_hits\n        / len(query_tokens)\n    )\n\n    if not candidate_tokens:\n        return exact_coverage, 0.0\n\n    approximate_hits = 0\n\n    for query_token in query_tokens:\n        best = max(\n            (\n                float(\n                    fuzz.ratio(\n                        query_token,\n                        candidate_token,\n                    )\n                )\n                for candidate_token\n                in candidate_tokens\n            ),\n            default=0.0,\n        )\n\n        if len(query_token) <= 2:\n            threshold = 92.0\n        elif len(query_token) == 3:\n            threshold = 82.0\n        else:\n            threshold = 75.0\n\n        if best >= threshold:\n            approximate_hits += 1\n\n    return (\n        exact_coverage,\n        (\n            approximate_hits\n            / len(query_tokens)\n        ),\n    )\n\n\nclass OCRSearchEngine:\n    MODES = {\n        "precision": {\n            "allow_weak": False,\n            "min_det": 0.70,\n            "min_rec": 0.72,\n            "min_textuality": 0.72,\n            "min_score": 53.0,\n        },\n        "balanced": {\n            "allow_weak": True,\n            "min_det": 0.55,\n            "min_rec": 0.58,\n            "min_textuality": 0.62,\n            "min_score": 43.0,\n        },\n        "recall": {\n            "allow_weak": True,\n            "min_det": 0.40,\n            "min_rec": 0.45,\n            "min_textuality": 0.50,\n            "min_score": 31.0,\n        },\n    }\n\n    def __init__(\n        self,\n        database_path: str | Path,\n    ) -> None:\n        self.v1 = _BaseOCRSearchEngine(\n            database_path\n        )\n        self.metadata = self.v1.metadata\n        self.database_path = self.v1.database_path\n        self.connection = self.v1.connection\n        self.normalize = normalize_unicode\n        self.remove_accents = remove_accents\n        self.extract_numbers = extract_numbers\n\n    def close(self) -> None:\n        self.v1.close()\n\n    def get_frame(\n        self,\n        frame_name: str,\n    ) -> dict[str, Any] | None:\n        return self.v1.get_frame(frame_name)\n\n    def _region_valid(\n        self,\n        region,\n        *,\n        query_norm: str,\n        config: dict[str, Any],\n    ) -> tuple[bool, str | None]:\n        quality = (\n            region["quality"]\n            or ""\n        )\n\n        if quality == "drop":\n            return False, "quality_drop"\n\n        if (\n            quality == "weak"\n            and not config["allow_weak"]\n        ):\n            return False, "weak_not_allowed"\n\n        text = (\n            region["text"]\n            or ""\n        )\n        normalized = self.normalize(\n            text\n        )\n        compact = normalized.replace(\n            " ",\n            "",\n        )\n\n        if not compact:\n            return False, "empty_text"\n\n        if (\n            len(compact) == 1\n            and compact\n            != query_norm.replace(\n                " ",\n                "",\n            )\n        ):\n            return False, "single_character"\n\n        if (\n            textuality_score(text)\n            < config[\n                "min_textuality"\n            ]\n        ):\n            return False, "low_textuality"\n\n        det_conf = region[\n            "det_conf"\n        ]\n        rec_conf = region[\n            "rec_conf"\n        ]\n\n        if (\n            det_conf is not None\n            and float(det_conf)\n            < config["min_det"]\n        ):\n            return False, "low_det_conf"\n\n        if (\n            rec_conf is not None\n            and float(rec_conf)\n            < config["min_rec"]\n        ):\n            return False, "low_rec_conf"\n\n        return True, None\n\n    @staticmethod\n    def _gate(\n        *,\n        query_token_count: int,\n        exact_coverage: float,\n        approximate_coverage: float,\n        phrase_score: float,\n        region_score: float,\n        exact_phrase: bool,\n        mode: str,\n    ) -> tuple[bool, str | None]:\n        if exact_phrase:\n            return True, None\n\n        if query_token_count <= 1:\n            threshold = {\n                "precision": 84.0,\n                "balanced": 76.0,\n                "recall": 68.0,\n            }[mode]\n\n            if (\n                phrase_score < threshold\n                and region_score < threshold\n                and exact_coverage < 1.0\n            ):\n                return (\n                    False,\n                    "short_query_low_match",\n                )\n            return True, None\n\n        if query_token_count <= 3:\n            threshold = {\n                "precision": 0.67,\n                "balanced": 0.50,\n                "recall": 0.34,\n            }[mode]\n        else:\n            threshold = {\n                "precision": 0.55,\n                "balanced": 0.40,\n                "recall": 0.25,\n            }[mode]\n\n        if max(\n            exact_coverage,\n            approximate_coverage,\n        ) < threshold:\n            return (\n                False,\n                "low_token_coverage",\n            )\n\n        return True, None\n\n    def search(\n        self,\n        query: str,\n        *,\n        top_k: int = 50,\n        candidate_limit: int = 1800,\n        mode: str = "balanced",\n        collection_id: str | None = None,\n        video_id: str | None = None,\n        deduplicate_seconds: float = 2.4,\n        include_rejected: bool = False,\n    ):\n        if mode not in self.MODES:\n            raise ValueError(\n                f"Unknown mode: {mode}"\n            )\n\n        config = self.MODES[\n            mode\n        ]\n        query_norm = self.normalize(\n            query\n        )\n        query_ascii = (\n            self.remove_accents(\n                query\n            )\n        )\n        query_tokens = (\n            query_norm.split()\n        )\n        query_token_count = len(\n            query_tokens\n        )\n\n        if not query_norm:\n            return []\n\n        lexical_scores = (\n            self.v1._retrieve(\n                query,\n                candidate_limit,\n            )\n        )\n\n        if not lexical_scores:\n            empty_report = {\n                "results": [],\n                "rejected": [],\n                "candidate_count": 0,\n                "accepted_before_dedup": 0,\n                "diagnosis": (\n                    "NO_FTS_CANDIDATES"\n                ),\n            }\n            return (\n                empty_report\n                if include_rejected\n                else []\n            )\n\n        candidate_ids = [\n            frame_id\n            for frame_id, _\n            in sorted(\n                lexical_scores.items(),\n                key=lambda item: item[1],\n                reverse=True,\n            )[\n                :candidate_limit\n            ]\n        ]\n\n        frames = self.v1._fetch_frames(\n            candidate_ids\n        )\n\n        candidate_ids = [\n            frame_id\n            for frame_id in candidate_ids\n            if (\n                frame_id in frames\n                and (\n                    collection_id is None\n                    or frames[\n                        frame_id\n                    ][\n                        "collection_id"\n                    ]\n                    == collection_id\n                )\n                and (\n                    video_id is None\n                    or frames[\n                        frame_id\n                    ][\n                        "video_id"\n                    ]\n                    == video_id\n                )\n            )\n        ]\n\n        if not candidate_ids:\n            empty_report = {\n                "results": [],\n                "rejected": [],\n                "candidate_count": 0,\n                "accepted_before_dedup": 0,\n                "diagnosis": (\n                    "FILTERED_BY_SCOPE"\n                ),\n            }\n            return (\n                empty_report\n                if include_rejected\n                else []\n            )\n\n        regions = (\n            self.v1._fetch_regions(\n                candidate_ids\n            )\n        )\n\n        max_lexical = max(\n            lexical_scores.get(\n                frame_id,\n                0.0,\n            )\n            for frame_id in candidate_ids\n        )\n        max_lexical = max(\n            max_lexical,\n            1e-12,\n        )\n\n        query_numbers = (\n            self.extract_numbers(\n                query\n            )\n        )\n\n        accepted = []\n        rejected = []\n\n        for frame_id in candidate_ids:\n            row = frames[\n                frame_id\n            ]\n\n            primary_norm = row[\n                "primary_norm"\n            ]\n            recall_norm = row[\n                "recall_norm"\n            ]\n            primary_ascii = row[\n                "primary_ascii"\n            ]\n            recall_ascii = row[\n                "recall_ascii"\n            ]\n\n            unicode_text = (\n                primary_norm\n                + " "\n                + recall_norm\n            )\n            ascii_text = (\n                primary_ascii\n                + " "\n                + recall_ascii\n            )\n\n            phrase_score = max(\n                float(\n                    fuzz.partial_ratio(\n                        query_norm,\n                        primary_norm,\n                    )\n                ),\n                0.92\n                * float(\n                    fuzz.partial_ratio(\n                        query_norm,\n                        recall_norm,\n                    )\n                ),\n                float(\n                    fuzz.partial_ratio(\n                        query_ascii,\n                        primary_ascii,\n                    )\n                ),\n                0.92\n                * float(\n                    fuzz.partial_ratio(\n                        query_ascii,\n                        recall_ascii,\n                    )\n                ),\n            )\n\n            token_set_score = max(\n                float(\n                    fuzz.token_set_ratio(\n                        query_norm,\n                        primary_norm,\n                    )\n                ),\n                0.92\n                * float(\n                    fuzz.token_set_ratio(\n                        query_norm,\n                        recall_norm,\n                    )\n                ),\n                float(\n                    fuzz.token_set_ratio(\n                        query_ascii,\n                        primary_ascii,\n                    )\n                ),\n                0.92\n                * float(\n                    fuzz.token_set_ratio(\n                        query_ascii,\n                        recall_ascii,\n                    )\n                ),\n            )\n\n            (\n                exact_unicode,\n                approximate_unicode,\n            ) = token_coverage(\n                self.normalize,\n                query_norm,\n                unicode_text,\n            )\n            (\n                exact_ascii,\n                approximate_ascii,\n            ) = token_coverage(\n                self.normalize,\n                query_ascii,\n                ascii_text,\n            )\n\n            exact_coverage = max(\n                exact_unicode,\n                exact_ascii,\n            )\n            approximate_coverage = max(\n                approximate_unicode,\n                approximate_ascii,\n            )\n\n            best_region = None\n            best_region_score = 0.0\n            invalid_reasons = (\n                defaultdict(int)\n            )\n\n            for region in regions.get(\n                frame_id,\n                [],\n            ):\n                valid, reason = (\n                    self._region_valid(\n                        region,\n                        query_norm=query_norm,\n                        config=config,\n                    )\n                )\n\n                if not valid:\n                    invalid_reasons[\n                        reason or "unknown"\n                    ] += 1\n                    continue\n\n                region_text = (\n                    region["text"]\n                    or ""\n                )\n                region_norm = (\n                    self.normalize(\n                        region_text\n                    )\n                )\n                region_ascii = (\n                    self.remove_accents(\n                        region_text\n                    )\n                )\n\n                current_score = max(\n                    float(\n                        fuzz.WRatio(\n                            query_norm,\n                            region_norm,\n                        )\n                    ),\n                    float(\n                        fuzz.WRatio(\n                            query_ascii,\n                            region_ascii,\n                        )\n                    ),\n                    float(\n                        fuzz.partial_ratio(\n                            query_norm,\n                            region_norm,\n                        )\n                    ),\n                )\n\n                if (\n                    region["quality"]\n                    == "weak"\n                ):\n                    current_score *= 0.94\n\n                if (\n                    current_score\n                    > best_region_score\n                ):\n                    best_region_score = (\n                        current_score\n                    )\n                    best_region = region\n\n            lexical_score = (\n                100.0\n                * lexical_scores.get(\n                    frame_id,\n                    0.0,\n                )\n                / max_lexical\n            )\n\n            if query_numbers:\n                matched_numbers = sum(\n                    number in ascii_text\n                    for number\n                    in query_numbers\n                )\n                number_score = (\n                    100.0\n                    * matched_numbers\n                    / len(query_numbers)\n                )\n            else:\n                number_score = None\n\n            exact_phrase = (\n                query_norm\n                in primary_norm\n                or query_norm\n                in recall_norm\n                or query_ascii\n                in primary_ascii\n                or query_ascii\n                in recall_ascii\n            )\n\n            gate_passed, gate_reason = (\n                self._gate(\n                    query_token_count=(\n                        query_token_count\n                    ),\n                    exact_coverage=(\n                        exact_coverage\n                    ),\n                    approximate_coverage=(\n                        approximate_coverage\n                    ),\n                    phrase_score=(\n                        phrase_score\n                    ),\n                    region_score=(\n                        best_region_score\n                    ),\n                    exact_phrase=(\n                        exact_phrase\n                    ),\n                    mode=mode,\n                )\n            )\n\n            score = (\n                0.18 * lexical_score\n                + 0.20\n                * 100.0\n                * exact_coverage\n                + 0.20\n                * 100.0\n                * approximate_coverage\n                + 0.16 * phrase_score\n                + 0.10 * token_set_score\n                + 0.16\n                * best_region_score\n            )\n\n            if number_score is not None:\n                score = (\n                    0.92 * score\n                    + 0.08\n                    * number_score\n                )\n\n            if exact_phrase:\n                score += 8.0\n\n            region_count = int(\n                row["region_count"]\n                or 0\n            )\n            drop_count = int(\n                row["drop_count"]\n                or 0\n            )\n            drop_ratio = (\n                drop_count\n                / max(\n                    1,\n                    region_count,\n                )\n            )\n            score -= (\n                5.0\n                * min(\n                    1.0,\n                    drop_ratio,\n                )\n            )\n\n            total_text_tokens = len(\n                unicode_text.split()\n            )\n\n            if (\n                query_token_count <= 2\n                and total_text_tokens >= 60\n                and exact_coverage < 1.0\n            ):\n                score -= 5.0\n\n            score = max(\n                0.0,\n                min(\n                    100.0,\n                    score,\n                ),\n            )\n\n            reject_reason = None\n\n            if not gate_passed:\n                reject_reason = (\n                    gate_reason\n                )\n            elif score < config[\n                "min_score"\n            ]:\n                reject_reason = (\n                    "below_min_score"\n                )\n\n            result = {\n                "modality": "ocr",\n                "search_version": "v2",\n                "search_mode": mode,\n                "collection_id": (\n                    row["collection_id"]\n                ),\n                "dataset_id": (\n                    row["dataset_id"]\n                ),\n                "video_id": (\n                    row["video_id"]\n                ),\n                "timestamp_sec": (\n                    row["timestamp_sec"]\n                ),\n                "frame_index": (\n                    row["frame_index"]\n                ),\n                "shot_id": (\n                    row["shot_id"]\n                ),\n                "frame": (\n                    row["frame"]\n                ),\n                "frame_text_index": (\n                    row[\n                        "frame_text_index"\n                    ]\n                ),\n                "frame_text_recall": (\n                    row[\n                        "frame_text_recall"\n                    ]\n                ),\n                "score": round(\n                    score,\n                    4,\n                ),\n                "accepted": (\n                    reject_reason is None\n                ),\n                "reject_reason": (\n                    reject_reason\n                ),\n                "score_breakdown": {\n                    "lexical": round(\n                        lexical_score,\n                        4,\n                    ),\n                    "exact_coverage": round(\n                        exact_coverage,\n                        4,\n                    ),\n                    "approximate_coverage": round(\n                        approximate_coverage,\n                        4,\n                    ),\n                    "phrase": round(\n                        phrase_score,\n                        4,\n                    ),\n                    "token_set": round(\n                        token_set_score,\n                        4,\n                    ),\n                    "region": round(\n                        best_region_score,\n                        4,\n                    ),\n                    "number": (\n                        round(\n                            number_score,\n                            4,\n                        )\n                        if number_score\n                        is not None\n                        else None\n                    ),\n                    "drop_ratio": round(\n                        drop_ratio,\n                        4,\n                    ),\n                },\n                "invalid_region_reasons": (\n                    dict(\n                        invalid_reasons\n                    )\n                ),\n            }\n\n            if best_region is None:\n                result[\n                    "matched_region"\n                ] = None\n            else:\n                result[\n                    "matched_region"\n                ] = {\n                    "text": (\n                        best_region["text"]\n                    ),\n                    "quality": (\n                        best_region[\n                            "quality"\n                        ]\n                    ),\n                    "polygon": json.loads(\n                        best_region[\n                            "polygon_json"\n                        ]\n                    ),\n                    "det_conf": (\n                        best_region[\n                            "det_conf"\n                        ]\n                    ),\n                    "rec_conf": (\n                        best_region[\n                            "rec_conf"\n                        ]\n                    ),\n                    "rotation": (\n                        best_region[\n                            "rotation"\n                        ]\n                    ),\n                    "textuality": round(\n                        textuality_score(\n                            best_region[\n                                "text"\n                            ]\n                        ),\n                        4,\n                    ),\n                }\n\n            if reject_reason is None:\n                accepted.append(\n                    result\n                )\n            else:\n                rejected.append(\n                    result\n                )\n\n        accepted.sort(\n            key=lambda item: item[\n                "score"\n            ],\n            reverse=True,\n        )\n        rejected.sort(\n            key=lambda item: item[\n                "score"\n            ],\n            reverse=True,\n        )\n\n        deduplicated = []\n\n        for result in accepted:\n            duplicate = False\n\n            for selected in deduplicated:\n                if (\n                    result["video_id"]\n                    != selected["video_id"]\n                ):\n                    continue\n\n                first_time = (\n                    result[\n                        "timestamp_sec"\n                    ]\n                )\n                second_time = (\n                    selected[\n                        "timestamp_sec"\n                    ]\n                )\n\n                if (\n                    first_time is not None\n                    and second_time\n                    is not None\n                    and abs(\n                        first_time\n                        - second_time\n                    )\n                    <= deduplicate_seconds\n                ):\n                    duplicate = True\n                    break\n\n            if not duplicate:\n                deduplicated.append(\n                    result\n                )\n\n            if (\n                len(deduplicated)\n                >= top_k\n            ):\n                break\n\n        for rank, result in enumerate(\n            deduplicated,\n            start=1,\n        ):\n            result["rank"] = rank\n\n        if include_rejected:\n            return {\n                "results": deduplicated,\n                "rejected": rejected[:100],\n                "candidate_count": len(\n                    candidate_ids\n                ),\n                "accepted_before_dedup": len(\n                    accepted\n                ),\n                "diagnosis": (\n                    "OK"\n                    if deduplicated\n                    else (\n                        "CANDIDATES_REJECTED"\n                        if rejected\n                        else "NO_CANDIDATES"\n                    )\n                ),\n            }\n\n        return deduplicated\n'
V2_ENGINE_SOURCE = '\nfrom __future__ import annotations\n\nimport json\nimport re\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Any\n\nfrom rapidfuzz import fuzz\n\n\n_ALNUM_RE = re.compile(\n    r"[^\\W_]",\n    flags=re.UNICODE,\n)\n\n\ndef textuality_score(\n    value: Any,\n) -> float:\n    raw = (\n        ""\n        if value is None\n        else str(value)\n    )\n    compact = "".join(\n        character\n        for character in raw\n        if not character.isspace()\n    )\n\n    if not compact:\n        return 0.0\n\n    alnum_count = sum(\n        character.isalnum()\n        for character in compact\n    )\n    ratio = (\n        alnum_count\n        / max(\n            1,\n            len(compact),\n        )\n    )\n\n    lowered = "".join(\n        character.lower()\n        for character in compact\n        if character.isalnum()\n    )\n\n    if not lowered:\n        return 0.0\n\n    unique_ratio = (\n        len(set(lowered))\n        / len(lowered)\n    )\n\n    if (\n        len(lowered) >= 5\n        and unique_ratio < 0.18\n    ):\n        ratio *= 0.45\n\n    return max(\n        0.0,\n        min(\n            1.0,\n            ratio,\n        ),\n    )\n\n\ndef token_coverage(\n    normalize_function,\n    query: str,\n    candidate: str,\n) -> tuple[float, float]:\n    query_tokens = [\n        token\n        for token in normalize_function(\n            query\n        ).split()\n        if token\n    ]\n    candidate_tokens = [\n        token\n        for token in normalize_function(\n            candidate\n        ).split()\n        if token\n    ]\n\n    if not query_tokens:\n        return 0.0, 0.0\n\n    candidate_set = set(\n        candidate_tokens\n    )\n    exact_hits = sum(\n        token in candidate_set\n        for token in query_tokens\n    )\n    exact_coverage = (\n        exact_hits\n        / len(query_tokens)\n    )\n\n    if not candidate_tokens:\n        return exact_coverage, 0.0\n\n    approximate_hits = 0\n\n    for query_token in query_tokens:\n        best = max(\n            (\n                float(\n                    fuzz.ratio(\n                        query_token,\n                        candidate_token,\n                    )\n                )\n                for candidate_token\n                in candidate_tokens\n            ),\n            default=0.0,\n        )\n\n        if len(query_token) <= 2:\n            threshold = 92.0\n        elif len(query_token) == 3:\n            threshold = 82.0\n        else:\n            threshold = 75.0\n\n        if best >= threshold:\n            approximate_hits += 1\n\n    return (\n        exact_coverage,\n        (\n            approximate_hits\n            / len(query_tokens)\n        ),\n    )\n\n\nclass OCRSearchEngineV2:\n    MODES = {\n        "precision": {\n            "allow_weak": False,\n            "min_det": 0.70,\n            "min_rec": 0.72,\n            "min_textuality": 0.72,\n            "min_score": 53.0,\n        },\n        "balanced": {\n            "allow_weak": True,\n            "min_det": 0.55,\n            "min_rec": 0.58,\n            "min_textuality": 0.62,\n            "min_score": 43.0,\n        },\n        "recall": {\n            "allow_weak": True,\n            "min_det": 0.40,\n            "min_rec": 0.45,\n            "min_textuality": 0.50,\n            "min_score": 31.0,\n        },\n    }\n\n    def __init__(\n        self,\n        v1_engine,\n    ) -> None:\n        self.v1 = v1_engine\n        self.metadata = v1_engine.metadata\n        self.normalize_unicode = (\n            v1_engine.__class__.__module__\n        )\n\n        module = __import__(\n            self.normalize_unicode,\n            fromlist=[\n                "normalize_unicode",\n                "remove_accents",\n                "extract_numbers",\n            ],\n        )\n        self.normalize = (\n            module.normalize_unicode\n        )\n        self.remove_accents = (\n            module.remove_accents\n        )\n        self.extract_numbers = (\n            module.extract_numbers\n        )\n\n    def _region_valid(\n        self,\n        region,\n        *,\n        query_norm: str,\n        config: dict[str, Any],\n    ) -> tuple[bool, str | None]:\n        quality = (\n            region["quality"]\n            or ""\n        )\n\n        if quality == "drop":\n            return False, "quality_drop"\n\n        if (\n            quality == "weak"\n            and not config["allow_weak"]\n        ):\n            return False, "weak_not_allowed"\n\n        text = (\n            region["text"]\n            or ""\n        )\n        normalized = self.normalize(\n            text\n        )\n        compact = normalized.replace(\n            " ",\n            "",\n        )\n\n        if not compact:\n            return False, "empty_text"\n\n        if (\n            len(compact) == 1\n            and compact\n            != query_norm.replace(\n                " ",\n                "",\n            )\n        ):\n            return False, "single_character"\n\n        if (\n            textuality_score(text)\n            < config[\n                "min_textuality"\n            ]\n        ):\n            return False, "low_textuality"\n\n        det_conf = region[\n            "det_conf"\n        ]\n        rec_conf = region[\n            "rec_conf"\n        ]\n\n        if (\n            det_conf is not None\n            and float(det_conf)\n            < config["min_det"]\n        ):\n            return False, "low_det_conf"\n\n        if (\n            rec_conf is not None\n            and float(rec_conf)\n            < config["min_rec"]\n        ):\n            return False, "low_rec_conf"\n\n        return True, None\n\n    @staticmethod\n    def _gate(\n        *,\n        query_token_count: int,\n        exact_coverage: float,\n        approximate_coverage: float,\n        phrase_score: float,\n        region_score: float,\n        exact_phrase: bool,\n        mode: str,\n    ) -> tuple[bool, str | None]:\n        if exact_phrase:\n            return True, None\n\n        if query_token_count <= 1:\n            threshold = {\n                "precision": 84.0,\n                "balanced": 76.0,\n                "recall": 68.0,\n            }[mode]\n\n            if (\n                phrase_score < threshold\n                and region_score < threshold\n                and exact_coverage < 1.0\n            ):\n                return (\n                    False,\n                    "short_query_low_match",\n                )\n            return True, None\n\n        if query_token_count <= 3:\n            threshold = {\n                "precision": 0.67,\n                "balanced": 0.50,\n                "recall": 0.34,\n            }[mode]\n        else:\n            threshold = {\n                "precision": 0.55,\n                "balanced": 0.40,\n                "recall": 0.25,\n            }[mode]\n\n        if max(\n            exact_coverage,\n            approximate_coverage,\n        ) < threshold:\n            return (\n                False,\n                "low_token_coverage",\n            )\n\n        return True, None\n\n    def search(\n        self,\n        query: str,\n        *,\n        top_k: int = 50,\n        candidate_limit: int = 1800,\n        mode: str = "balanced",\n        collection_id: str | None = None,\n        video_id: str | None = None,\n        deduplicate_seconds: float = 2.4,\n        include_rejected: bool = False,\n    ):\n        if mode not in self.MODES:\n            raise ValueError(\n                f"Unknown mode: {mode}"\n            )\n\n        config = self.MODES[\n            mode\n        ]\n        query_norm = self.normalize(\n            query\n        )\n        query_ascii = (\n            self.remove_accents(\n                query\n            )\n        )\n        query_tokens = (\n            query_norm.split()\n        )\n        query_token_count = len(\n            query_tokens\n        )\n\n        if not query_norm:\n            return []\n\n        lexical_scores = (\n            self.v1._retrieve(\n                query,\n                candidate_limit,\n            )\n        )\n\n        if not lexical_scores:\n            empty_report = {\n                "results": [],\n                "rejected": [],\n                "candidate_count": 0,\n                "accepted_before_dedup": 0,\n                "diagnosis": (\n                    "NO_FTS_CANDIDATES"\n                ),\n            }\n            return (\n                empty_report\n                if include_rejected\n                else []\n            )\n\n        candidate_ids = [\n            frame_id\n            for frame_id, _\n            in sorted(\n                lexical_scores.items(),\n                key=lambda item: item[1],\n                reverse=True,\n            )[\n                :candidate_limit\n            ]\n        ]\n\n        frames = self.v1._fetch_frames(\n            candidate_ids\n        )\n\n        candidate_ids = [\n            frame_id\n            for frame_id in candidate_ids\n            if (\n                frame_id in frames\n                and (\n                    collection_id is None\n                    or frames[\n                        frame_id\n                    ][\n                        "collection_id"\n                    ]\n                    == collection_id\n                )\n                and (\n                    video_id is None\n                    or frames[\n                        frame_id\n                    ][\n                        "video_id"\n                    ]\n                    == video_id\n                )\n            )\n        ]\n\n        if not candidate_ids:\n            empty_report = {\n                "results": [],\n                "rejected": [],\n                "candidate_count": 0,\n                "accepted_before_dedup": 0,\n                "diagnosis": (\n                    "FILTERED_BY_SCOPE"\n                ),\n            }\n            return (\n                empty_report\n                if include_rejected\n                else []\n            )\n\n        regions = (\n            self.v1._fetch_regions(\n                candidate_ids\n            )\n        )\n\n        max_lexical = max(\n            lexical_scores.get(\n                frame_id,\n                0.0,\n            )\n            for frame_id in candidate_ids\n        )\n        max_lexical = max(\n            max_lexical,\n            1e-12,\n        )\n\n        query_numbers = (\n            self.extract_numbers(\n                query\n            )\n        )\n\n        accepted = []\n        rejected = []\n\n        for frame_id in candidate_ids:\n            row = frames[\n                frame_id\n            ]\n\n            primary_norm = row[\n                "primary_norm"\n            ]\n            recall_norm = row[\n                "recall_norm"\n            ]\n            primary_ascii = row[\n                "primary_ascii"\n            ]\n            recall_ascii = row[\n                "recall_ascii"\n            ]\n\n            unicode_text = (\n                primary_norm\n                + " "\n                + recall_norm\n            )\n            ascii_text = (\n                primary_ascii\n                + " "\n                + recall_ascii\n            )\n\n            phrase_score = max(\n                float(\n                    fuzz.partial_ratio(\n                        query_norm,\n                        primary_norm,\n                    )\n                ),\n                0.92\n                * float(\n                    fuzz.partial_ratio(\n                        query_norm,\n                        recall_norm,\n                    )\n                ),\n                float(\n                    fuzz.partial_ratio(\n                        query_ascii,\n                        primary_ascii,\n                    )\n                ),\n                0.92\n                * float(\n                    fuzz.partial_ratio(\n                        query_ascii,\n                        recall_ascii,\n                    )\n                ),\n            )\n\n            token_set_score = max(\n                float(\n                    fuzz.token_set_ratio(\n                        query_norm,\n                        primary_norm,\n                    )\n                ),\n                0.92\n                * float(\n                    fuzz.token_set_ratio(\n                        query_norm,\n                        recall_norm,\n                    )\n                ),\n                float(\n                    fuzz.token_set_ratio(\n                        query_ascii,\n                        primary_ascii,\n                    )\n                ),\n                0.92\n                * float(\n                    fuzz.token_set_ratio(\n                        query_ascii,\n                        recall_ascii,\n                    )\n                ),\n            )\n\n            (\n                exact_unicode,\n                approximate_unicode,\n            ) = token_coverage(\n                self.normalize,\n                query_norm,\n                unicode_text,\n            )\n            (\n                exact_ascii,\n                approximate_ascii,\n            ) = token_coverage(\n                self.normalize,\n                query_ascii,\n                ascii_text,\n            )\n\n            exact_coverage = max(\n                exact_unicode,\n                exact_ascii,\n            )\n            approximate_coverage = max(\n                approximate_unicode,\n                approximate_ascii,\n            )\n\n            best_region = None\n            best_region_score = 0.0\n            invalid_reasons = (\n                defaultdict(int)\n            )\n\n            for region in regions.get(\n                frame_id,\n                [],\n            ):\n                valid, reason = (\n                    self._region_valid(\n                        region,\n                        query_norm=query_norm,\n                        config=config,\n                    )\n                )\n\n                if not valid:\n                    invalid_reasons[\n                        reason or "unknown"\n                    ] += 1\n                    continue\n\n                region_text = (\n                    region["text"]\n                    or ""\n                )\n                region_norm = (\n                    self.normalize(\n                        region_text\n                    )\n                )\n                region_ascii = (\n                    self.remove_accents(\n                        region_text\n                    )\n                )\n\n                current_score = max(\n                    float(\n                        fuzz.WRatio(\n                            query_norm,\n                            region_norm,\n                        )\n                    ),\n                    float(\n                        fuzz.WRatio(\n                            query_ascii,\n                            region_ascii,\n                        )\n                    ),\n                    float(\n                        fuzz.partial_ratio(\n                            query_norm,\n                            region_norm,\n                        )\n                    ),\n                )\n\n                if (\n                    region["quality"]\n                    == "weak"\n                ):\n                    current_score *= 0.94\n\n                if (\n                    current_score\n                    > best_region_score\n                ):\n                    best_region_score = (\n                        current_score\n                    )\n                    best_region = region\n\n            lexical_score = (\n                100.0\n                * lexical_scores.get(\n                    frame_id,\n                    0.0,\n                )\n                / max_lexical\n            )\n\n            if query_numbers:\n                matched_numbers = sum(\n                    number in ascii_text\n                    for number\n                    in query_numbers\n                )\n                number_score = (\n                    100.0\n                    * matched_numbers\n                    / len(query_numbers)\n                )\n            else:\n                number_score = None\n\n            exact_phrase = (\n                query_norm\n                in primary_norm\n                or query_norm\n                in recall_norm\n                or query_ascii\n                in primary_ascii\n                or query_ascii\n                in recall_ascii\n            )\n\n            gate_passed, gate_reason = (\n                self._gate(\n                    query_token_count=(\n                        query_token_count\n                    ),\n                    exact_coverage=(\n                        exact_coverage\n                    ),\n                    approximate_coverage=(\n                        approximate_coverage\n                    ),\n                    phrase_score=(\n                        phrase_score\n                    ),\n                    region_score=(\n                        best_region_score\n                    ),\n                    exact_phrase=(\n                        exact_phrase\n                    ),\n                    mode=mode,\n                )\n            )\n\n            score = (\n                0.18 * lexical_score\n                + 0.20\n                * 100.0\n                * exact_coverage\n                + 0.20\n                * 100.0\n                * approximate_coverage\n                + 0.16 * phrase_score\n                + 0.10 * token_set_score\n                + 0.16\n                * best_region_score\n            )\n\n            if number_score is not None:\n                score = (\n                    0.92 * score\n                    + 0.08\n                    * number_score\n                )\n\n            if exact_phrase:\n                score += 8.0\n\n            region_count = int(\n                row["region_count"]\n                or 0\n            )\n            drop_count = int(\n                row["drop_count"]\n                or 0\n            )\n            drop_ratio = (\n                drop_count\n                / max(\n                    1,\n                    region_count,\n                )\n            )\n            score -= (\n                5.0\n                * min(\n                    1.0,\n                    drop_ratio,\n                )\n            )\n\n            total_text_tokens = len(\n                unicode_text.split()\n            )\n\n            if (\n                query_token_count <= 2\n                and total_text_tokens >= 60\n                and exact_coverage < 1.0\n            ):\n                score -= 5.0\n\n            score = max(\n                0.0,\n                min(\n                    100.0,\n                    score,\n                ),\n            )\n\n            reject_reason = None\n\n            if not gate_passed:\n                reject_reason = (\n                    gate_reason\n                )\n            elif score < config[\n                "min_score"\n            ]:\n                reject_reason = (\n                    "below_min_score"\n                )\n\n            result = {\n                "modality": "ocr",\n                "search_version": "v2",\n                "search_mode": mode,\n                "collection_id": (\n                    row["collection_id"]\n                ),\n                "dataset_id": (\n                    row["dataset_id"]\n                ),\n                "video_id": (\n                    row["video_id"]\n                ),\n                "timestamp_sec": (\n                    row["timestamp_sec"]\n                ),\n                "frame_index": (\n                    row["frame_index"]\n                ),\n                "shot_id": (\n                    row["shot_id"]\n                ),\n                "frame": (\n                    row["frame"]\n                ),\n                "frame_text_index": (\n                    row[\n                        "frame_text_index"\n                    ]\n                ),\n                "frame_text_recall": (\n                    row[\n                        "frame_text_recall"\n                    ]\n                ),\n                "score": round(\n                    score,\n                    4,\n                ),\n                "accepted": (\n                    reject_reason is None\n                ),\n                "reject_reason": (\n                    reject_reason\n                ),\n                "score_breakdown": {\n                    "lexical": round(\n                        lexical_score,\n                        4,\n                    ),\n                    "exact_coverage": round(\n                        exact_coverage,\n                        4,\n                    ),\n                    "approximate_coverage": round(\n                        approximate_coverage,\n                        4,\n                    ),\n                    "phrase": round(\n                        phrase_score,\n                        4,\n                    ),\n                    "token_set": round(\n                        token_set_score,\n                        4,\n                    ),\n                    "region": round(\n                        best_region_score,\n                        4,\n                    ),\n                    "number": (\n                        round(\n                            number_score,\n                            4,\n                        )\n                        if number_score\n                        is not None\n                        else None\n                    ),\n                    "drop_ratio": round(\n                        drop_ratio,\n                        4,\n                    ),\n                },\n                "invalid_region_reasons": (\n                    dict(\n                        invalid_reasons\n                    )\n                ),\n            }\n\n            if best_region is None:\n                result[\n                    "matched_region"\n                ] = None\n            else:\n                result[\n                    "matched_region"\n                ] = {\n                    "text": (\n                        best_region["text"]\n                    ),\n                    "quality": (\n                        best_region[\n                            "quality"\n                        ]\n                    ),\n                    "polygon": json.loads(\n                        best_region[\n                            "polygon_json"\n                        ]\n                    ),\n                    "det_conf": (\n                        best_region[\n                            "det_conf"\n                        ]\n                    ),\n                    "rec_conf": (\n                        best_region[\n                            "rec_conf"\n                        ]\n                    ),\n                    "rotation": (\n                        best_region[\n                            "rotation"\n                        ]\n                    ),\n                    "textuality": round(\n                        textuality_score(\n                            best_region[\n                                "text"\n                            ]\n                        ),\n                        4,\n                    ),\n                }\n\n            if reject_reason is None:\n                accepted.append(\n                    result\n                )\n            else:\n                rejected.append(\n                    result\n                )\n\n        accepted.sort(\n            key=lambda item: item[\n                "score"\n            ],\n            reverse=True,\n        )\n        rejected.sort(\n            key=lambda item: item[\n                "score"\n            ],\n            reverse=True,\n        )\n\n        deduplicated = []\n\n        for result in accepted:\n            duplicate = False\n\n            for selected in deduplicated:\n                if (\n                    result["video_id"]\n                    != selected["video_id"]\n                ):\n                    continue\n\n                first_time = (\n                    result[\n                        "timestamp_sec"\n                    ]\n                )\n                second_time = (\n                    selected[\n                        "timestamp_sec"\n                    ]\n                )\n\n                if (\n                    first_time is not None\n                    and second_time\n                    is not None\n                    and abs(\n                        first_time\n                        - second_time\n                    )\n                    <= deduplicate_seconds\n                ):\n                    duplicate = True\n                    break\n\n            if not duplicate:\n                deduplicated.append(\n                    result\n                )\n\n            if (\n                len(deduplicated)\n                >= top_k\n            ):\n                break\n\n        for rank, result in enumerate(\n            deduplicated,\n            start=1,\n        ):\n            result["rank"] = rank\n\n        if include_rejected:\n            return {\n                "results": deduplicated,\n                "rejected": rejected[:100],\n                "candidate_count": len(\n                    candidate_ids\n                ),\n                "accepted_before_dedup": len(\n                    accepted\n                ),\n                "diagnosis": (\n                    "OK"\n                    if deduplicated\n                    else (\n                        "CANDIDATES_REJECTED"\n                        if rejected\n                        else "NO_CANDIDATES"\n                    )\n                ),\n            }\n\n        return deduplicated\n'

BASE_ENGINE_PATH = RUNTIME_ROOT / "ocr_search_engine_base.py"
V2_ENGINE_PATH = RUNTIME_ROOT / "ocr_search_engine_v2.py"

BASE_ENGINE_PATH.write_text(
    BASE_ENGINE_SOURCE,
    encoding="utf-8",
)
V2_ENGINE_PATH.write_text(
    V2_ENGINE_SOURCE,
    encoding="utf-8",
)

compile(
    BASE_ENGINE_SOURCE,
    str(BASE_ENGINE_PATH),
    "exec",
)
compile(
    V2_ENGINE_SOURCE,
    str(V2_ENGINE_PATH),
    "exec",
)


def load_module(module_name: str, path: Path):
    specification = importlib.util.spec_from_file_location(
        module_name,
        path,
    )
    if specification is None or specification.loader is None:
        raise ImportError(path)

    module = importlib.util.module_from_spec(specification)
    sys.modules[module_name] = module
    specification.loader.exec_module(module)
    return module


base_module = load_module(
    "ocr_search_engine_base",
    BASE_ENGINE_PATH,
)
v2_module = load_module(
    "ocr_search_engine_v2",
    V2_ENGINE_PATH,
)

engine_v1 = base_module._BaseOCRSearchEngine(
    DATABASE_PATH
)

engine_v2 = v2_module.OCRSearchEngineV2(
    engine_v1
)

print("Search V2 ready.")
print(
    json.dumps(
        engine_v2.MODES,
        ensure_ascii=False,
        indent=2,
    )
)



# ------------------------------------------------------------
# 4. FRAME PATH ADAPTER — READ MILVUS LITE PARQUET DIRECTLY
#
# OCR only needs:
#     frame basename -> image_path
#
# Do NOT call MilvusClient.query_iterator() here. The previous version
# enumerated the entire collection through localhost gRPC and was the
# source of:
#     GOAWAY / ENHANCE_YOUR_CALM / too_many_pings
#
# Milvus Lite stores immutable collection segments as Parquet. We scan
# ONLY the `metadata` column once, create a tiny SQLite lookup, then all
# OCR searches use indexed local SQL lookups.
# ------------------------------------------------------------

import pyarrow.parquet as pq

OCR_FRAME_LOOKUP_DB = (
    RUNTIME_ROOT / "ocr_frame_paths_from_milvus_parquet.sqlite"
)

OCR_FRAME_MILVUS_COLLECTION = os.getenv(
    "AIC_OCR_FRAME_COLLECTION",
    MILVUS_COLLECTION_BY_PREFIX.get(
        "metaclip_image_emb_chunk",
        "embeddings_metaclip_openclip__metaclip_image_emb__d768",
    ),
)

OCR_FRAME_LOOKUP_WARMUP = False

OCR_FRAME_PARQUET_BATCH_SIZE = int(
    os.getenv("AIC_OCR_FRAME_PARQUET_BATCH_SIZE", "16384")
)

OCR_FRAME_PATTERN = re.compile(
    r"(?P<video_id>L\d{2}_V\d{3})"
    r"_t(?P<timestamp>\d+(?:\.\d+)?)"
    r"_f(?P<frame_idx>\d+)"
    r"_s(?P<shot_id>\d+)"
)

_OCR_FRAME_LOOKUP_READY = False
_OCR_FRAME_LOOKUP_ERROR: str | None = None
_OCR_FRAME_LOOKUP_CONN: sqlite3.Connection | None = None


def _normalize_kaggle_path(path_value: Any) -> str:
    return str(path_value).replace(
        "https://www.kaggle.com/kaggle/input/",
        "/kaggle/input/",
        1,
    )


def _decode_ocr_milvus_metadata(value: Any) -> dict[str, Any]:
    if isinstance(value, dict):
        return dict(value)

    # pyarrow may return map-like pairs for some JSON representations.
    if isinstance(value, list):
        try:
            if all(
                isinstance(item, (list, tuple)) and len(item) == 2
                for item in value
            ):
                return dict(value)
        except Exception:
            pass

    if isinstance(value, (bytes, bytearray, memoryview)):
        try:
            value = bytes(value).decode("utf-8")
        except Exception:
            return {}

    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            return parsed if isinstance(parsed, dict) else {}
        except Exception:
            return {}

    # Last-resort mapping-like object.
    try:
        mapped = dict(value)
        return mapped if isinstance(mapped, dict) else {}
    except Exception:
        return {}


def _milvus_collection_data_dir() -> Path:
    """
    Locate one collection inside the active Milvus Lite data directory.
    """
    runtime_root = (
        _MILVUS_DB_RUNTIME_PATH
        if _MILVUS_DB_RUNTIME_PATH is not None
        else prepare_milvus_db_path()
    )

    collection_dir = (
        Path(runtime_root)
        / "collections"
        / OCR_FRAME_MILVUS_COLLECTION
    )

    if not collection_dir.is_dir():
        raise FileNotFoundError(
            "Milvus Lite collection directory not found:\n"
            f"  {collection_dir}\n"
            "This OCR adapter expects the current directory-based "
            "Milvus Lite format (manifest + Parquet segments)."
        )

    return collection_dir


def _ocr_frame_lookup_cache_signature() -> str:
    collection_dir = _milvus_collection_data_dir()
    manifest_path = collection_dir / "manifest.json"

    manifest_stat = None
    if manifest_path.is_file():
        stat = manifest_path.stat()
        manifest_stat = {
            "size": int(stat.st_size),
            "mtime_ns": int(stat.st_mtime_ns),
        }

    return json.dumps(
        {
            "milvus_db": str(MILVUS_DB_SOURCE),
            "collection": OCR_FRAME_MILVUS_COLLECTION,
            "manifest": manifest_stat,
            "schema_version": 3,
            "backend": "direct_parquet_metadata",
        },
        sort_keys=True,
        ensure_ascii=False,
    )


def _open_ocr_frame_lookup_db() -> sqlite3.Connection:
    global _OCR_FRAME_LOOKUP_CONN

    if _OCR_FRAME_LOOKUP_CONN is None:
        OCR_FRAME_LOOKUP_DB.parent.mkdir(parents=True, exist_ok=True)
        _OCR_FRAME_LOOKUP_CONN = sqlite3.connect(
            str(OCR_FRAME_LOOKUP_DB),
            check_same_thread=False,
        )
        _OCR_FRAME_LOOKUP_CONN.row_factory = sqlite3.Row
        _OCR_FRAME_LOOKUP_CONN.execute("PRAGMA journal_mode=WAL")
        _OCR_FRAME_LOOKUP_CONN.execute("PRAGMA synchronous=NORMAL")
        _OCR_FRAME_LOOKUP_CONN.execute("PRAGMA temp_store=MEMORY")

    return _OCR_FRAME_LOOKUP_CONN


def _ocr_frame_lookup_is_valid(
    connection: sqlite3.Connection,
) -> bool:
    try:
        tables = {
            row[0]
            for row in connection.execute(
                "SELECT name FROM sqlite_master WHERE type='table'"
            )
        }

        if not {"frame_paths", "lookup_meta"}.issubset(tables):
            return False

        signature_row = connection.execute(
            "SELECT value FROM lookup_meta WHERE key='signature'"
        ).fetchone()
        complete_row = connection.execute(
            "SELECT value FROM lookup_meta WHERE key='complete'"
        ).fetchone()

        return bool(
            signature_row
            and complete_row
            and signature_row[0] == _ocr_frame_lookup_cache_signature()
            and complete_row[0] == "1"
        )

    except Exception:
        return False


def _iter_milvus_metadata_from_parquet():
    """
    Yield metadata JSON values directly from immutable Milvus Lite
    Parquet segments. Only the `metadata` column is decoded.
    """
    collection_dir = _milvus_collection_data_dir()

    parquet_files = sorted(
        collection_dir.glob(
            "partitions/*/data/*.parquet"
        )
    )

    if not parquet_files:
        # Compatibility fallback for slightly different partition depth.
        parquet_files = sorted(
            collection_dir.rglob("data/*.parquet")
        )

    if not parquet_files:
        raise FileNotFoundError(
            "No Milvus Lite data parquet files found under:\n"
            f"  {collection_dir}"
        )

    print(
        f"[OCR FRAME] Parquet segments: {len(parquet_files):,}"
    )

    for file_index, parquet_path in enumerate(
        parquet_files,
        start=1,
    ):
        parquet_file = pq.ParquetFile(str(parquet_path))
        field_names = set(parquet_file.schema_arrow.names)

        if "metadata" not in field_names:
            # Do not read vector columns accidentally.
            print(
                "[OCR FRAME WARNING] metadata column missing, skip:",
                parquet_path.name,
            )
            continue

        for batch in parquet_file.iter_batches(
            batch_size=max(
                1024,
                OCR_FRAME_PARQUET_BATCH_SIZE,
            ),
            columns=["metadata"],
        ):
            metadata_array = batch.column(0)
            for value in metadata_array.to_pylist():
                yield value

        if (
            file_index == len(parquet_files)
            or file_index % 20 == 0
        ):
            print(
                f"[OCR FRAME] parquet "
                f"{file_index:,}/{len(parquet_files):,}"
            )


def build_ocr_frame_lookup_from_milvus(
    *,
    force_rebuild: bool = False,
) -> dict[str, Any]:
    """
    Build frame_name -> image_path from Milvus Lite Parquet metadata.

    Despite the legacy function name, this function NEVER opens a
    MilvusClient/query_iterator and therefore NEVER uses localhost gRPC.
    """
    global _OCR_FRAME_LOOKUP_READY
    global _OCR_FRAME_LOOKUP_ERROR

    connection = _open_ocr_frame_lookup_db()

    if (
        not force_rebuild
        and _ocr_frame_lookup_is_valid(connection)
    ):
        count = int(
            connection.execute(
                "SELECT COUNT(*) FROM frame_paths"
            ).fetchone()[0]
        )

        _OCR_FRAME_LOOKUP_READY = count > 0
        _OCR_FRAME_LOOKUP_ERROR = (
            None if _OCR_FRAME_LOOKUP_READY else "empty cache"
        )

        print(
            f"[OCR FRAME] Using Parquet-derived cache: "
            f"{count:,} frames | {OCR_FRAME_LOOKUP_DB}"
        )

        return {
            "ready": _OCR_FRAME_LOOKUP_READY,
            "frames": count,
            "cache_db": str(OCR_FRAME_LOOKUP_DB),
            "collection": OCR_FRAME_MILVUS_COLLECTION,
            "backend": "milvus_parquet",
            "reused": True,
        }

    print(
        "[OCR FRAME] Building frame lookup from "
        "Milvus Lite Parquet metadata (NO gRPC)..."
    )
    print(
        f"[OCR FRAME] Source collection: "
        f"{OCR_FRAME_MILVUS_COLLECTION}"
    )

    connection.execute("DROP TABLE IF EXISTS frame_paths")
    connection.execute("DROP TABLE IF EXISTS lookup_meta")

    connection.execute(
        """
        CREATE TABLE frame_paths (
            frame_name TEXT PRIMARY KEY,
            image_path TEXT NOT NULL,
            video_id TEXT,
            timestamp_sec REAL,
            frame_index INTEGER,
            shot_id INTEGER
        ) WITHOUT ROWID
        """
    )

    connection.execute(
        """
        CREATE TABLE lookup_meta (
            key TEXT PRIMARY KEY,
            value TEXT NOT NULL
        ) WITHOUT ROWID
        """
    )

    connection.execute(
        "INSERT INTO lookup_meta(key, value) VALUES('signature', ?)",
        (_ocr_frame_lookup_cache_signature(),),
    )
    connection.execute(
        "INSERT INTO lookup_meta(key, value) VALUES('complete', '0')"
    )
    connection.commit()

    total_entities = 0
    total_with_image = 0
    inserted = 0
    pending: list[tuple[Any, ...]] = []
    started = time.perf_counter()

    try:
        for raw_metadata in _iter_milvus_metadata_from_parquet():
            total_entities += 1

            metadata = _decode_ocr_milvus_metadata(
                raw_metadata
            )
            raw_image_path = metadata.get("image_path")

            if not raw_image_path:
                continue

            total_with_image += 1
            image_path = _normalize_kaggle_path(
                raw_image_path
            )
            frame_name = Path(image_path).name

            if not frame_name:
                continue

            match = OCR_FRAME_PATTERN.search(frame_name)

            if match is None:
                video_id = (
                    str(metadata.get("video_id") or "")
                    or None
                )
                timestamp_sec = metadata.get(
                    "timestamp",
                    metadata.get("timestamp_sec"),
                )
                frame_index = metadata.get(
                    "frame_idx",
                    metadata.get("frame_index"),
                )
                shot_id = metadata.get("shot_id")
            else:
                video_id = match.group("video_id").upper()
                timestamp_sec = float(
                    match.group("timestamp")
                )
                frame_index = int(
                    match.group("frame_idx")
                )
                shot_id = int(
                    match.group("shot_id")
                )

            pending.append(
                (
                    frame_name,
                    image_path,
                    video_id,
                    timestamp_sec,
                    frame_index,
                    shot_id,
                )
            )

            if len(pending) >= 10000:
                before = connection.total_changes
                connection.executemany(
                    """
                    INSERT OR IGNORE INTO frame_paths(
                        frame_name,
                        image_path,
                        video_id,
                        timestamp_sec,
                        frame_index,
                        shot_id
                    ) VALUES (?, ?, ?, ?, ?, ?)
                    """,
                    pending,
                )
                inserted += (
                    connection.total_changes - before
                )
                pending.clear()
                connection.commit()

            if (
                total_entities % 100000 == 0
            ):
                print(
                    f"[OCR FRAME] scanned={total_entities:,} | "
                    f"with_image={total_with_image:,} | "
                    f"unique={inserted:,}"
                )

        if pending:
            before = connection.total_changes
            connection.executemany(
                """
                INSERT OR IGNORE INTO frame_paths(
                    frame_name,
                    image_path,
                    video_id,
                    timestamp_sec,
                    frame_index,
                    shot_id
                ) VALUES (?, ?, ?, ?, ?, ?)
                """,
                pending,
            )
            inserted += (
                connection.total_changes - before
            )
            pending.clear()

        connection.execute(
            "UPDATE lookup_meta SET value='1' "
            "WHERE key='complete'"
        )
        connection.commit()

        _OCR_FRAME_LOOKUP_READY = inserted > 0
        _OCR_FRAME_LOOKUP_ERROR = (
            None
            if _OCR_FRAME_LOOKUP_READY
            else "Milvus Parquet metadata had no image_path"
        )

        if not _OCR_FRAME_LOOKUP_READY:
            raise RuntimeError(
                "OCR frame lookup is empty. "
                "The selected collection Parquet metadata "
                "does not expose metadata.image_path."
            )

    except Exception as error:
        _OCR_FRAME_LOOKUP_READY = False
        _OCR_FRAME_LOOKUP_ERROR = repr(error)

        try:
            connection.execute(
                "UPDATE lookup_meta SET value='0' "
                "WHERE key='complete'"
            )
            connection.commit()
        except Exception:
            pass

        raise

    elapsed = time.perf_counter() - started

    print(
        f"[OCR FRAME] Ready: {inserted:,} unique frames "
        f"from {total_entities:,} rows in {elapsed:.2f}s"
    )

    return {
        "ready": _OCR_FRAME_LOOKUP_READY,
        "frames": inserted,
        "scanned_entities": total_entities,
        "elapsed_seconds": elapsed,
        "cache_db": str(OCR_FRAME_LOOKUP_DB),
        "collection": OCR_FRAME_MILVUS_COLLECTION,
        "backend": "milvus_parquet",
        "reused": False,
    }


def ensure_ocr_frame_lookup() -> sqlite3.Connection:
    global _OCR_FRAME_LOOKUP_READY

    connection = _open_ocr_frame_lookup_db()

    if not _OCR_FRAME_LOOKUP_READY:
        if _ocr_frame_lookup_is_valid(connection):
            count = int(
                connection.execute(
                    "SELECT COUNT(*) FROM frame_paths"
                ).fetchone()[0]
            )
            _OCR_FRAME_LOOKUP_READY = count > 0
        else:
            build_ocr_frame_lookup_from_milvus()

    return connection


def resolve_ocr_frame_paths(
    frame_names: list[str],
) -> dict[str, str]:
    """Resolve OCR frame basenames with one indexed SQLite query."""
    safe_names = list(
        dict.fromkeys(
            Path(str(name)).name
            for name in frame_names
            if str(name).strip()
        )
    )

    if not safe_names:
        return {}

    connection = ensure_ocr_frame_lookup()
    resolved: dict[str, str] = {}

    for start in range(0, len(safe_names), 500):
        part = safe_names[start:start + 500]
        placeholders = ",".join("?" for _ in part)

        rows = connection.execute(
            f"""
            SELECT frame_name, image_path
            FROM frame_paths
            WHERE frame_name IN ({placeholders})
            """,
            part,
        ).fetchall()

        for row in rows:
            resolved[
                str(row["frame_name"])
            ] = str(row["image_path"])

    return resolved


def resolve_ocr_frame_path(
    frame_name: str,
    video_id: str | None = None,
) -> Optional[str]:
    del video_id

    direct_candidate = Path(
        _normalize_kaggle_path(frame_name)
    )

    if direct_candidate.is_file():
        return str(direct_candidate)

    safe_name = Path(str(frame_name)).name
    return resolve_ocr_frame_paths(
        [safe_name]
    ).get(safe_name)

# ------------------------------------------------------------
# 5. SEARCH V2 -> AIC MULTIMODAL DATAFRAME ADAPTER
# ------------------------------------------------------------

def normalize_ocr_query(value: Any) -> str:
    text = "" if value is None else str(value)
    text = unicodedata.normalize("NFC", text)
    return " ".join(text.split()).strip()


def search_ocr_sqlite(
    query: str,
    top_k: int = OCR_DEFAULT_TOP_K,
    *,
    mode: str = OCR_DEFAULT_MODE,
    candidate_limit: int = OCR_DEFAULT_CANDIDATE_LIMIT,
    collection_id: str | None = None,
    video_id: str | None = None,
    deduplicate_seconds: float = OCR_DEFAULT_DEDUPLICATE_SECONDS,
    require_image: bool = True,
) -> pd.DataFrame:
    """
    Run OCR Search V2 and return the schema expected by Weighted RRF.

    Required fusion columns:
      image_path, rank, score

    Extra columns are retained for API/UI diagnostics.
    """
    normalized_query = normalize_ocr_query(query)

    if not normalized_query:
        return pd.DataFrame()

    normalized_mode = str(mode).strip().lower()
    if normalized_mode not in OCR_VALID_MODES:
        raise ValueError(
            "OCR mode must be 'precision', 'balanced', or 'recall'."
        )

    raw_results = engine_v2.search(
        normalized_query,
        top_k=max(int(top_k), 1),
        candidate_limit=max(int(candidate_limit), int(top_k)),
        mode=normalized_mode,
        collection_id=collection_id,
        video_id=video_id,
        deduplicate_seconds=float(deduplicate_seconds),
    )

    rows: list[dict[str, Any]] = []

    frame_names = [
        str(raw.get("frame") or "")
        for raw in raw_results
    ]
    resolved_frame_paths = resolve_ocr_frame_paths(frame_names)

    for raw in raw_results:
        frame_name = str(raw.get("frame") or "")
        safe_frame_name = Path(frame_name).name
        local_image_path = resolved_frame_paths.get(safe_frame_name)

        if not local_image_path:
            direct_candidate = Path(_normalize_kaggle_path(frame_name))
            if direct_candidate.is_file():
                local_image_path = str(direct_candidate)

        if require_image and not local_image_path:
            continue

        matched_region = raw.get("matched_region") or {}
        score_breakdown = raw.get("score_breakdown") or {}

        timestamp_value = raw.get("timestamp_sec")
        frame_index_value = raw.get("frame_index")
        shot_id_value = raw.get("shot_id")

        rows.append(
            {
                "image_path": (
                    local_image_path
                    if local_image_path
                    else frame_name
                ),
                "rank": int(raw.get("rank") or len(rows) + 1),
                "score": float(raw.get("score") or 0.0),
                "source": "ocr_sqlite",
                "modality": "ocr",
                "ocr_search_version": raw.get(
                    "search_version",
                    "v2",
                ),
                "ocr_search_mode": raw.get(
                    "search_mode",
                    normalized_mode,
                ),
                "collection_id": raw.get("collection_id"),
                "dataset_id": raw.get("dataset_id"),
                "video_id": raw.get("video_id"),
                "timestamp": (
                    float(timestamp_value)
                    if timestamp_value is not None
                    else 0.0
                ),
                "frame_idx": (
                    int(frame_index_value)
                    if frame_index_value is not None
                    else 0
                ),
                "shot_id": (
                    int(shot_id_value)
                    if shot_id_value is not None
                    else 0
                ),
                "frame": frame_name,
                "ocr_text": raw.get(
                    "frame_text_index",
                    "",
                ),
                "ocr_text_recall": raw.get(
                    "frame_text_recall",
                    "",
                ),
                "ocr_snippet": matched_region.get("text") or raw.get(
                    "frame_text_index",
                    "",
                ),
                "ocr_matched_text": matched_region.get("text"),
                "ocr_match_quality": matched_region.get("quality"),
                "ocr_polygon": matched_region.get("polygon"),
                "ocr_det_conf": matched_region.get("det_conf"),
                "ocr_rec_conf": matched_region.get("rec_conf"),
                "ocr_score_breakdown": score_breakdown,
                "ocr_exact_coverage": score_breakdown.get(
                    "exact_coverage"
                ),
                "ocr_approximate_coverage": score_breakdown.get(
                    "approximate_coverage"
                ),
                "ocr_phrase_score": score_breakdown.get("phrase"),
                "ocr_region_score": score_breakdown.get("region"),
            }
        )

    result = pd.DataFrame(rows)

    if result.empty:
        return result

    result = result.reset_index(drop=True)
    result["rank"] = result.index + 1
    return result


def get_ocr_database_stats() -> dict[str, Any]:
    """Return cached/lightweight OCR stats without full-table scans."""
    if not DATABASE_PATH.is_file():
        return {
            "ready": False,
            "db_path": str(DATABASE_PATH),
            "frame_count": 0,
            "region_count": 0,
        }

    return {
        "ready": True,
        "db_path": str(DATABASE_PATH),
        "manifest_path": (
            str(MANIFEST_PATH)
            if MANIFEST_PATH.is_file()
            else None
        ),
        "quick_check": quick_check,
        "frame_count": max(int(ocr_frame_count), 0),
        "row_count": max(int(ocr_frame_count), 0),
        "region_count": max(int(ocr_region_count), 0),
        "dataset_count": int(_ocr_db_metadata.get("dataset_count", 0) or 0),
        "collection_count": int(_ocr_db_metadata.get("collection_count", 0) or 0),
        "video_count": int(_ocr_db_metadata.get("video_count", 0) or 0),
        "default_mode": OCR_DEFAULT_MODE,
        "available_modes": sorted(OCR_VALID_MODES),
        "manifest": OCR_SEARCH_MANIFEST,
        "frame_lookup_backend": "milvus_parquet_sqlite_cache",
        "frame_lookup_collection": OCR_FRAME_MILVUS_COLLECTION,
        "frame_lookup_cache": str(OCR_FRAME_LOOKUP_DB),
        "frame_lookup_ready": _OCR_FRAME_LOOKUP_READY,
        "frame_lookup_error": _OCR_FRAME_LOOKUP_ERROR,
    }


print("OCR Search V2 adapter is ready (Milvus Parquet frame lookup; NO gRPC).")


In [ ]:
# ============================================================
# ASR RETRIEVAL — BM25 + BGE-M3 Dense + Cross-Encoder Reranker
# Data: Kaggle notebook outputs supplied by the user.
#
# Supported layouts (recursive):
#   .../refined_transcripts/aic2026-l25/L25_V001_processed.json
#   .../refined_transcripts/Videos_L21_a/L21_V001_processed.json
#
# IMPORTANT:
# - Dense corpus encoding MUST use GPU on Kaggle. CPU encoding BGE-M3 over
#   ~33k transcript segments can take many hours.
# - Default is CUDA when available; override with ASR_TEXT_DEVICE only if needed.
# - /kaggle/working cache is session-local. To reuse across sessions, publish
#   asr_index_store and mount it, then set ASR_INDEX_INPUT_DIR to that folder.
# ============================================================

import os
import pickle
import random
import re
import unicodedata
from bisect import bisect_left
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Optional

import numpy as np
import orjson
import pandas as pd
import torch
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer

try:
    from underthesea import word_tokenize as _asr_vi_word_tokenize
    _ASR_HAS_UNDERTHESEA = True
except Exception:
    _ASR_HAS_UNDERTHESEA = False


@dataclass(frozen=True)
class ASRConfig:
    transcript_roots: tuple[str, ...] = (
        "/kaggle/input/notebooks/quii29/fork-of-aic26-asr-optimised-enhancement/refined_transcripts",
        "/kaggle/input/notebooks/quii29/aic26-asr-optimised-enhancement/refined_transcripts",
    )
    index_dir: str = os.getenv("ASR_INDEX_DIR", "/kaggle/working/asr_index_store").strip() or "/kaggle/working/asr_index_store"

    dense_model_name: str = "BAAI/bge-m3"
    reranker_model_name: str = "BAAI/bge-reranker-v2-m3"

    # GPU is essential for the one-time BGE-M3 corpus build.
    model_device: str = (
        os.getenv(
            "ASR_TEXT_DEVICE",
            "cuda" if torch.cuda.is_available() else "cpu",
        ).strip()
        or ("cuda" if torch.cuda.is_available() else "cpu")
    )

    rrf_k: int = 60
    pool_size_per_indexer: int = 120
    top_k_rerank: int = 40
    max_rerank_candidates: int = 100
    dedup_window: int = 1

    max_seq_length: int = 512
    dense_batch_size: int = 64 if torch.cuda.is_available() else 8
    rerank_batch_size: int = 64 if torch.cuda.is_available() else 8
    seed: int = 42


ASR_CFG = ASRConfig()


@dataclass
class ASRSegment:
    video_id: str
    index: int
    start_time: float
    end_time: float
    start_frame: int
    end_frame: int
    text: str


@dataclass
class ASRRetrievalResult:
    rank: int
    video_id: str
    segment_index: int
    start_time: float
    end_time: float
    start_frame: int
    end_frame: int
    score: float
    text: str


def _asr_set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


_asr_set_seed(ASR_CFG.seed)


def _asr_safe_int(value: Any, default: int = -1) -> int:
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def _asr_safe_float(value: Any, default: float = 0.0) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def _discover_asr_json_files(
    roots: tuple[str, ...],
) -> tuple[list[Path], list[Path]]:
    existing_roots: list[Path] = []
    json_files: list[Path] = []

    for raw_root in roots:
        root = Path(raw_root)
        if not root.is_dir():
            continue

        existing_roots.append(root)
        json_files.extend(sorted(root.rglob("*_processed.json")))

    # Preserve root priority while removing exact duplicate paths.
    seen: set[str] = set()
    unique_files: list[Path] = []
    for path in json_files:
        key = str(path.resolve())
        if key not in seen:
            seen.add(key)
            unique_files.append(path)

    return existing_roots, unique_files


def _parse_asr_transcript_file(file_path: Path) -> list[ASRSegment]:
    payload = orjson.loads(file_path.read_bytes())
    transcripts = payload.get("transcripts")

    if not isinstance(transcripts, list):
        raise ValueError("missing/invalid 'transcripts' list")

    video_id = file_path.stem.removesuffix("_processed").upper()
    parsed: list[ASRSegment] = []

    for item in transcripts:
        if not isinstance(item, dict):
            continue

        text = str(item.get("refined_transcript", "")).strip()
        if not text:
            continue

        parsed.append(
            ASRSegment(
                video_id=video_id,
                index=_asr_safe_int(item.get("index"), -1),
                start_time=_asr_safe_float(item.get("start_time"), 0.0),
                end_time=_asr_safe_float(item.get("end_time"), 0.0),
                start_frame=_asr_safe_int(item.get("start_index"), -1),
                end_frame=_asr_safe_int(item.get("end_index"), -1),
                text=text,
            )
        )

    return parsed


def load_asr_segments(
    json_files: list[Path],
) -> list[ASRSegment]:
    if not json_files:
        raise FileNotFoundError(
            "No *_processed.json transcripts were found under ASR_CFG.transcript_roots."
        )

    segments_out: list[ASRSegment] = []
    seen_segment_keys: set[tuple[Any, ...]] = set()
    skipped_files = 0

    for file_path in json_files:
        try:
            file_segments = _parse_asr_transcript_file(file_path)
        except (orjson.JSONDecodeError, OSError, ValueError) as error:
            print(f"[ASR] Skip {file_path}: {error}")
            skipped_files += 1
            continue

        for seg in file_segments:
            # Removes accidental overlap if both notebook outputs contain the
            # same video/segment while keeping deterministic root priority.
            key = (
                seg.video_id,
                seg.index,
                round(seg.start_time, 3),
                round(seg.end_time, 3),
                seg.text,
            )
            if key in seen_segment_keys:
                continue
            seen_segment_keys.add(key)
            segments_out.append(seg)

    if not segments_out:
        raise ValueError(
            f"Scanned {len(json_files)} transcript files but extracted no refined_transcript segments."
        )

    print(
        f"[ASR] Loaded {len(segments_out):,} segments from "
        f"{len(json_files) - skipped_files}/{len(json_files)} JSON files."
    )
    return segments_out


_ASR_WORD_RE = re.compile(r"\w+", re.UNICODE)


def asr_tokenize(text: str) -> list[str]:
    normalized = unicodedata.normalize("NFC", str(text)).lower()
    if _ASR_HAS_UNDERTHESEA:
        return _asr_vi_word_tokenize(normalized, format="text").split()
    return _ASR_WORD_RE.findall(normalized)


class ASRSparseIndexer:
    def __init__(self, corpus: list[str]):
        if not corpus:
            raise ValueError("ASR corpus is empty.")
        self.bm25 = BM25Okapi([asr_tokenize(doc) for doc in corpus])

    def search(self, query: str, top_k: int) -> list[tuple[int, float]]:
        scores = self.bm25.get_scores(asr_tokenize(query))
        top_k = min(int(top_k), len(scores))
        if top_k <= 0:
            return []

        indices = np.argsort(scores)[::-1][:top_k]
        return [(int(i), float(scores[i])) for i in indices]

    def save(self, path: Path) -> None:
        with path.open("wb") as handle:
            pickle.dump(self.bm25, handle)

    @classmethod
    def load(cls, path: Path) -> "ASRSparseIndexer":
        instance = cls.__new__(cls)
        with path.open("rb") as handle:
            instance.bm25 = pickle.load(handle)
        return instance


class ASRDenseIndexer:
    """
    BGE-M3 dense retrieval backed by a CPU FAISS IndexFlatIP.
    Keeping FAISS on CPU avoids another large GPU allocation.
    """

    def __init__(self, cfg: ASRConfig):
        requested_device = cfg.model_device
        if requested_device.startswith("cuda") and not torch.cuda.is_available():
            print("[ASR] CUDA requested but unavailable; falling back to CPU.")
            requested_device = "cpu"

        self.device = requested_device
        self.model = SentenceTransformer(
            cfg.dense_model_name,
            device=self.device,
        )
        self.model.max_seq_length = cfg.max_seq_length

        # Do not use FP16 on CPU.
        if self.device.startswith("cuda"):
            self.model.half()

        self.index: faiss.Index | None = None

    def build(self, corpus: list[str], batch_size: int) -> None:
        # On GPU, retry with smaller batches if the visual stack leaves
        # insufficient free VRAM. Never silently fall back to CPU.
        candidate_batch_sizes = [int(batch_size)]
        if self.device.startswith("cuda"):
            for candidate in (48, 32, 24, 16, 8):
                if candidate < candidate_batch_sizes[-1]:
                    candidate_batch_sizes.append(candidate)

        last_error = None
        embeddings = None

        for current_batch_size in candidate_batch_sizes:
            try:
                if self.device.startswith("cuda"):
                    torch.cuda.empty_cache()
                print(
                    f"[ASR] Encoding {len(corpus):,} segments on {self.device} "
                    f"(batch_size={current_batch_size})..."
                )
                embeddings = self.model.encode(
                    corpus,
                    batch_size=current_batch_size,
                    show_progress_bar=True,
                    convert_to_numpy=True,
                    normalize_embeddings=True,
                ).astype(np.float32)
                break
            except torch.cuda.OutOfMemoryError as error:
                last_error = error
                print(
                    f"[ASR] CUDA OOM at batch_size={current_batch_size}; "
                    "retrying with a smaller batch..."
                )
                torch.cuda.empty_cache()

        if embeddings is None:
            raise RuntimeError(
                "Unable to build ASR dense index on GPU even at the smallest "
                "batch size. Free some VRAM or publish/load a prebuilt index."
            ) from last_error

        self.index = faiss.IndexFlatIP(embeddings.shape[1])
        self.index.add(embeddings)

    def search(self, query: str, top_k: int) -> list[tuple[int, float]]:
        if self.index is None:
            raise RuntimeError("ASR dense index is not loaded.")

        query_emb = self.model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(np.float32)

        top_k = min(int(top_k), self.index.ntotal)
        if top_k <= 0:
            return []

        scores, indices = self.index.search(query_emb, top_k)
        return [
            (int(idx), float(score))
            for idx, score in zip(indices[0], scores[0])
            if int(idx) >= 0
        ]

    def save(self, path: Path) -> None:
        if self.index is None:
            raise RuntimeError("No ASR dense index to save.")
        faiss.write_index(self.index, str(path))

    def load(self, path: Path) -> None:
        self.index = faiss.read_index(str(path))


def _asr_reciprocal_ranks(
    results: list[tuple[int, float]],
    k: int,
) -> dict[int, float]:
    return {
        idx: 1.0 / (k + rank)
        for rank, (idx, _) in enumerate(results, start=1)
    }


def _asr_fuse(
    sparse_results: list[tuple[int, float]],
    dense_results: list[tuple[int, float]],
    rrf_k: int,
    top_k: int,
) -> list[tuple[int, float]]:
    sparse_rrf = _asr_reciprocal_ranks(sparse_results, rrf_k)
    dense_rrf = _asr_reciprocal_ranks(dense_results, rrf_k)

    all_indices = sorted(set(sparse_rrf) | set(dense_rrf))
    fused = {
        idx: sparse_rrf.get(idx, 0.0) + dense_rrf.get(idx, 0.0)
        for idx in all_indices
    }
    return sorted(
        fused.items(),
        key=lambda item: item[1],
        reverse=True,
    )[:top_k]


def _asr_dedupe_adjacent(
    ranked: list[tuple[int, float]],
    segments_in: list[ASRSegment],
    window: int,
) -> list[tuple[int, float]]:
    if window < 0:
        return ranked

    kept: list[tuple[int, float]] = []
    kept_keys: list[tuple[str, int]] = []

    for idx, score in ranked:
        seg = segments_in[idx]
        if any(
            seg.video_id == video_id
            and abs(seg.index - segment_index) <= window
            for video_id, segment_index in kept_keys
        ):
            continue

        kept.append((idx, float(score)))
        kept_keys.append((seg.video_id, seg.index))

    return kept


_ASR_INDEX_FILES = (
    "segments.json",
    "bm25.pkl",
    "dense.index",
    "meta.json",
)


def _asr_index_files_exist(index_dir: Path) -> bool:
    return all((index_dir / name).exists() for name in _ASR_INDEX_FILES)


def _asr_index_signature(
    cfg: ASRConfig,
    json_files: list[Path],
) -> dict[str, Any]:
    # Portable signature: do NOT include absolute Kaggle mount paths or mtime.
    # That lets a published index be remounted under /kaggle/input/... and reused.
    file_manifest: list[dict[str, Any]] = []
    for path in json_files:
        try:
            stat = path.stat()
            file_manifest.append(
                {
                    "name": path.name,
                    "size": int(stat.st_size),
                }
            )
        except OSError:
            file_manifest.append(
                {
                    "name": path.name,
                    "size": None,
                }
            )

    file_manifest.sort(key=lambda item: (item["name"], item["size"] or -1))

    return {
        "dense_model_name": cfg.dense_model_name,
        "max_seq_length": cfg.max_seq_length,
        "n_json_files": len(json_files),
        "json_manifest": file_manifest,
    }


def _save_asr_index(
    index_dir: Path,
    segments_in: list[ASRSegment],
    sparse_indexer: ASRSparseIndexer,
    dense_indexer: ASRDenseIndexer,
    signature: dict[str, Any],
) -> None:
    index_dir.mkdir(parents=True, exist_ok=True)

    (index_dir / "segments.json").write_bytes(
        orjson.dumps([asdict(segment) for segment in segments_in])
    )
    (index_dir / "meta.json").write_bytes(orjson.dumps(signature))
    sparse_indexer.save(index_dir / "bm25.pkl")
    dense_indexer.save(index_dir / "dense.index")
    print(f"[ASR] Saved index to {index_dir}")


def _load_asr_index(
    index_dir: Path,
    cfg: ASRConfig,
    expected_signature: dict[str, Any],
) -> tuple[list[ASRSegment], ASRSparseIndexer, ASRDenseIndexer]:
    stored_signature = orjson.loads((index_dir / "meta.json").read_bytes())

    if stored_signature != expected_signature:
        raise ValueError("ASR index signature changed; rebuild required.")

    segments_data = orjson.loads((index_dir / "segments.json").read_bytes())
    loaded_segments = [ASRSegment(**item) for item in segments_data]

    sparse_indexer = ASRSparseIndexer.load(index_dir / "bm25.pkl")
    dense_indexer = ASRDenseIndexer(cfg)
    dense_indexer.load(index_dir / "dense.index")

    print(
        f"[ASR] Loaded {len(loaded_segments):,} segments from cached index."
    )
    return loaded_segments, sparse_indexer, dense_indexer


ASR_EXISTING_ROOTS, ASR_JSON_FILES = _discover_asr_json_files(
    ASR_CFG.transcript_roots
)
ASR_READY = False
ASR_INIT_ERROR: str | None = None
ASR_SEGMENTS: list[ASRSegment] = []
ASR_SPARSE_IDX: ASRSparseIndexer | None = None
ASR_DENSE_IDX: ASRDenseIndexer | None = None
ASR_CROSS_ENCODER: CrossEncoder | None = None



def _resolve_asr_index_dir(cfg: ASRConfig) -> tuple[Path, bool]:
    """
    Returns (index_dir, is_read_only_input_cache).

    Priority:
      1) ASR_INDEX_INPUT_DIR: a published/mounted Kaggle input containing
         segments.json + bm25.pkl + dense.index + meta.json.
      2) cfg.index_dir under /kaggle/working for first-time build.
    """
    mounted = os.getenv("ASR_INDEX_INPUT_DIR", "").strip()
    if mounted:
        mounted_path = Path(mounted)
        if _asr_index_files_exist(mounted_path):
            print(f"[ASR] Using mounted prebuilt index: {mounted_path}")
            return mounted_path, True
        print(
            f"[ASR WARNING] ASR_INDEX_INPUT_DIR={mounted_path} does not contain "
            f"all required index files; falling back to {cfg.index_dir}."
        )

    return Path(cfg.index_dir), False


def initialize_asr_retriever(force_rebuild: bool = False) -> None:
    global ASR_READY
    global ASR_INIT_ERROR
    global ASR_SEGMENTS
    global ASR_SPARSE_IDX
    global ASR_DENSE_IDX
    global ASR_CROSS_ENCODER

    if ASR_READY and not force_rebuild:
        return

    ASR_READY = False
    ASR_INIT_ERROR = None

    if force_rebuild:
        ASR_SEGMENTS = []
        ASR_SPARSE_IDX = None
        ASR_DENSE_IDX = None
        ASR_CROSS_ENCODER = None

    try:
        if not ASR_EXISTING_ROOTS:
            raise FileNotFoundError(
                "None of the configured ASR transcript roots are mounted. "
                "Add both ASR notebook outputs as Kaggle Inputs."
            )

        signature = _asr_index_signature(ASR_CFG, ASR_JSON_FILES)
        index_dir, index_is_input_cache = _resolve_asr_index_dir(ASR_CFG)

        if force_rebuild and index_is_input_cache:
            print(
                "[ASR] force_rebuild=True ignores the read-only mounted cache "
                "and rebuilds into /kaggle/working."
            )
            index_dir = Path(ASR_CFG.index_dir)
            index_is_input_cache = False

        if force_rebuild and index_dir.exists():
            import shutil
            shutil.rmtree(index_dir)

        if _asr_index_files_exist(index_dir):
            try:
                (
                    ASR_SEGMENTS,
                    ASR_SPARSE_IDX,
                    ASR_DENSE_IDX,
                ) = _load_asr_index(
                    index_dir,
                    ASR_CFG,
                    signature,
                )
            except Exception as cache_error:
                print(f"[ASR] Cache invalid: {cache_error}. Rebuilding...")
                import shutil
                if index_is_input_cache:
                    index_dir = Path(ASR_CFG.index_dir)
                    index_is_input_cache = False
                shutil.rmtree(index_dir, ignore_errors=True)

        if not ASR_SEGMENTS:
            ASR_SEGMENTS = load_asr_segments(ASR_JSON_FILES)
            corpus = [segment.text for segment in ASR_SEGMENTS]

            print("[ASR] Building BM25 index...")
            ASR_SPARSE_IDX = ASRSparseIndexer(corpus)

            print(
                f"[ASR] Building BGE-M3 dense index on {ASR_CFG.model_device!r}..."
            )
            ASR_DENSE_IDX = ASRDenseIndexer(ASR_CFG)
            ASR_DENSE_IDX.build(
                corpus,
                batch_size=ASR_CFG.dense_batch_size,
            )

            _save_asr_index(
                index_dir=index_dir,
                segments_in=ASR_SEGMENTS,
                sparse_indexer=ASR_SPARSE_IDX,
                dense_indexer=ASR_DENSE_IDX,
                signature=signature,
            )

        print(
            f"[ASR] Loading reranker {ASR_CFG.reranker_model_name} "
            f"on {ASR_CFG.model_device!r}..."
        )
        ASR_CROSS_ENCODER = CrossEncoder(
            ASR_CFG.reranker_model_name,
            device=ASR_CFG.model_device,
        )
        if ASR_CFG.model_device.startswith("cuda"):
            ASR_CROSS_ENCODER.model.half()

        ASR_READY = True
        print(
            f"[ASR] Ready: {len(ASR_SEGMENTS):,} segments, "
            f"{len({segment.video_id for segment in ASR_SEGMENTS}):,} videos."
        )

    except Exception as error:
        ASR_INIT_ERROR = f"{type(error).__name__}: {error}"
        print(f"[ASR WARNING] Retriever is not ready: {ASR_INIT_ERROR}")


def retrieve_asr(
    query: str,
    top_k: int = 20,
    cfg: ASRConfig = ASR_CFG,
) -> list[ASRRetrievalResult]:
    query = str(query).strip()
    if not query:
        raise ValueError("ASR query is empty.")

    if not ASR_READY:
        initialize_asr_retriever()

    if (
        not ASR_READY
        or ASR_SPARSE_IDX is None
        or ASR_DENSE_IDX is None
        or ASR_CROSS_ENCODER is None
    ):
        raise RuntimeError(
            "ASR retriever is unavailable. "
            + (ASR_INIT_ERROR or "Unknown initialization error.")
        )

    requested_top_k = max(1, int(top_k))
    rerank_limit = min(
        max(cfg.top_k_rerank, requested_top_k),
        cfg.max_rerank_candidates,
    )
    pool_size = max(cfg.pool_size_per_indexer, rerank_limit)

    sparse_results = ASR_SPARSE_IDX.search(query, pool_size)
    dense_results = ASR_DENSE_IDX.search(query, pool_size)
    candidates = _asr_fuse(
        sparse_results,
        dense_results,
        cfg.rrf_k,
        rerank_limit,
    )

    if not candidates:
        return []

    candidate_indices = [idx for idx, _ in candidates]
    pairs = [
        [query, ASR_SEGMENTS[idx].text]
        for idx in candidate_indices
    ]

    ce_scores = ASR_CROSS_ENCODER.predict(
        pairs,
        batch_size=cfg.rerank_batch_size,
        show_progress_bar=False,
    )

    ranked = sorted(
        zip(candidate_indices, ce_scores),
        key=lambda item: float(item[1]),
        reverse=True,
    )
    ranked = _asr_dedupe_adjacent(
        ranked,
        ASR_SEGMENTS,
        cfg.dedup_window,
    )
    ranked = ranked[: min(requested_top_k, cfg.max_rerank_candidates)]

    results: list[ASRRetrievalResult] = []
    for rank, (idx, score) in enumerate(ranked, start=1):
        seg = ASR_SEGMENTS[idx]
        results.append(
            ASRRetrievalResult(
                rank=rank,
                video_id=seg.video_id,
                segment_index=seg.index,
                start_time=seg.start_time,
                end_time=seg.end_time,
                start_frame=seg.start_frame,
                end_frame=seg.end_frame,
                score=float(score),
                text=seg.text,
            )
        )
    return results


# ------------------------------------------------------------
# ASR -> representative keyframe mapping
# ------------------------------------------------------------

_ASR_FRAME_LOOKUP: dict[str, dict[str, Any]] = {}
_ASR_FRAME_LOOKUP_READY = False
_ASR_FRAME_LOOKUP_ERROR: str | None = None


def _normalize_frame_path_for_asr(path_value: Any) -> str:
    return str(path_value).replace(
        "https://www.kaggle.com/kaggle/input/",
        "/kaggle/input/",
        1,
    )


def build_asr_frame_lookup(force: bool = False) -> None:
    """
    Build a lightweight lookup from existing visual embedding metadata.
    No .npy embeddings are loaded.
    """
    global _ASR_FRAME_LOOKUP
    global _ASR_FRAME_LOOKUP_READY
    global _ASR_FRAME_LOOKUP_ERROR

    if _ASR_FRAME_LOOKUP_READY and not force:
        return

    _ASR_FRAME_LOOKUP = {}
    _ASR_FRAME_LOOKUP_READY = False
    _ASR_FRAME_LOOKUP_ERROR = None

    records_by_video: dict[str, list[tuple[int, float, int, str]]] = {}
    metadata_files: list[Path] = []

    # BGE metadata already covers the same frame universe used by the UI.
    for raw_dir in BGE_DIR:
        directory = Path(raw_dir)
        if directory.is_dir():
            metadata_files.extend(
                sorted(directory.glob("metadata_chunk_*.csv"))
            )

    if not metadata_files:
        _ASR_FRAME_LOOKUP_ERROR = (
            "No metadata_chunk_*.csv files were found in BGE_DIR."
        )
        print(f"[ASR FRAME WARNING] {_ASR_FRAME_LOOKUP_ERROR}")
        return

    seen_paths: set[str] = set()

    for metadata_file in metadata_files:
        try:
            header = pd.read_csv(metadata_file, nrows=0)
            if "image_path" not in header.columns:
                continue

            chunk = pd.read_csv(
                metadata_file,
                usecols=["image_path"],
                dtype={"image_path": "string"},
            )
        except Exception as error:
            print(f"[ASR FRAME] Skip {metadata_file.name}: {error}")
            continue

        for raw_path in chunk["image_path"].dropna().tolist():
            image_path = _normalize_frame_path_for_asr(raw_path)
            if image_path in seen_paths:
                continue
            seen_paths.add(image_path)

            match = FRAME_PATTERN.search(Path(image_path).name)
            if match is None:
                continue

            video_id = match.group("video_id").upper()
            frame_idx = int(match.group("frame_idx"))
            timestamp = float(match.group("timestamp"))
            shot_id = int(match.group("shot_id"))

            records_by_video.setdefault(video_id, []).append(
                (frame_idx, timestamp, shot_id, image_path)
            )

    for video_id, records in records_by_video.items():
        records.sort(key=lambda item: (item[0], item[1], item[3]))
        _ASR_FRAME_LOOKUP[video_id] = {
            "records": records,
            "frame_indices": [item[0] for item in records],
            "timestamps": [item[1] for item in records],
        }

    _ASR_FRAME_LOOKUP_READY = bool(_ASR_FRAME_LOOKUP)
    if not _ASR_FRAME_LOOKUP_READY:
        _ASR_FRAME_LOOKUP_ERROR = (
            "Frame metadata was read but no filenames matched FRAME_PATTERN."
        )

    print(
        f"[ASR FRAME] Indexed representative frames for "
        f"{len(_ASR_FRAME_LOOKUP):,} videos."
    )


def _nearest_sorted_position(
    sorted_values: list[float] | list[int],
    target: float,
) -> int | None:
    if not sorted_values:
        return None

    pos = bisect_left(sorted_values, target)
    if pos <= 0:
        return 0
    if pos >= len(sorted_values):
        return len(sorted_values) - 1

    left = pos - 1
    right = pos
    return (
        left
        if abs(float(sorted_values[left]) - target)
        <= abs(float(sorted_values[right]) - target)
        else right
    )


def resolve_asr_representative_frame(
    result: ASRRetrievalResult,
) -> dict[str, Any] | None:
    if not _ASR_FRAME_LOOKUP_READY:
        build_asr_frame_lookup()

    lookup = _ASR_FRAME_LOOKUP.get(result.video_id.upper())
    if not lookup:
        return None

    records = lookup["records"]
    position: int | None = None

    valid_start_frame = result.start_frame >= 0
    valid_end_frame = result.end_frame >= 0

    if valid_start_frame or valid_end_frame:
        if valid_start_frame and valid_end_frame:
            target_frame = (result.start_frame + result.end_frame) / 2.0
        elif valid_start_frame:
            target_frame = float(result.start_frame)
        else:
            target_frame = float(result.end_frame)

        position = _nearest_sorted_position(
            lookup["frame_indices"],
            target_frame,
        )

    if position is None:
        target_time = (
            (result.start_time + result.end_time) / 2.0
            if result.end_time >= result.start_time
            else result.start_time
        )
        position = _nearest_sorted_position(
            lookup["timestamps"],
            target_time,
        )

    if position is None:
        return None

    frame_idx, timestamp, shot_id, image_path = records[position]
    return {
        "image_path": image_path,
        "video_id": result.video_id.upper(),
        "timestamp": float(timestamp),
        "frame_idx": int(frame_idx),
        "shot_id": int(shot_id),
    }


def search_asr_dataframe(
    query: str,
    top_k: int = 20,
) -> pd.DataFrame:
    """
    ASR retrieval adapter for the shared multimodal RRF pipeline.
    Each transcript hit is attached to the nearest existing keyframe.
    """
    retrieved = retrieve_asr(query, top_k=top_k)
    rows: list[dict[str, Any]] = []
    missing_frame_matches = 0

    for item in retrieved:
        representative = resolve_asr_representative_frame(item)
        if representative is None:
            missing_frame_matches += 1
            continue

        rows.append(
            {
                **representative,
                "rank": item.rank,
                "score": item.score,
                "asr_score": item.score,
                "asr_text": item.text,
                "asr_segment_index": item.segment_index,
                "asr_start_time": item.start_time,
                "asr_end_time": item.end_time,
                "asr_start_frame": item.start_frame,
                "asr_end_frame": item.end_frame,
            }
        )

    if missing_frame_matches:
        print(
            f"[ASR FRAME WARNING] {missing_frame_matches} transcript hit(s) "
            "had no representative visual frame and were skipped."
        )

    if not rows:
        return pd.DataFrame()

    result_df = pd.DataFrame(rows)
    result_df = (
        result_df
        .sort_values(["rank", "score"], ascending=[True, False])
        .drop_duplicates(subset=["image_path"], keep="first")
        .reset_index(drop=True)
    )
    result_df["rank"] = range(1, len(result_df) + 1)
    return result_df


def get_asr_stats() -> dict[str, Any]:
    return {
        "ready": ASR_READY,
        "initialization_error": ASR_INIT_ERROR,
        "device": ASR_CFG.model_device,
        "dense_model": ASR_CFG.dense_model_name,
        "reranker_model": ASR_CFG.reranker_model_name,
        "configured_roots": list(ASR_CFG.transcript_roots),
        "mounted_roots": [str(path) for path in ASR_EXISTING_ROOTS],
        "json_files": len(ASR_JSON_FILES),
        "segments": len(ASR_SEGMENTS),
        "videos": len({segment.video_id for segment in ASR_SEGMENTS}),
        "frame_lookup_ready": _ASR_FRAME_LOOKUP_READY,
        "frame_lookup_videos": len(_ASR_FRAME_LOOKUP),
        "frame_lookup_error": _ASR_FRAME_LOOKUP_ERROR,
    }


# Initialize once here. With CUDA this builds quickly enough for a first run.
# If ASR_INDEX_INPUT_DIR points to a published prebuilt index, this only loads it.
print(
    f"[ASR CONFIG] device={ASR_CFG.model_device}, "
    f"dense_batch={ASR_CFG.dense_batch_size}, "
    f"index_dir={ASR_CFG.index_dir}"
)
initialize_asr_retriever()

print(
    "[ASR] Tokenizer:",
    "underthesea" if _ASR_HAS_UNDERTHESEA else "regex fallback",
)

In [ ]:
# ============================================================
# MULTIMODAL RETRIEVAL:
# VISUAL + OCR SQLITE + ASR HYBRID
# Run AFTER the OCR SQLite cell and BEFORE FastAPI.
# ============================================================

from typing import Any, Literal, Optional
from concurrent.futures import ThreadPoolExecutor
from time import perf_counter
import os

# GPU query encoders remain serial; Milvus scans overlap in worker threads.
VISUAL_DB_PARALLEL = os.getenv(
    "AIC_VISUAL_DB_PARALLEL",
    "1",
).strip().lower() not in {"0", "false", "no"}

VISUAL_DB_WORKERS = max(
    1,
    min(4, int(os.getenv("AIC_VISUAL_DB_WORKERS", "4"))),
)

# "all" = original 4-model ensemble.
# "fast" = BGE + MetaCLIP + BEiT-3 (skip slow Perception branch).
VISUAL_SEARCH_PROFILE = os.getenv(
    "AIC_VISUAL_SEARCH_PROFILE",
    "all",
).strip().lower()
if VISUAL_SEARCH_PROFILE not in {"all", "fast"}:
    VISUAL_SEARCH_PROFILE = "all"

VisualSearchModel = Literal[
    "bge",
    "metaclip",
    "beit3",
    "perception",
]

DEFAULT_VISUAL_MODELS: tuple[VisualSearchModel, ...] = (
    "bge",
    "metaclip",
    "beit3",
    "perception",
)

_old_visual_executor = globals().get("_VISUAL_DB_EXECUTOR")
if _old_visual_executor is not None:
    try:
        _old_visual_executor.shutdown(
            wait=False,
            cancel_futures=True,
        )
    except Exception:
        pass

_VISUAL_DB_EXECUTOR = ThreadPoolExecutor(
    max_workers=VISUAL_DB_WORKERS,
    thread_name_prefix="aic-milvus",
)
_LAST_VISUAL_TIMINGS: dict[str, Any] = {}



def normalize_candidate_path(path_value: Any) -> str:
    """
    Normalize Kaggle URL paths before fusion so the same frame from
    visual retrieval and OCR retrieval receives one shared RRF key.
    """
    path = str(path_value)
    return path.replace(
        "https://www.kaggle.com/kaggle/input/",
        "/kaggle/input/",
        1,
    )


def compute_weighted_rrf(
    named_dfs: dict[str, pd.DataFrame],
    weights: Optional[dict[str, float]] = None,
    k: int = 60,
) -> pd.DataFrame:
    """
    Weighted Reciprocal Rank Fusion across visual, OCR and ASR branches.

    Fusion key = normalized representative image_path so all modalities can
    vote for the same frame. Modality-specific metadata is merged instead of
    being overwritten, which lets the API return OCR text and ASR transcript
    together when they hit the same frame.
    """
    weights = weights or {}

    rrf_scores: dict[str, float] = {}
    metadata: dict[str, dict[str, Any]] = {}
    source_hits: dict[str, list[str]] = {}

    for source_name, df in named_dfs.items():
        if df is None or df.empty:
            continue

        weight = float(weights.get(source_name, 1.0))
        seen_source_paths: set[str] = set()

        for _, row in df.iterrows():
            if "image_path" not in row or pd.isna(row["image_path"]):
                continue

            image_path = normalize_candidate_path(row["image_path"])
            if not image_path or image_path in seen_source_paths:
                continue
            seen_source_paths.add(image_path)

            rank = int(row.get("rank", 1))
            row_metadata = row.to_dict()
            row_metadata["image_path"] = image_path

            if image_path not in metadata:
                metadata[image_path] = row_metadata
            else:
                # Keep the existing representative-frame fields, but merge
                # modality-specific evidence from the incoming branch.
                for key, value in row_metadata.items():
                    if key == "image_path":
                        continue

                    is_auxiliary = (
                        key.startswith("ocr_")
                        or key.startswith("asr_")
                    )
                    existing_value = metadata[image_path].get(key)

                    if is_auxiliary:
                        if value is None:
                            continue

                        if isinstance(value, np.ndarray):
                            if value.size == 0:
                                continue
                        elif isinstance(
                            value,
                            (list, tuple, dict, set),
                        ):
                            if len(value) == 0:
                                continue
                        elif (
                            isinstance(value, float)
                            and np.isnan(value)
                        ):
                            continue

                        metadata[image_path][key] = value

                    elif (
                        key not in metadata[image_path]
                        or existing_value is None
                        or (
                            isinstance(existing_value, float)
                            and np.isnan(existing_value)
                        )
                    ):
                        metadata[image_path][key] = value

            source_hits.setdefault(image_path, [])
            if source_name not in source_hits[image_path]:
                source_hits[image_path].append(source_name)

            rrf_scores[image_path] = (
                rrf_scores.get(image_path, 0.0)
                + weight / (k + rank)
            )

    if not rrf_scores:
        return pd.DataFrame()

    fused = pd.DataFrame(
        [
            {
                **metadata[image_path],
                "image_path": image_path,
                "rrf_score": score,
                "score": score,
                "fusion_sources": source_hits[image_path],
            }
            for image_path, score in rrf_scores.items()
        ]
    )

    fused = fused.sort_values(
        "rrf_score",
        ascending=False,
    ).reset_index(drop=True)
    fused["final_rank"] = range(1, len(fused) + 1)
    return fused


def search_all_visual_models(
    visual_query_en: str,
    top_k: int = 100,
    visual_models: Optional[list[str]] = None,
) -> dict[str, pd.DataFrame]:
    """
    Encode on GPU serially, but overlap read-only Milvus searches.

    Perception is submitted first because it is the slowest collection in
    current logs, giving its FLAT scan the largest overlap window.
    """
    global _LAST_VISUAL_TIMINGS

    visual_query_en = visual_query_en.strip()
    if not visual_query_en:
        _LAST_VISUAL_TIMINGS = {}
        return {}

    # Initialize/load handles on this thread before executor workers use them.
    preload_milvus_collections()

    specs = [
        (
            "text_perception",
            encode_perception_text,
            PERCEPTION_DIR,
            "perception_image_emb_chunk",
        ),
        (
            "text_bge",
            encode_bge_text,
            BGE_DIR,
            "bge_image_emb_chunk",
        ),
        (
            "text_metaclip",
            encode_metaclip_text,
            METACLIP_DIR,
            "metaclip_image_emb_chunk",
        ),
        (
            "text_beit3",
            encode_beit3_text,
            BEIT3_DIR,
            "beit3_image_emb_chunk",
        ),
    ]

    selected_visual_models = (
        [
            str(model_name).strip().lower()
            for model_name in visual_models
        ]
        if visual_models is not None
        else None
    )

    if selected_visual_models is not None:
        allowed_visual_models = set(DEFAULT_VISUAL_MODELS)
        invalid_visual_models = [
            model_name
            for model_name in selected_visual_models
            if model_name not in allowed_visual_models
        ]
        if invalid_visual_models:
            raise ValueError(
                "Unsupported visual model(s): "
                + ", ".join(invalid_visual_models)
            )

        selected_branch_names = {
            f"text_{model_name}"
            for model_name in selected_visual_models
        }
        specs = [
            spec
            for spec in specs
            if spec[0] in selected_branch_names
        ]
    elif VISUAL_SEARCH_PROFILE == "fast":
        specs = [
            spec
            for spec in specs
            if spec[0] != "text_perception"
        ]

    print(
        "[VISUAL SELECTOR] "
        f"requested={selected_visual_models!r} | "
        f"branches={[spec[0] for spec in specs]!r}"
    )

    if not specs:
        _LAST_VISUAL_TIMINGS = {
            "profile": VISUAL_SEARCH_PROFILE,
            "parallel": False,
            "workers": VISUAL_DB_WORKERS,
            "encode_ms": {},
            "milvus_search_ms": {},
            "milvus_sum_ms": 0.0,
            "visual_wall_ms": 0.0,
            "branches": [],
        }
        return {}

    total_started = perf_counter()
    encode_ms: dict[str, float] = {}
    results: dict[str, pd.DataFrame] = {}
    futures = {}

    if VISUAL_DB_PARALLEL and len(specs) > 1:
        for branch_name, encode_fn, emb_dirs, file_prefix in specs:
            encode_started = perf_counter()
            query_embedding = encode_fn(visual_query_en)
            encode_ms[branch_name] = round(
                (perf_counter() - encode_started) * 1000.0,
                2,
            )

            futures[branch_name] = _VISUAL_DB_EXECUTOR.submit(
                search_embeddings,
                query_embedding,
                emb_dirs,
                file_prefix,
                top_k,
            )

        # Searches are already running. Read results in stable branch order.
        for branch_name, *_ in specs:
            try:
                results[branch_name] = futures[branch_name].result()
            except Exception as error:
                # Correctness-first fallback if a runtime cannot handle
                # concurrent direct-engine reads.
                print(
                    f"[VISUAL PARALLEL WARNING] {branch_name}: "
                    f"{error!r}; retrying sequentially."
                )
                spec = next(
                    item for item in specs
                    if item[0] == branch_name
                )
                _, encode_fn, emb_dirs, file_prefix = spec
                retry_started = perf_counter()
                query_embedding = encode_fn(visual_query_en)
                encode_ms[f"{branch_name}_retry"] = round(
                    (perf_counter() - retry_started) * 1000.0,
                    2,
                )
                results[branch_name] = search_embeddings(
                    query_embedding,
                    emb_dirs,
                    file_prefix,
                    top_k=top_k,
                )
    else:
        for branch_name, encode_fn, emb_dirs, file_prefix in specs:
            encode_started = perf_counter()
            query_embedding = encode_fn(visual_query_en)
            encode_ms[branch_name] = round(
                (perf_counter() - encode_started) * 1000.0,
                2,
            )
            results[branch_name] = search_embeddings(
                query_embedding,
                emb_dirs,
                file_prefix,
                top_k=top_k,
            )

    search_ms = {
        branch_name: round(
            float(
                dataframe.attrs.get(
                    "milvus_search_seconds",
                    0.0,
                )
            ) * 1000.0,
            2,
        )
        for branch_name, dataframe in results.items()
    }

    visual_wall_ms = round(
        (perf_counter() - total_started) * 1000.0,
        2,
    )

    _LAST_VISUAL_TIMINGS = {
        "profile": VISUAL_SEARCH_PROFILE,
        "parallel": bool(
            VISUAL_DB_PARALLEL and len(specs) > 1
        ),
        "workers": VISUAL_DB_WORKERS,
        "encode_ms": encode_ms,
        "milvus_search_ms": search_ms,
        "milvus_sum_ms": round(sum(search_ms.values()), 2),
        "visual_wall_ms": visual_wall_ms,
        "branches": [name for name, *_ in specs],
    }

    print(
        "[VISUAL TIMING] "
        f"wall={visual_wall_ms:.1f}ms | "
        f"Milvus sum={sum(search_ms.values()):.1f}ms | "
        f"parallel={_LAST_VISUAL_TIMINGS['parallel']} | "
        f"profile={VISUAL_SEARCH_PROFILE}"
    )
    return results



def finalize_fused_results(
    fused_results: pd.DataFrame,
    exclude_image_path: Optional[str] = None,
    max_results: int = 20,
) -> pd.DataFrame:
    if fused_results.empty:
        return fused_results

    filtered = fused_results.copy()

    if exclude_image_path:
        excluded = normalize_candidate_path(
            Path(exclude_image_path)
        )
        filtered = filtered[
            filtered["image_path"].astype(str) != excluded
        ].reset_index(drop=True)

    fixed = fix_results_metadata(filtered)

    return diversify_results(
        fixed,
        min_gap_seconds=5.0,
        max_per_video=5,
        max_results=max_results,
    )



QueryMode = Literal[
    "visual",
    "ocr",
    "asr",
    "visual+ocr",
    "visual+asr",
    "ocr+asr",
    "visual+ocr+asr",
]


FUSION_WEIGHTS = {
    "text_bge": 1.0,
    "text_metaclip": 1.0,
    "text_beit3": 1.0,
    "text_perception": 1.0,
    "ocr_sqlite": 1.75,
    # Tunable without changing ASR internal scores because RRF uses rank.
    "asr_hybrid": float(os.getenv("ASR_RRF_WEIGHT", "1.50")),
}


def normalize_asr_query(text: str) -> str:
    return " ".join(str(text or "").split()).strip()



def detect_query_mode(
    visual_query: str = "",
    ocr_query: str = "",
    asr_query: str = "",
) -> QueryMode:
    """
    Route every non-empty combination of Visual / OCR / ASR.
    """
    active: list[str] = []

    if (visual_query or "").strip():
        active.append("visual")
    if normalize_ocr_query(ocr_query):
        active.append("ocr")
    if normalize_asr_query(asr_query):
        active.append("asr")

    if not active:
        raise ValueError(
            "Provide visual_query, ocr_query, asr_query, or a combination."
        )

    return "+".join(active)  # type: ignore[return-value]


def run_multimodal_search(
    visual_query: str = "",
    ocr_query: str = "",
    asr_query: str = "",
    query_mode: Optional[QueryMode] = None,
    visual_models: Optional[list[str]] = None,
    visual_top_k: int = 300,
    ocr_top_k: int = 100,
    asr_top_k: int = 100,
    max_results: int = 20,
):
    """
    Multimodal retrieval without query reconstruction.

    Visual:
      The raw visual query is encoded directly by the selected visual retrievers.
    OCR:
      SQLite FTS5.
    ASR:
      BM25 + BGE-M3 -> RRF -> CrossEncoder rerank -> representative frame.
    Final:
      Weighted RRF across whichever modalities are active.

    Returns:
      search_meta: effective queries + timing information.
      final_results: fused result DataFrame.
    """
    resolved_visual_query = str(visual_query or "").strip()
    resolved_ocr_query = normalize_ocr_query(ocr_query)
    resolved_asr_query = normalize_asr_query(asr_query)

    mode = query_mode or detect_query_mode(
        visual_query=resolved_visual_query,
        ocr_query=resolved_ocr_query,
        asr_query=resolved_asr_query,
    )
    active_modalities = set(str(mode).split("+"))

    if (
        "visual" in active_modalities
        and visual_models is not None
        and not visual_models
    ):
        raise ValueError(
            "Select at least one Visual model before searching."
        )

    branches: dict[str, pd.DataFrame] = {}
    pipeline_started = perf_counter()
    stage_timings: dict[str, Any] = {}

    # --------------------------------------------------------
    # Visual branch: use the query exactly as provided.
    # --------------------------------------------------------
    if "visual" in active_modalities:
        if not resolved_visual_query:
            raise ValueError(
                f"Mode {mode!r} requires a non-empty visual_query."
            )

        visual_started = perf_counter()
        branches.update(
            search_all_visual_models(
                visual_query_en=resolved_visual_query,
                top_k=visual_top_k,
                visual_models=visual_models,
            )
        )
        stage_timings["visual_ms"] = round(
            (perf_counter() - visual_started) * 1000.0,
            2,
        )
        stage_timings["visual_detail"] = dict(
            _LAST_VISUAL_TIMINGS
        )

    # --------------------------------------------------------
    # OCR branch
    # --------------------------------------------------------
    if "ocr" in active_modalities:
        if not resolved_ocr_query:
            raise ValueError(
                f"Mode {mode!r} requires a non-empty ocr_query."
            )

        ocr_started = perf_counter()
        ocr_results = search_ocr_sqlite(
            resolved_ocr_query,
            top_k=ocr_top_k,
        )
        stage_timings["ocr_ms"] = round(
            (perf_counter() - ocr_started) * 1000.0,
            2,
        )
        if not ocr_results.empty:
            branches["ocr_sqlite"] = ocr_results

    # --------------------------------------------------------
    # ASR branch
    # --------------------------------------------------------
    if "asr" in active_modalities:
        if not resolved_asr_query:
            raise ValueError(
                f"Mode {mode!r} requires a non-empty asr_query."
            )

        asr_started = perf_counter()
        asr_results = search_asr_dataframe(
            resolved_asr_query,
            top_k=asr_top_k,
        )
        stage_timings["asr_ms"] = round(
            (perf_counter() - asr_started) * 1000.0,
            2,
        )
        if not asr_results.empty:
            branches["asr_hybrid"] = asr_results

    search_meta: dict[str, Any] = {
        "query_mode": mode,
        "effective_visual_query": (
            resolved_visual_query
            if "visual" in active_modalities
            else ""
        ),
        "effective_ocr_query": (
            resolved_ocr_query
            if "ocr" in active_modalities
            else ""
        ),
        "effective_asr_query": (
            resolved_asr_query
            if "asr" in active_modalities
            else ""
        ),
    }

    if not branches:
        stage_timings["pipeline_total_ms"] = round(
            (perf_counter() - pipeline_started) * 1000.0,
            2,
        )
        search_meta["timings"] = stage_timings
        return search_meta, pd.DataFrame()

    fusion_started = perf_counter()
    fused = compute_weighted_rrf(
        branches,
        weights=FUSION_WEIGHTS,
        k=60,
    )
    stage_timings["fusion_ms"] = round(
        (perf_counter() - fusion_started) * 1000.0,
        2,
    )

    finalize_started = perf_counter()
    final_results = finalize_fused_results(
        fused_results=fused,
        max_results=max_results,
    )
    stage_timings["finalize_ms"] = round(
        (perf_counter() - finalize_started) * 1000.0,
        2,
    )
    stage_timings["pipeline_total_ms"] = round(
        (perf_counter() - pipeline_started) * 1000.0,
        2,
    )
    search_meta["timings"] = stage_timings

    return search_meta, final_results


print("Visual + OCR SQLite + ASR hybrid search helpers are ready.")


In [ ]:
# ============================================================
# VERIFYQUEUE / ACASE COMPATIBILITY LAYER
# Add AFTER Cell 11 and BEFORE ACASE cells
# ============================================================

import re
import sqlite3
import numpy as np
import pandas as pd

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Optional


# ------------------------------------------------------------
# 1. DATA STRUCTURES
# ------------------------------------------------------------

@dataclass
class VerifyFrame:
    image_path: str
    video_id: str
    timestamp: float
    frame_index: int
    shot_id: int | None

    visual_score: float = 0.0
    visual_score_smooth: float = 0.0
    is_anchor: bool = False


@dataclass
class VerifySegment:
    video_id: str
    anchor_path: str
    anchor_timestamp: float

    start_time: float
    end_time: float

    threshold: float
    frames: list[VerifyFrame]


# ------------------------------------------------------------
# 2. MODEL CONFIG
# ------------------------------------------------------------

MODEL_CONFIG = {

    "bge": {
        "dirs_name": "BGE_DIR",
        "prefix": "bge_image_emb_chunk",
        "query_encoder_name": "encode_bge_text",
    },

    "metaclip": {
        "dirs_name": "METACLIP_DIR",
        "prefix": "metaclip_image_emb_chunk",
        "query_encoder_name": "encode_metaclip_text",
    },

    "beit3": {
        "dirs_name": "BEIT3_DIR",
        "prefix": "beit3_image_emb_chunk",
        "query_encoder_name": "encode_beit3_text",
    },

    "perception": {
        "dirs_name": "PERCEPTION_DIR",
        "prefix": "perception_image_emb_chunk",
        "query_encoder_name": "encode_perception_text",
    },
}


VERIFY_LOOKUP_DB = Path(
    "/kaggle/working/verify_embedding_lookup.sqlite"
)


# ------------------------------------------------------------
# 3. BASIC HELPERS
# ------------------------------------------------------------

def _normalize_path(value):
    """
    Chuẩn hóa Kaggle frame path.
    """

    value = str(value)

    # Notebook hiện tại đã có canonical_frame_path()
    if "canonical_frame_path" in globals():
        try:
            return str(
                canonical_frame_path(value)
            )
        except Exception:
            pass

    return value.replace(
        "https://www.kaggle.com/kaggle/input/",
        "/kaggle/input/",
        1,
    )


def _safe_float(value, default=0.0):
    try:
        return float(value)
    except Exception:
        return float(default)


def _safe_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return int(default)


def _trailing_number(path: Path):
    """
    metadata_chunk_12.csv -> 12
    bge_image_emb_chunk_12.npy -> 12
    """

    match = re.search(
        r"(\d+)(?=\.[^.]+$)",
        path.name,
    )

    if match is None:
        return None

    return int(
        match.group(1)
    )


# ------------------------------------------------------------
# 4. DISCOVER METADATA + EMBEDDING CHUNKS
# ------------------------------------------------------------

def _discover_chunk_pairs(
    root: Path,
    embedding_prefix: str,
):

    metadata_files = sorted(
        root.glob(
            "metadata_chunk_*.csv"
        )
    )

    embedding_files = sorted(
        root.glob(
            f"{embedding_prefix}_*.npy"
        )
    )

    emb_by_chunk = {}

    for emb_file in embedding_files:

        chunk_id = _trailing_number(
            emb_file
        )

        if chunk_id is not None:

            emb_by_chunk[
                chunk_id
            ] = emb_file

    pairs = []

    for metadata_file in metadata_files:

        chunk_id = _trailing_number(
            metadata_file
        )

        if chunk_id is None:
            continue

        emb_file = emb_by_chunk.get(
            chunk_id
        )

        if emb_file is not None:

            pairs.append(
                (
                    metadata_file,
                    emb_file,
                )
            )

    return pairs


# ------------------------------------------------------------
# 5. SQLITE LOOKUP
#
# filename
#    ->
# embedding .npy file
#    +
# row index
# ------------------------------------------------------------

def _open_verify_lookup():

    VERIFY_LOOKUP_DB.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    con = sqlite3.connect(
        str(
            VERIFY_LOOKUP_DB
        )
    )

    con.execute(
        """
        CREATE TABLE IF NOT EXISTS embedding_lookup (

            model TEXT NOT NULL,

            filename TEXT NOT NULL,

            emb_file TEXT NOT NULL,

            row_idx INTEGER NOT NULL,

            image_path TEXT,

            video_id TEXT,

            timestamp REAL,

            shot_id INTEGER,

            PRIMARY KEY (
                model,
                filename
            )
        )
        """
    )

    con.execute(
        """
        CREATE INDEX IF NOT EXISTS
        idx_embedding_lookup_filename

        ON embedding_lookup(filename)
        """
    )

    return con


# ------------------------------------------------------------
# 6. BUILD LOOKUP
# ------------------------------------------------------------

def build_verify_embedding_lookup(
    models=(
        "bge",
        "metaclip",
        "beit3",
        "perception",
    ),
    force=False,
):

    con = _open_verify_lookup()

    stats = {}

    for model_name in models:

        print(
            "\nBuilding:",
            model_name,
        )

        cfg = MODEL_CONFIG[
            model_name
        ]

        dirs = globals().get(
            cfg["dirs_name"]
        )

        if dirs is None:

            raise RuntimeError(
                f"{cfg['dirs_name']} "
                "chưa được định nghĩa."
            )

        if isinstance(
            dirs,
            (str, Path),
        ):
            dirs = [dirs]

        existing = con.execute(
            """
            SELECT COUNT(*)
            FROM embedding_lookup
            WHERE model=?
            """,
            (
                model_name,
            ),
        ).fetchone()[0]

        if existing > 0 and not force:

            print(
                f"Reuse {existing:,} rows"
            )

            stats[
                model_name
            ] = int(existing)

            continue

        if force:

            con.execute(
                """
                DELETE FROM embedding_lookup
                WHERE model=?
                """,
                (
                    model_name,
                ),
            )

            con.commit()

        for root_value in dirs:

            root = Path(
                root_value
            )

            if not root.exists():

                print(
                    "Skip missing:",
                    root,
                )

                continue

            pairs = _discover_chunk_pairs(
                root,
                cfg["prefix"],
            )

            print(
                root,
                "->",
                len(pairs),
                "chunks",
            )

            for (
                metadata_file,
                embedding_file,
            ) in pairs:

                try:

                    df = pd.read_csv(
                        metadata_file
                    )

                except Exception as exc:

                    print(
                        "Metadata error:",
                        metadata_file,
                        exc,
                    )

                    continue

                if (
                    "image_path"
                    not in df.columns
                ):

                    continue

                embeddings = np.load(
                    embedding_file,
                    mmap_mode="r",
                )

                n = min(
                    len(df),
                    embeddings.shape[0],
                )

                rows = []

                for row_idx in range(n):

                    row = df.iloc[
                        row_idx
                    ]

                    image_path = (
                        _normalize_path(
                            row[
                                "image_path"
                            ]
                        )
                    )

                    filename = Path(
                        image_path
                    ).name

                    if not filename:
                        continue

                    rows.append(
                        (
                            model_name,
                            filename,
                            str(
                                embedding_file
                            ),
                            row_idx,
                            image_path,

                            str(
                                row.get(
                                    "video_id",
                                    "",
                                )
                            ),

                            _safe_float(
                                row.get(
                                    "timestamp",
                                    0,
                                )
                            ),

                            _safe_int(
                                row.get(
                                    "shot_id",
                                    0,
                                )
                            ),
                        )
                    )

                con.executemany(
                    """
                    INSERT OR IGNORE INTO
                    embedding_lookup (

                        model,
                        filename,
                        emb_file,
                        row_idx,
                        image_path,
                        video_id,
                        timestamp,
                        shot_id

                    )

                    VALUES (
                        ?, ?, ?, ?, ?, ?, ?, ?
                    )
                    """,
                    rows,
                )

                con.commit()

        total = con.execute(
            """
            SELECT COUNT(*)
            FROM embedding_lookup
            WHERE model=?
            """,
            (
                model_name,
            ),
        ).fetchone()[0]

        stats[
            model_name
        ] = int(total)

        print(
            f"{model_name}: "
            f"{total:,} indexed frames"
        )

    con.close()

    return stats


# ------------------------------------------------------------
# 7. FETCH PRECOMPUTED EMBEDDINGS
# ------------------------------------------------------------

def _fetch_embeddings(
    model_name,
    filenames,
):

    if not VERIFY_LOOKUP_DB.exists():

        raise RuntimeError(
            "Embedding lookup chưa được build.\n"
            "Hãy chạy:\n\n"
            "build_verify_embedding_lookup()"
        )

    filenames = list(
        dict.fromkeys(
            filenames
        )
    )

    if len(filenames) == 0:
        return {}

    con = sqlite3.connect(
        str(
            VERIFY_LOOKUP_DB
        )
    )

    locators = {}

    # tránh SQLite variable limit
    for start in range(
        0,
        len(filenames),
        400,
    ):

        chunk = filenames[
            start:start + 400
        ]

        placeholders = ",".join(
            "?"
            for _ in chunk
        )

        sql = (
            "SELECT filename, emb_file, row_idx "
            "FROM embedding_lookup "
            "WHERE model=? "
            f"AND filename IN ({placeholders})"
        )

        params = [
            model_name,
            *chunk,
        ]

        for (
            filename,
            emb_file,
            row_idx,
        ) in con.execute(
            sql,
            params,
        ):

            locators[
                filename
            ] = (
                emb_file,
                int(row_idx),
            )

    con.close()

    # Group theo .npy để chỉ mmap mỗi file một lần
    grouped = {}

    for (
        filename,
        locator,
    ) in locators.items():

        emb_file, row_idx = locator

        grouped.setdefault(
            emb_file,
            []
        ).append(
            (
                filename,
                row_idx,
            )
        )

    result = {}

    for (
        emb_file,
        items,
    ) in grouped.items():

        arr = np.load(
            emb_file,
            mmap_mode="r",
        )

        for (
            filename,
            row_idx,
        ) in items:

            vec = np.asarray(
                arr[row_idx],
                dtype=np.float32,
            )

            norm = np.linalg.norm(
                vec
            )

            if norm > 0:

                vec = vec / norm

            result[
                filename
            ] = vec

    return result


# ------------------------------------------------------------
# 8. SCORE NORMALIZATION
# ------------------------------------------------------------

def _robust_minmax(values):

    values = np.asarray(
        values,
        dtype=np.float32,
    )

    finite = np.isfinite(
        values
    )

    if not finite.any():

        return np.zeros_like(
            values
        )

    clean = values[
        finite
    ]

    lo = float(
        np.quantile(
            clean,
            0.10,
        )
    )

    hi = float(
        np.quantile(
            clean,
            0.90,
        )
    )

    if hi - lo < 1e-6:

        lo = float(
            np.min(clean)
        )

        hi = float(
            np.max(clean)
        )

    if hi - lo < 1e-6:

        result = np.zeros_like(
            values
        )

        result[
            finite
        ] = 1.0

        return result

    result = (
        values - lo
    ) / (
        hi - lo
    )

    result = np.clip(
        result,
        0.0,
        1.0,
    )

    result[
        ~finite
    ] = 0.0

    return result.astype(
        np.float32
    )


def _smooth_1d(values):

    values = np.asarray(
        values,
        dtype=np.float32,
    )

    if len(values) <= 2:

        return values

    kernel = np.array(
        [
            0.25,
            0.50,
            0.25,
        ],
        dtype=np.float32,
    )

    padded = np.pad(
        values,
        (1, 1),
        mode="edge",
    )

    return np.convolve(
        padded,
        kernel,
        mode="valid",
    ).astype(
        np.float32
    )


# ------------------------------------------------------------
# 9. GET LOCAL WINDOW AROUND ANCHOR
# ------------------------------------------------------------

def get_anchor_window(
    anchor_path,
    radius_frames=24,
    max_seconds_from_anchor=24.0,
):

    anchor_path = _normalize_path(
        anchor_path
    )

    anchor = parse_frame_file_metadata(
        anchor_path
    )

    if anchor is None:

        raise ValueError(
            "Không parse được metadata "
            f"của anchor:\n{anchor_path}"
        )

    _, cache_entry = (
        get_cached_frame_directory_index(
            center_path=anchor_path,
            video_id=anchor[
                "video_id"
            ],
        )
    )

    center_pos = (
        find_center_position(
            cache_entry=cache_entry,
            center_path=anchor_path,
            frame_index=anchor[
                "frame_index"
            ],
        )
    )

    if center_pos is None:

        raise RuntimeError(
            "Không tìm thấy anchor trong "
            "frame directory index."
        )

    records = cache_entry[
        "records"
    ]

    lo = max(
        0,
        center_pos
        - int(radius_frames),
    )

    hi = min(
        len(records),
        center_pos
        + int(radius_frames)
        + 1,
    )

    anchor_time = float(
        anchor[
            "timestamp_seconds"
        ]
    )

    window = []

    for record in records[
        lo:hi
    ]:

        t = float(
            record[
                "timestamp_seconds"
            ]
        )

        if (
            abs(
                t - anchor_time
            )
            <=
            float(
                max_seconds_from_anchor
            )
        ):

            window.append(
                dict(record)
            )

    window.sort(
        key=lambda x:
        float(
            x[
                "timestamp_seconds"
            ]
        )
    )

    return (
        anchor,
        window,
    )


# ------------------------------------------------------------
# 10. SCORE TEMPORAL WINDOW
# ------------------------------------------------------------

def score_temporal_window(
    query,
    window_records,

    models=(
        "bge",
        "metaclip",
        "beit3",
        "perception",
    ),

    model_weights=None,
):

    if not query.strip():

        raise ValueError(
            "Query rỗng."
        )

    if len(window_records) == 0:

        return pd.DataFrame()

    models = list(
        models
    )

    if model_weights is None:

        model_weights = {
            model: 1.0
            for model in models
        }

    rows = []
    filenames = []

    for record in window_records:

        image_path = (
            _normalize_path(
                record[
                    "local_image_path"
                ]
            )
        )

        filename = Path(
            image_path
        ).name

        filenames.append(
            filename
        )

        rows.append(
            {
                "image_path":
                    image_path,

                "filename":
                    filename,

                "video_id":
                    str(
                        record[
                            "video_id"
                        ]
                    ).upper(),

                "frame_index":
                    int(
                        record[
                            "frame_index"
                        ]
                    ),

                "timestamp":
                    float(
                        record[
                            "timestamp_seconds"
                        ]
                    ),

                "shot_id":
                    (
                        int(
                            record[
                                "shot_id"
                            ]
                        )
                        if record.get(
                            "shot_id"
                        ) is not None
                        else None
                    ),
            }
        )

    df = pd.DataFrame(
        rows
    )

    weighted_sum = np.zeros(
        len(df),
        dtype=np.float32,
    )

    weight_sum = np.zeros(
        len(df),
        dtype=np.float32,
    )

    # --------------------------------------------------------
    # score từng visual encoder
    # --------------------------------------------------------

    for model_name in models:

        if model_name not in MODEL_CONFIG:

            raise ValueError(
                "Unknown model: "
                + model_name
            )

        config = MODEL_CONFIG[
            model_name
        ]

        encoder_name = config[
            "query_encoder_name"
        ]

        encoder = globals().get(
            encoder_name
        )

        if encoder is None:

            raise RuntimeError(
                f"{encoder_name} chưa tồn tại."
            )

        # Encode query
        query_embedding = np.asarray(
            encoder(query),
            dtype=np.float32,
        ).reshape(-1)

        norm = np.linalg.norm(
            query_embedding
        )

        if norm > 0:

            query_embedding = (
                query_embedding
                / norm
            )

        # Load neighboring image embeddings
        frame_embeddings = (
            _fetch_embeddings(
                model_name,
                filenames,
            )
        )

        raw_scores = np.full(
            len(df),
            np.nan,
            dtype=np.float32,
        )

        for i, filename in enumerate(
            filenames
        ):

            vec = frame_embeddings.get(
                filename
            )

            if vec is None:

                continue

            if len(vec) != len(
                query_embedding
            ):

                continue

            raw_scores[i] = float(
                np.dot(
                    vec,
                    query_embedding,
                )
            )

        normalized = (
            _robust_minmax(
                raw_scores
            )
        )

        available = np.isfinite(
            raw_scores
        )

        weight = float(
            model_weights.get(
                model_name,
                1.0,
            )
        )

        weighted_sum[
            available
        ] += (
            normalized[
                available
            ]
            * weight
        )

        weight_sum[
            available
        ] += weight

        df[
            f"score_{model_name}"
        ] = raw_scores

        df[
            f"score_{model_name}_norm"
        ] = normalized

    # --------------------------------------------------------
    # fused visual relevance
    # --------------------------------------------------------

    fused = np.divide(

        weighted_sum,

        np.maximum(
            weight_sum,
            1e-8,
        ),

        out=np.zeros_like(
            weighted_sum
        ),

        where=weight_sum > 0,
    )

    df[
        "visual_score"
    ] = fused

    df[
        "visual_score_smooth"
    ] = _smooth_1d(
        fused
    )

    return df


print(
    "VerifyQueue compatibility layer ready."
)

In [ ]:
VERIFY_LOOKUP_STATS = (
    build_verify_embedding_lookup()
)

print(
    VERIFY_LOOKUP_STATS
)

In [ ]:
# ============================================================
# ACASE PATCH
# SELF-CONTAINED ANCHOR / NEIGHBOR FRAME LOADER
# Không phụ thuộc FastAPI helpers
# ============================================================

import re
from pathlib import Path


FRAME_IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
}

NEIGHBOR_FRAME_PATTERN = re.compile(
    r"(?P<video_id>L\d{2}_V\d{3})"
    r"_t(?P<timestamp>\d+(?:\.\d+)?)"
    r"_f(?P<frame_idx>\d+)"
    r"_s(?P<shot_id>\d+)",
    re.IGNORECASE,
)


# Cache directory để không scan lại nhiều lần
_ACASE_FRAME_DIR_CACHE = {}


def parse_frame_file_metadata(
    frame_path,
    expected_video_id=None,
):
    """
    Ví dụ filename:

    L21_V002_t00005.600_f00000168_s000001.jpg
    """

    path = Path(
        _normalize_path(frame_path)
    )

    match = NEIGHBOR_FRAME_PATTERN.search(
        path.name
    )

    if match is None:
        return None

    video_id = (
        match
        .group("video_id")
        .upper()
    )

    if expected_video_id is not None:

        if (
            video_id
            != str(expected_video_id).strip().upper()
        ):
            return None

    return {
        "local_image_path": str(path),

        "video_id": video_id,

        "frame_index": int(
            match.group("frame_idx")
        ),

        "timestamp_seconds": float(
            match.group("timestamp")
        ),

        "shot_id": int(
            match.group("shot_id")
        ),
    }


def _load_video_frame_directory(
    anchor_path,
    video_id,
):
    """
    Scan toàn bộ frame trong cùng directory với anchor.
    Kết quả được cache.
    """

    anchor_path = Path(
        _normalize_path(anchor_path)
    )

    directory = anchor_path.parent

    cache_key = str(directory)

    if cache_key in _ACASE_FRAME_DIR_CACHE:
        return _ACASE_FRAME_DIR_CACHE[
            cache_key
        ]

    if not directory.is_dir():

        raise FileNotFoundError(
            f"Không tìm thấy frame directory:\n"
            f"{directory}"
        )

    records = []

    for candidate in directory.iterdir():

        if not candidate.is_file():
            continue

        if (
            candidate.suffix.lower()
            not in FRAME_IMAGE_EXTENSIONS
        ):
            continue

        metadata = (
            parse_frame_file_metadata(
                candidate,
                expected_video_id=video_id,
            )
        )

        if metadata is not None:

            records.append(
                metadata
            )

    records.sort(
        key=lambda x: (
            x["frame_index"],
            x["timestamp_seconds"],
        )
    )

    _ACASE_FRAME_DIR_CACHE[
        cache_key
    ] = records

    print(
        f"[ACASE] indexed "
        f"{len(records):,} frames from "
        f"{directory}"
    )

    return records


def get_anchor_window(
    anchor_path,
    radius_frames=24,
    max_seconds_from_anchor=24.0,
):
    """
    Anchor
        ↓
    scan cùng video directory
        ↓
    lấy ± radius_frames
        ↓
    giữ frame trong ± max_seconds_from_anchor
    """

    anchor_path = _normalize_path(
        anchor_path
    )

    # ----------------------------------------
    # Parse anchor
    # ----------------------------------------

    anchor = parse_frame_file_metadata(
        anchor_path
    )

    if anchor is None:

        raise ValueError(
            "Không parse được anchor filename.\n"
            "Anchor phải có dạng kiểu:\n"
            "L21_V002_t00005.600_"
            "f00000168_s000001.jpg\n\n"
            f"Received:\n{anchor_path}"
        )

    # ----------------------------------------
    # Load toàn bộ frame của directory
    # ----------------------------------------

    records = _load_video_frame_directory(
        anchor_path,
        anchor["video_id"],
    )

    if len(records) == 0:

        raise RuntimeError(
            "Không tìm thấy frame nào "
            "trong directory."
        )

    # ----------------------------------------
    # Tìm vị trí anchor
    # ----------------------------------------

    anchor_name = Path(
        anchor_path
    ).name

    center_pos = None

    # Ưu tiên filename exact match
    for i, record in enumerate(records):

        if (
            Path(
                record["local_image_path"]
            ).name
            == anchor_name
        ):

            center_pos = i
            break

    # Fallback bằng frame index
    if center_pos is None:

        target_frame_idx = int(
            anchor["frame_index"]
        )

        for i, record in enumerate(records):

            if (
                int(record["frame_index"])
                == target_frame_idx
            ):

                center_pos = i
                break

    if center_pos is None:

        raise RuntimeError(
            "Không tìm thấy anchor frame "
            "trong directory index."
        )

    # ----------------------------------------
    # Neighbor window
    # ----------------------------------------

    lo = max(
        0,
        center_pos - int(radius_frames),
    )

    hi = min(
        len(records),
        center_pos + int(radius_frames) + 1,
    )

    anchor_time = float(
        anchor["timestamp_seconds"]
    )

    window = []

    for record in records[lo:hi]:

        timestamp = float(
            record["timestamp_seconds"]
        )

        if (
            abs(
                timestamp - anchor_time
            )
            <= float(
                max_seconds_from_anchor
            )
        ):

            window.append(
                dict(record)
            )

    window.sort(
        key=lambda x:
        float(
            x["timestamp_seconds"]
        )
    )

    print(
        "[ACASE]",
        "anchor =",
        anchor["video_id"],
        f'{anchor_time:.1f}s',
        "| shot =",
        anchor["shot_id"],
        "| local frames =",
        len(window),
    )

    return anchor, window


print(
    "ACASE anchor-window patch ready."
)

In [ ]:
# ============================================================
# VERIFYQUEUE V2
# CONSENSUS-AWARE + BOUNDARY-AWARE SHOT EXPANSION
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# 1. MULTI-ENCODER CONSENSUS
# ------------------------------------------------------------

def add_consensus_score(
    df: pd.DataFrame,
    models=("bge", "metaclip", "beit3", "perception"),
    disagreement_penalty=0.25,
    smooth_window=3,
):
    """
    Gộp score của nhiều visual encoder.

    Ý tưởng:
        score cao + các model đồng thuận
        => evidence mạnh hơn.

    consensus =
        mean(model_scores)
        - lambda * std(model_scores)
    """

    df = df.copy()

    score_cols = [
        f"score_{m}_norm"
        for m in models
        if f"score_{m}_norm" in df.columns
    ]

    if len(score_cols) == 0:
        raise ValueError(
            "Không tìm thấy normalized score của visual models."
        )

    X = df[score_cols].to_numpy(dtype=np.float32)

    # số model thực sự có score ở mỗi frame
    valid_count = np.sum(np.isfinite(X), axis=1)

    mean_score = np.nanmean(X, axis=1)
    disagreement = np.nanstd(X, axis=1)

    # frame không có model nào -> 0
    mean_score = np.nan_to_num(mean_score, nan=0.0)
    disagreement = np.nan_to_num(disagreement, nan=1.0)

    consensus = (
        mean_score
        - disagreement_penalty * disagreement
    )

    consensus = np.clip(
        consensus,
        0.0,
        1.0,
    )

    df["encoder_mean"] = mean_score
    df["encoder_disagreement"] = disagreement
    df["encoder_count"] = valid_count
    df["consensus_score"] = consensus

    # Smooth temporal relevance
    df["consensus_smooth"] = (
        pd.Series(consensus)
        .rolling(
            window=smooth_window,
            center=True,
            min_periods=1,
        )
        .mean()
        .to_numpy(dtype=np.float32)
    )

    return df


# ------------------------------------------------------------
# 2. BUILD CONTIGUOUS SHOT BLOCKS
# ------------------------------------------------------------

def build_shot_blocks(df: pd.DataFrame):
    """
    Output:
        [
            {
                shot_id,
                left,
                right
            },
            ...
        ]
    """

    if df.empty:
        return []

    blocks = []

    start = 0
    current = df.iloc[0]["shot_id"]

    for i in range(1, len(df)):

        shot = df.iloc[i]["shot_id"]

        same = (
            (pd.isna(shot) and pd.isna(current))
            or shot == current
        )

        if not same:

            blocks.append(
                {
                    "shot_id": current,
                    "left": start,
                    "right": i - 1,
                }
            )

            start = i
            current = shot

    blocks.append(
        {
            "shot_id": current,
            "left": start,
            "right": len(df) - 1,
        }
    )

    return blocks


# ------------------------------------------------------------
# 3. SHOT RELEVANCE
# ------------------------------------------------------------

def shot_relevance(
    df: pd.DataFrame,
    block: dict,
    top_k=3,
):
    """
    Không chỉ lấy MAX vì dễ bị outlier.

    60% = mean top relevant frames
    40% = average cả shot
    """

    l = block["left"]
    r = block["right"]

    values = (
        df.iloc[l:r + 1]["consensus_smooth"]
        .to_numpy(dtype=np.float32)
    )

    if len(values) == 0:
        return 0.0

    k = min(
        int(top_k),
        len(values),
    )

    top_values = np.sort(values)[-k:]

    return (
        0.60 * float(np.mean(top_values))
        +
        0.40 * float(np.mean(values))
    )


# ------------------------------------------------------------
# 4. BOUNDARY RELEVANCE
# ------------------------------------------------------------

def boundary_relevance(
    df: pd.DataFrame,
    left_block: dict,
    right_block: dict,
    n_frames=2,
):
    """
    Kiểm tra event có tiếp tục qua shot boundary không.

    Lấy vài frame cuối shot trái +
    vài frame đầu shot phải.
    """

    left_end = left_block["right"]
    right_start = right_block["left"]

    left_values = (
        df.iloc[
            max(0, left_end - n_frames + 1):
            left_end + 1
        ]["consensus_smooth"]
        .to_numpy(dtype=np.float32)
    )

    right_values = (
        df.iloc[
            right_start:
            min(len(df), right_start + n_frames)
        ]["consensus_smooth"]
        .to_numpy(dtype=np.float32)
    )

    values = np.concatenate(
        [
            left_values,
            right_values,
        ]
    )

    if len(values) == 0:
        return 0.0

    return float(np.mean(values))


# ------------------------------------------------------------
# 5. ADAPTIVE BIDIRECTIONAL EXPANSION
# ------------------------------------------------------------

def adaptive_shot_expansion(
    df: pd.DataFrame,
    anchor_idx: int,
    max_shots_each_side=2,
    max_duration=20.0,
    min_threshold=0.38,
    alpha=0.70,
    beta=0.30,
):
    """
    Bắt đầu từ anchor shot.

    Mỗi vòng:
        - xét shot bên trái
        - xét shot bên phải
        - tính gain
        - chọn hướng tốt hơn
        - nếu gain thấp -> stop

    gain =
        alpha * shot relevance
        +
        beta * boundary relevance
    """

    blocks = build_shot_blocks(df)

    if len(blocks) == 0:
        raise RuntimeError("Không có shot block.")

    # ----------------------------------------
    # Tìm block chứa anchor
    # ----------------------------------------

    anchor_block_idx = None

    for i, block in enumerate(blocks):

        if (
            block["left"]
            <= anchor_idx
            <= block["right"]
        ):
            anchor_block_idx = i
            break

    if anchor_block_idx is None:
        raise RuntimeError(
            "Không tìm được anchor shot."
        )

    left_block_idx = anchor_block_idx
    right_block_idx = anchor_block_idx

    used_left = 0
    used_right = 0

    # ----------------------------------------
    # Adaptive threshold
    # ----------------------------------------

    local_scores = (
        df["consensus_smooth"]
        .to_numpy(dtype=np.float32)
    )

    adaptive_threshold = max(
        float(min_threshold),
        float(
            np.quantile(
                local_scores,
                0.55,
            )
        ),
    )

    adaptive_threshold = min(
        adaptive_threshold,
        0.70,
    )

    # ----------------------------------------
    # Expansion
    # ----------------------------------------

    expansion_log = []

    while True:

        candidates = []

        # ====================================================
        # LEFT
        # ====================================================

        if (
            left_block_idx > 0
            and used_left < max_shots_each_side
        ):

            candidate_idx = left_block_idx - 1

            candidate_block = blocks[candidate_idx]
            current_block = blocks[left_block_idx]

            relevance = shot_relevance(
                df,
                candidate_block,
            )

            boundary = boundary_relevance(
                df,
                candidate_block,
                current_block,
            )

            gain = (
                alpha * relevance
                +
                beta * boundary
            )

            new_start_idx = candidate_block["left"]
            current_end_idx = (
                blocks[right_block_idx]["right"]
            )

            duration = float(
                df.iloc[current_end_idx]["timestamp"]
                -
                df.iloc[new_start_idx]["timestamp"]
            )

            if duration <= max_duration:

                candidates.append(
                    {
                        "side": "left",
                        "block_idx": candidate_idx,
                        "relevance": relevance,
                        "boundary": boundary,
                        "gain": gain,
                        "duration": duration,
                    }
                )

        # ====================================================
        # RIGHT
        # ====================================================

        if (
            right_block_idx + 1 < len(blocks)
            and used_right < max_shots_each_side
        ):

            candidate_idx = right_block_idx + 1

            current_block = blocks[right_block_idx]
            candidate_block = blocks[candidate_idx]

            relevance = shot_relevance(
                df,
                candidate_block,
            )

            boundary = boundary_relevance(
                df,
                current_block,
                candidate_block,
            )

            gain = (
                alpha * relevance
                +
                beta * boundary
            )

            current_start_idx = (
                blocks[left_block_idx]["left"]
            )

            new_end_idx = candidate_block["right"]

            duration = float(
                df.iloc[new_end_idx]["timestamp"]
                -
                df.iloc[current_start_idx]["timestamp"]
            )

            if duration <= max_duration:

                candidates.append(
                    {
                        "side": "right",
                        "block_idx": candidate_idx,
                        "relevance": relevance,
                        "boundary": boundary,
                        "gain": gain,
                        "duration": duration,
                    }
                )

        # Không còn candidate
        if len(candidates) == 0:
            break

        # ----------------------------------------
        # Chọn hướng tốt nhất
        # ----------------------------------------

        best = max(
            candidates,
            key=lambda x: x["gain"],
        )

        # Nếu shot tốt nhất cũng không đủ relevance
        # => dừng.
        if best["gain"] < adaptive_threshold:
            break

        expansion_log.append(best)

        if best["side"] == "left":

            left_block_idx = best["block_idx"]
            used_left += 1

        else:

            right_block_idx = best["block_idx"]
            used_right += 1

    # ----------------------------------------
    # Final interval
    # ----------------------------------------

    left_idx = blocks[left_block_idx]["left"]
    right_idx = blocks[right_block_idx]["right"]

    return {
        "left_idx": left_idx,
        "right_idx": right_idx,
        "left_shot": blocks[left_block_idx]["shot_id"],
        "right_shot": blocks[right_block_idx]["shot_id"],
        "anchor_shot": blocks[anchor_block_idx]["shot_id"],
        "threshold": adaptive_threshold,
        "expansion_log": expansion_log,
    }


print("ACASE functions ready.")

In [ ]:
# ============================================================
# VERIFYQUEUE V2
# ANCHOR -> ADAPTIVE TEMPORAL SEGMENT
# ============================================================

def localize_adaptive_segment_v2(
    anchor_path,
    query,
    *,
    radius_frames=24,
    max_seconds_from_anchor=24.0,
    max_shots_each_side=2,
    max_duration=20.0,

    models=(
        "bge",
        "metaclip",
        "beit3",
        "perception",
    ),

    model_weights=None,

    disagreement_penalty=0.25,

    min_threshold=0.38,

    alpha=0.70,
    beta=0.30,
):

    # --------------------------------------------------------
    # STEP 1
    # Lấy neighborhood quanh anchor
    # --------------------------------------------------------

    anchor, window = get_anchor_window(
        anchor_path,
        radius_frames=radius_frames,
        max_seconds_from_anchor=max_seconds_from_anchor,
    )

    if len(window) == 0:
        raise RuntimeError(
            "Không tìm thấy temporal neighborhood."
        )

    # --------------------------------------------------------
    # STEP 2
    # Query-frame relevance bằng embedding đã precompute
    # --------------------------------------------------------

    scores = score_temporal_window(
        query=query,
        window_records=window,
        models=models,
        model_weights=model_weights,
    )

    if scores.empty:
        raise RuntimeError(
            "Không có frame nào được score."
        )

    # --------------------------------------------------------
    # STEP 3
    # Multi-model consensus
    # --------------------------------------------------------

    scores = add_consensus_score(
        scores,
        models=models,
        disagreement_penalty=disagreement_penalty,
    )

    # --------------------------------------------------------
    # STEP 4
    # Tìm anchor trong DataFrame
    # --------------------------------------------------------

    canonical_anchor = _normalize_path(
        anchor_path
    )

    anchor_mask = (
        scores["image_path"]
        .map(_normalize_path)
        == canonical_anchor
    )

    if anchor_mask.any():

        anchor_idx = int(
            np.flatnonzero(
                anchor_mask.to_numpy()
            )[0]
        )

    else:

        # fallback nearest timestamp
        anchor_idx = int(
            np.argmin(
                np.abs(
                    scores["timestamp"]
                    .to_numpy(dtype=np.float32)
                    -
                    float(
                        anchor["timestamp_seconds"]
                    )
                )
            )
        )

    scores["is_anchor"] = False
    scores.loc[
        scores.index[anchor_idx],
        "is_anchor"
    ] = True

    # --------------------------------------------------------
    # STEP 5
    # Adaptive shot expansion
    # --------------------------------------------------------

    expansion = adaptive_shot_expansion(
        scores,
        anchor_idx=anchor_idx,

        max_shots_each_side=
            max_shots_each_side,

        max_duration=
            max_duration,

        min_threshold=
            min_threshold,

        alpha=alpha,
        beta=beta,
    )

    l = expansion["left_idx"]
    r = expansion["right_idx"]

    selected_df = (
        scores
        .iloc[l:r + 1]
        .copy()
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # STEP 6
    # Convert thành VerifyFrame
    # --------------------------------------------------------

    verify_frames = []

    for _, row in selected_df.iterrows():

        verify_frames.append(
            VerifyFrame(
                image_path=str(
                    row["image_path"]
                ),

                video_id=str(
                    row["video_id"]
                ),

                timestamp=float(
                    row["timestamp"]
                ),

                frame_index=int(
                    row["frame_index"]
                ),

                shot_id=(
                    int(row["shot_id"])
                    if pd.notna(
                        row["shot_id"]
                    )
                    else None
                ),

                # V2 dùng consensus
                visual_score=float(
                    row["consensus_score"]
                ),

                visual_score_smooth=float(
                    row["consensus_smooth"]
                ),

                is_anchor=bool(
                    row["is_anchor"]
                ),
            )
        )

    # --------------------------------------------------------
    # STEP 7
    # Final VerifySegment
    # --------------------------------------------------------

    segment = VerifySegment(

        video_id=str(
            anchor["video_id"]
        ).upper(),

        anchor_path=
            canonical_anchor,

        anchor_timestamp=float(
            anchor["timestamp_seconds"]
        ),

        start_time=float(
            selected_df.iloc[0][
                "timestamp"
            ]
        ),

        end_time=float(
            selected_df.iloc[-1][
                "timestamp"
            ]
        ),

        threshold=float(
            expansion["threshold"]
        ),

        frames=verify_frames,
    )

    return (
        segment,
        scores,
        expansion,
    )


print("localize_adaptive_segment_v2 ready.")

In [ ]:
# ============================================================
# TEST ACASE - RETRIEVAL -> CHOOSE ANCHOR -> LOCALIZE
# ============================================================

# Query bạn muốn test
QUERY = "a man riding a motorbike on the street"

# ------------------------------------------------------------
# STEP 1: RETRIEVAL
# ------------------------------------------------------------

search_meta, retrieval_results = run_multimodal_search(
    visual_query=QUERY,
    query_mode="visual",

    visual_models=[
        "bge",
        "metaclip",
        "beit3",
        "perception",
    ],

    visual_top_k=100,
    max_results=20,
)

print("Query:", QUERY)
print("Number of results:", len(retrieval_results))

display(
    retrieval_results[
        [
            "final_rank",
            "video_id_fixed",
            "time_fixed",
            "local_image_path",
        ]
    ].head(12)
)

# Hiện ảnh để bạn chọn
visualize_results(
    retrieval_results,
    n=12,
    cols=4,
)

In [ ]:
# ============================================================
# CHOOSE ANCHOR
# ============================================================

ANCHOR_RANK = 3

anchor_row = retrieval_results.iloc[
    ANCHOR_RANK - 1
]

ANCHOR_PATH = str(
    anchor_row["local_image_path"]
)

print("Selected anchor")
print("----------------")
print("Rank:", ANCHOR_RANK)
print(
    "Video:",
    anchor_row["video_id_fixed"],
)
print(
    "Time:",
    anchor_row["time_fixed"],
)
print(
    "Path:",
    ANCHOR_PATH,
)

In [ ]:
# ============================================================
# RUN ACASE LOCALIZATION
# ============================================================

segment, scores, expansion = localize_adaptive_segment_v2(
    anchor_path=ANCHOR_PATH,
    query=QUERY,
)

print("=" * 70)

print("Video:", segment.video_id)
print("Anchor:", segment.anchor_timestamp)

print(
    "Selected segment:",
    f"{segment.start_time:.1f}s",
    "->",
    f"{segment.end_time:.1f}s",
)

print(
    "Duration:",
    f"{segment.end_time - segment.start_time:.1f}s",
)

print(
    "Shots:",
    expansion["left_shot"],
    "->",
    expansion["right_shot"],
)

print(
    "Threshold:",
    round(expansion["threshold"], 4),
)

print("Number of scored frames:", len(scores))

print("=" * 70)

In [ ]:
# ============================================================
# PLOT TEMPORAL RELEVANCE
# ============================================================

import matplotlib.pyplot as plt

print("scores shape:", scores.shape)
print(
    scores[
        [
            "timestamp",
            "shot_id",
            "consensus_smooth",
            "is_anchor",
        ]
    ]
)

plt.figure(figsize=(14, 5))

plt.plot(
    scores["timestamp"],
    scores["consensus_smooth"],
    marker="o",
    label="Consensus relevance",
)

plt.axvline(
    segment.anchor_timestamp,
    linestyle="--",
    label="Anchor",
)

plt.axvspan(
    segment.start_time,
    segment.end_time,
    alpha=0.2,
    label="Selected segment",
)

plt.axhline(
    expansion["threshold"],
    linestyle=":",
    label="Expansion threshold",
)

plt.xlabel("Timestamp (seconds)")
plt.ylabel("Consensus relevance")

plt.title(
    f"{segment.video_id} - Anchor-conditioned temporal relevance"
)

plt.legend()

plt.show()

In [ ]:
# ============================================================
# V1 - INSTALL VLM DEPENDENCIES
# ============================================================

!pip install -q -U bitsandbytes qwen-vl-utils

print("VLM dependencies ready.")

In [ ]:
# ============================================================
# V2 - COMPACT TEMPORAL EVIDENCE SELECTION
# ============================================================

from pathlib import Path


def _frame_timestamp(frame):
    return float(
        getattr(
            frame,
            "timestamp",
            0.0,
        )
    )


def _frame_score(frame):

    for name in [
        "visual_score_smooth",
        "consensus_smooth",
        "visual_score",
        "score",
    ]:

        value = getattr(
            frame,
            name,
            None,
        )

        if value is not None:

            try:
                return float(value)

            except Exception:
                pass

    return 0.0


def select_vlm_evidence_frames(
    segment,
    max_frames=3,
    min_gap_seconds=1.0,
):
    """
    Compact evidence selection.

    Priority:
    1. anchor
    2. strong evidence before anchor
    3. strong evidence after anchor
    4. temporal boundaries
    5. remaining relevance peaks

    Output is always chronological.
    """

    frames = sorted(
        list(segment.frames),
        key=_frame_timestamp,
    )

    if not frames:
        return []

    if len(frames) <= max_frames:
        return frames

    anchor_time = float(
        segment.anchor_timestamp
    )

    # --------------------------------------------------------
    # Find frame closest to anchor
    # --------------------------------------------------------

    anchor = min(
        frames,
        key=lambda f: abs(
            _frame_timestamp(f)
            - anchor_time
        ),
    )

    selected = []


    def already_selected(frame):

        path = str(
            getattr(
                frame,
                "image_path",
                "",
            )
        )

        return any(
            str(
                getattr(
                    x,
                    "image_path",
                    "",
                )
            )
            == path

            for x in selected
        )


    def add(frame):

        if frame is None:
            return

        if already_selected(frame):
            return

        selected.append(frame)


    # --------------------------------------------------------
    # 1. Anchor
    # --------------------------------------------------------

    add(anchor)


    # --------------------------------------------------------
    # 2. Best evidence before anchor
    # --------------------------------------------------------

    before = [
        f
        for f in frames
        if _frame_timestamp(f)
        < anchor_time
    ]

    if before:

        best_before = max(
            before,
            key=_frame_score,
        )

        add(best_before)


    # --------------------------------------------------------
    # 3. Best evidence after anchor
    # --------------------------------------------------------

    after = [
        f
        for f in frames
        if _frame_timestamp(f)
        > anchor_time
    ]

    if after:

        best_after = max(
            after,
            key=_frame_score,
        )

        add(best_after)


    # --------------------------------------------------------
    # If max_frames already reached
    # --------------------------------------------------------

    if len(selected) >= max_frames:

        return sorted(
            selected[:max_frames],
            key=_frame_timestamp,
        )


    # --------------------------------------------------------
    # 4. Boundaries
    # --------------------------------------------------------

    add(frames[0])

    if len(selected) < max_frames:
        add(frames[-1])


    # --------------------------------------------------------
    # 5. Remaining relevance peaks
    # --------------------------------------------------------

    ranked = sorted(
        frames,
        key=_frame_score,
        reverse=True,
    )

    for frame in ranked:

        if len(selected) >= max_frames:
            break

        if already_selected(frame):
            continue

        timestamp = _frame_timestamp(
            frame
        )

        if selected:

            min_gap = min(
                abs(
                    timestamp
                    - _frame_timestamp(x)
                )

                for x in selected
            )

            if min_gap < min_gap_seconds:
                continue

        add(frame)


    # --------------------------------------------------------
    # Fill remaining slots if needed
    # --------------------------------------------------------

    for frame in frames:

        if len(selected) >= max_frames:
            break

        add(frame)


    return sorted(
        selected[:max_frames],
        key=_frame_timestamp,
    )


print(
    "Compact evidence selector ready."
)

In [ ]:
# ============================================================
# V3 - IMAGE PREPARATION FOR VLM
# ============================================================

from PIL import Image


VLM_IMAGE_MAX_SIDE = 336


def prepare_vlm_image(
    image_path,
    max_side=VLM_IMAGE_MAX_SIDE,
):

    with Image.open(
        image_path
    ) as image:

        image = image.convert(
            "RGB"
        )

        image.thumbnail(
            (
                max_side,
                max_side,
            )
        )

        return image.copy()


print(
    "VLM image preparation ready."
)

In [ ]:
# ============================================================
# V4 - CLEAR OLD QWEN
# ============================================================

import gc
import torch


for name in [
    "qwen_vlm",
    "processor",
    "inputs",
    "generated_ids",
    "generated_trimmed",
]:

    if name in globals():

        try:
            del globals()[name]

        except Exception:
            pass


gc.collect()

torch.cuda.empty_cache()


print("Old Qwen objects cleared.")


for i in range(
    torch.cuda.device_count()
):

    free, total = (
        torch.cuda.mem_get_info(i)
    )

    print(
        f"GPU {i}: "
        f"free="
        f"{free / 1024**3:.2f} GB | "
        f"used="
        f"{(total-free) / 1024**3:.2f} GB"
    )

In [ ]:
# ============================================================
# V5 - LOAD QWEN3-VL-4B 4BIT ON GPU 1 ONLY
# ============================================================

import torch

from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)


QWEN_MODEL_ID = (
    "Qwen/Qwen3-VL-4B-Instruct"
)


# ------------------------------------------------------------
# 4-bit quantization
# ------------------------------------------------------------

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_use_double_quant=True,

    bnb_4bit_compute_dtype=
        torch.float16,
)


# ------------------------------------------------------------
# Processor
# ------------------------------------------------------------

processor = (
    AutoProcessor
    .from_pretrained(
        QWEN_MODEL_ID
    )
)


# Limit visual resolution
if hasattr(
    processor,
    "image_processor",
):

    try:

        processor.image_processor.min_pixels = (
            224 * 224
        )

        processor.image_processor.max_pixels = (
            VLM_IMAGE_MAX_SIDE
            * VLM_IMAGE_MAX_SIDE
        )

    except Exception:
        pass


# ------------------------------------------------------------
# Model -> GPU 1 ONLY
# ------------------------------------------------------------

qwen_vlm = (
    Qwen3VLForConditionalGeneration
    .from_pretrained(

        QWEN_MODEL_ID,

        quantization_config=
            bnb_config,

        device_map={
            "": "cuda:1"
        },

        torch_dtype=
            torch.float16,

        attn_implementation=
            "sdpa",

        low_cpu_mem_usage=True,
    )
)


qwen_vlm.eval()


print(
    "Qwen3-VL-4B 4-bit loaded."
)

In [ ]:
# ============================================================
# V6 - CHECK QWEN DEVICE PLACEMENT
# ============================================================

import torch


devices = {}


for name, param in (
    qwen_vlm.named_parameters()
):

    device = str(
        param.device
    )

    devices[device] = (
        devices.get(
            device,
            0,
        )
        + param.numel()
    )


print(
    "Parameter devices:"
)


for device, count in (
    devices.items()
):

    print(
        device,
        f"{count / 1e9:.3f} B params"
    )


print(
    "\nVisual:",
    next(
        qwen_vlm
        .model
        .visual
        .parameters()
    ).device
)


print(
    "Language model:",
    next(
        qwen_vlm
        .model
        .language_model
        .parameters()
    ).device
)


print(
    "\nGPU memory:"
)


for i in range(
    torch.cuda.device_count()
):

    free, total = (
        torch.cuda.mem_get_info(i)
    )

    print(
        f"GPU {i}: "
        f"free="
        f"{free / 1024**3:.2f} GB | "
        f"used="
        f"{(total-free) / 1024**3:.2f} GB"
    )

In [ ]:
# ============================================================
# V7 - ORIGINAL-INTENT MULTIMODAL VIDEO EVIDENCE VERIFIER
#
# Verification target:
#   ORIGINAL USER INTENT (q_original)
#
# Search query is context only. It is NEVER the verification criterion.
#
# Evidence:
#   - compact chronological visual frames
#   - OCR metadata collected from the localized frame region
#   - ASR transcript segments overlapping the localized region
#
# One VLM call performs claim decomposition + verification to minimize latency.
# ============================================================

import gc
import json
import re
import time

import torch

from qwen_vl_utils import (
    process_vision_info,
)


VERIFY_SYSTEM_PROMPT = """
You are a conservative multimodal verifier for interactive video retrieval.

IMPORTANT:
- ORIGINAL USER INTENT is the ONLY criterion for deciding whether the candidate is correct.
- SEARCH QUERY is provided only to explain how the candidate was retrieved.
- Never replace, weaken, or reinterpret the original intent using the search query.

You receive:
1. ORIGINAL USER INTENT;
2. SEARCH QUERY used by the retrieval system;
3. a localized temporal video segment;
4. a small set of chronological visual frames;
5. optional OCR evidence from frames in the localized segment;
6. optional ASR transcript evidence overlapping the localized segment.

Verification procedure:
- Decompose the ORIGINAL USER INTENT into a small set of atomic essential claims.
- Verify each claim using the available evidence.
- Use visual evidence for visible objects/actions/relations.
- Use OCR evidence for text visible in the scene.
- Use ASR evidence for spoken content and contextual facts.
- Do not infer facts that are not supported by the evidence.
- Similar appearance alone is not enough for a complex event.
- If evidence is insufficient, use UNKNOWN instead of guessing.
- Keep at most 8 claims.
- Keep reasons concise.

Claim status:
SUPPORTED = evidence supports the claim.
CONFLICT = evidence contradicts the claim.
UNKNOWN = evidence is insufficient or ambiguous.

Overall status:
SUPPORTED = every essential claim is supported and no essential claim conflicts.
CONFLICT = at least one essential claim is clearly contradicted.
UNKNOWN = no clear conflict exists, but one or more essential claims remain unverified.

Return JSON only:
{
  "status": "SUPPORTED | CONFLICT | UNKNOWN",
  "confidence": 0.0,
  "claims": [
    {
      "claim": "atomic requirement",
      "status": "SUPPORTED | CONFLICT | UNKNOWN",
      "sources": ["visual", "ocr", "asr"],
      "evidence_frames": [1, 2],
      "evidence_text": ["short OCR/ASR quote or observation"],
      "reason": "short reason"
    }
  ],
  "evidence_frames": [1, 2],
  "explanation": "one or two short sentences"
}
""".strip()


def _extract_json(text):
    """
    Parse the first complete JSON object from model output.

    - Accepts ```json fences.
    - Ignores harmless trailing text after a valid JSON object.
    - Never crashes the verification queue because of malformed model JSON.
      Malformed output becomes UNKNOWN so the system stays conservative.
    """
    raw = str(text or "").strip()

    cleaned = re.sub(
        r"^\s*```(?:json)?\s*",
        "",
        raw,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"\s*```\s*$",
        "",
        cleaned,
    ).strip()

    decoder = json.JSONDecoder()

    # Try each object opening. raw_decode is better than greedy {.*}
    # because it stops exactly at the first complete JSON object.
    positions = [
        i
        for i, ch in enumerate(cleaned)
        if ch == "{"
    ]

    last_error = None

    for pos in positions:
        try:
            value, _ = decoder.raw_decode(
                cleaned[pos:]
            )

            if isinstance(value, dict):
                value["_parser_fallback"] = False
                return value

        except json.JSONDecodeError as error:
            last_error = error

    error_text = (
        f"{type(last_error).__name__}: {last_error}"
        if last_error is not None
        else "No JSON object found."
    )

    print(
        "[VERIFY WARNING] Malformed VLM JSON. "
        "Returning conservative UNKNOWN instead of failing the job."
    )
    print("[VERIFY WARNING]", error_text)

    return {
        "status": "UNKNOWN",
        "confidence": 0.0,
        "claims": [],
        "evidence_frames": [],
        "explanation": (
            "Verifier output was malformed or truncated; "
            "candidate requires another verification pass."
        ),
        "_parser_fallback": True,
        "_parser_error": error_text,
    }


def _format_ocr_evidence_for_prompt(
    ocr_evidence,
    max_items=10,
):

    rows = []

    for item in (
        ocr_evidence or []
    )[:max_items]:

        text = " ".join(
            str(
                item.get(
                    "text",
                    "",
                )
            ).split()
        ).strip()

        if not text:
            continue

        ts = float(
            item.get(
                "timestamp",
                0.0,
            )
        )

        anchor_tag = (
            " [ANCHOR]"
            if item.get(
                "is_anchor",
                False,
            )
            else ""
        )

        rows.append(
            f"- {ts:.1f}s{anchor_tag}: {text}"
        )

    return (
        "\n".join(rows)
        if rows
        else "- none"
    )


def _format_asr_evidence_for_prompt(
    asr_evidence,
    max_items=8,
):

    rows = []

    for item in (
        asr_evidence or []
    )[:max_items]:

        text = " ".join(
            str(
                item.get(
                    "text",
                    "",
                )
            ).split()
        ).strip()

        if not text:
            continue

        start = float(
            item.get(
                "start_time",
                0.0,
            )
        )

        end = float(
            item.get(
                "end_time",
                start,
            )
        )

        rows.append(
            f"- {start:.1f}-{end:.1f}s: {text}"
        )

    return (
        "\n".join(rows)
        if rows
        else "- none"
    )


def verify_segment_vlm_local(
    query=None,
    segment=None,
    max_frames=3,
    *,
    original_query=None,
    search_query="",
    ocr_evidence=None,
    asr_evidence=None,
):
    """
    Verify one retrieved candidate against the ORIGINAL intent.

    Backward compatibility:
      query=... is accepted and is used as original_query when
      original_query is not explicitly provided.
    """

    if segment is None:
        raise ValueError(
            "segment is required."
        )

    verification_query = str(
        original_query
        if original_query is not None
        else query
        or ""
    ).strip()

    if not verification_query:
        raise ValueError(
            "original_query is required for verification."
        )

    search_query = str(
        search_query or ""
    ).strip()

    evidence_frames = (
        select_vlm_evidence_frames(
            segment,
            max_frames=max_frames,
        )
    )

    if not evidence_frames:
        raise ValueError(
            "No evidence frames found."
        )

    print(
        f"Using {len(evidence_frames)} visual evidence frames | "
        f"OCR={len(ocr_evidence or [])} | "
        f"ASR={len(asr_evidence or [])}"
    )

    ocr_prompt = (
        _format_ocr_evidence_for_prompt(
            ocr_evidence,
        )
    )

    asr_prompt = (
        _format_asr_evidence_for_prompt(
            asr_evidence,
        )
    )

    content = [
        {
            "type": "text",
            "text": (
                VERIFY_SYSTEM_PROMPT
                + "\n\n"
                + "ORIGINAL USER INTENT:\n"
                + verification_query
                + "\n\n"
                + "SEARCH QUERY USED FOR RETRIEVAL "
                  "(context only; do NOT verify against this):\n"
                + (
                    search_query
                    if search_query
                    else "(not provided)"
                )
                + "\n\n"
                + f"VIDEO: {segment.video_id}\n"
                + (
                    f"LOCALIZED SEGMENT: "
                    f"{segment.start_time:.1f}s -> "
                    f"{segment.end_time:.1f}s\n"
                )
                + "\nOCR EVIDENCE:\n"
                + ocr_prompt
                + "\n\nASR EVIDENCE:\n"
                + asr_prompt
                + "\n\nVISUAL FRAMES BELOW ARE "
                  "IN CHRONOLOGICAL ORDER."
            ),
        }
    ]

    metadata = []

    for frame_id, frame in enumerate(
        evidence_frames,
        start=1,
    ):

        timestamp = float(
            frame.timestamp
        )

        image_path = str(
            frame.image_path
        )

        metadata.append(
            {
                "frame_id":
                    frame_id,

                "timestamp":
                    timestamp,

                "shot_id":
                    (
                        int(
                            frame.shot_id
                        )
                        if getattr(
                            frame,
                            "shot_id",
                            None,
                        )
                        is not None
                        else None
                    ),

                "image_path":
                    image_path,

                "score":
                    _frame_score(
                        frame
                    ),
            }
        )

        content.append(
            {
                "type": "text",
                "text": (
                    f"FRAME {frame_id} | "
                    f"timestamp={timestamp:.1f}s"
                ),
            }
        )

        content.append(
            {
                "type": "image",
                "image":
                    prepare_vlm_image(
                        image_path,
                        max_side=
                            VLM_IMAGE_MAX_SIDE,
                    ),
            }
        )

    messages = [
        {
            "role": "user",
            "content": content,
        }
    ]

    prompt_text = (
        processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    )

    image_inputs, video_inputs = (
        process_vision_info(
            messages
        )
    )

    inputs = processor(
        text=[
            prompt_text
        ],
        images=
            image_inputs,
        videos=
            video_inputs,
        padding=True,
        return_tensors="pt",
    )

    # Verifier is isolated on GPU 1.
    inputs = inputs.to(
        "cuda:1"
    )

    free_before, _ = (
        torch.cuda.mem_get_info(1)
    )

    print(
        "GPU1 free before inference:",
        f"{free_before / 1024**3:.2f} GB",
    )

    torch.cuda.reset_peak_memory_stats(
        1
    )

    started_at = (
        time.perf_counter()
    )

    with torch.inference_mode():

        generated_ids = (
            qwen_vlm.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                use_cache=True,
            )
        )

    latency_ms = (
        time.perf_counter()
        - started_at
    ) * 1000.0

    peak_memory_gb = (
        torch.cuda
        .max_memory_allocated(1)
        / 1024**3
    )

    generated_trimmed = [
        output_ids[
            len(input_ids):
        ]
        for input_ids, output_ids
        in zip(
            inputs.input_ids,
            generated_ids,
        )
    ]

    output_text = (
        processor.batch_decode(
            generated_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=
                False,
        )[0]
    )

    print(
        "\nRAW VLM OUTPUT:"
    )
    print(
        output_text
    )

    generated_token_count = int(
        generated_trimmed[0].numel()
    )

    if generated_token_count >= 508:
        print(
            "[VERIFY WARNING] Generation reached the token ceiling; "
            "JSON may have been truncated."
        )

    result = _extract_json(
        output_text
    )

    status = str(
        result.get(
            "status",
            "UNKNOWN",
        )
    ).upper()

    if status not in {
        "SUPPORTED",
        "CONFLICT",
        "UNKNOWN",
    }:
        status = "UNKNOWN"

    try:
        confidence = float(
            result.get(
                "confidence",
                0.0,
            )
        )
    except Exception:
        confidence = 0.0

    confidence = max(
        0.0,
        min(
            1.0,
            confidence,
        ),
    )

    evidence = []

    for frame_id in (
        result.get(
            "evidence_frames",
            [],
        )
        or []
    ):

        try:
            idx = (
                int(frame_id)
                - 1
            )
        except Exception:
            continue

        if (
            0
            <= idx
            < len(metadata)
        ):
            evidence.append(
                metadata[idx]
            )

    claims = []

    for raw_claim in (
        result.get(
            "claims",
            [],
        )
        or []
    )[:8]:

        if not isinstance(
            raw_claim,
            dict,
        ):
            continue

        claim_status = str(
            raw_claim.get(
                "status",
                "UNKNOWN",
            )
        ).upper()

        if claim_status not in {
            "SUPPORTED",
            "CONFLICT",
            "UNKNOWN",
        }:
            claim_status = "UNKNOWN"

        claim_frame_ids = []

        for value in (
            raw_claim.get(
                "evidence_frames",
                [],
            )
            or []
        ):
            try:
                frame_id = int(
                    value
                )
            except Exception:
                continue

            if (
                1
                <= frame_id
                <= len(metadata)
            ):
                claim_frame_ids.append(
                    frame_id
                )

        sources = [
            str(source).lower()
            for source in (
                raw_claim.get(
                    "sources",
                    [],
                )
                or []
            )
            if str(source).lower()
            in {
                "visual",
                "ocr",
                "asr",
            }
        ]

        claims.append(
            {
                "claim":
                    str(
                        raw_claim.get(
                            "claim",
                            "",
                        )
                    ).strip(),

                "status":
                    claim_status,

                "sources":
                    list(
                        dict.fromkeys(
                            sources
                        )
                    ),

                "evidence_frames":
                    claim_frame_ids,

                "evidence_text":
                    [
                        str(value).strip()
                        for value in (
                            raw_claim.get(
                                "evidence_text",
                                [],
                            )
                            or []
                        )[:1]
                        if str(
                            value
                        ).strip()
                    ],

                "reason":
                    str(
                        raw_claim.get(
                            "reason",
                            "",
                        )
                    ).strip(),
            }
        )

    verification = {
        "status":
            status,

        "confidence":
            confidence,

        "original_query":
            verification_query,

        "search_query":
            search_query,

        "claims":
            claims,

        "explanation":
            str(
                result.get(
                    "explanation",
                    "",
                )
            ).strip(),

        "evidence":
            evidence,

        "ocr_evidence":
            list(
                ocr_evidence
                or []
            ),

        "asr_evidence":
            list(
                asr_evidence
                or []
            ),

        "modalities_used": {
            "visual":
                len(
                    evidence_frames
                )
                > 0,

            "ocr":
                bool(
                    ocr_evidence
                ),

            "asr":
                bool(
                    asr_evidence
                ),
        },

        "segment": {
            "video_id":
                segment.video_id,

            "start_time":
                float(
                    segment.start_time
                ),

            "end_time":
                float(
                    segment.end_time
                ),

            "anchor_time":
                float(
                    segment.anchor_timestamp
                ),
        },

        "num_frames_sent":
            len(
                evidence_frames
            ),

        "latency_ms":
            latency_ms,

        "peak_gpu1_memory_gb":
            peak_memory_gb,

        "raw_output":
            output_text,

        "parser_fallback":
            bool(
                result.get(
                    "_parser_fallback",
                    False,
                )
            ),

        "parser_error":
            str(
                result.get(
                    "_parser_error",
                    "",
                )
            ),
    }

    del inputs
    del generated_ids
    del generated_trimmed

    gc.collect()
    torch.cuda.empty_cache()

    return verification


print(
    "Original-intent multimodal Qwen verifier ready."
)


In [ ]:
# ============================================================
# V8 - TEST VIDEO VERIFICATION
# ============================================================

import gc
import torch


gc.collect()

torch.cuda.empty_cache()


verification = (
    verify_segment_vlm_local(

        query=QUERY,

        segment=segment,

        # START SAFE
        max_frames=3,
    )
)


print(
    "\n"
    + "=" * 70
)


print(
    "STATUS:",
    verification[
        "status"
    ],
)


print(
    "CONFIDENCE:",
    verification[
        "confidence"
    ],
)


print(
    "FRAMES SENT:",
    verification[
        "num_frames_sent"
    ],
)


print(
    "LATENCY:",
    round(
        verification[
            "latency_ms"
        ],
        1,
    ),
    "ms",
)


print(
    "PEAK GPU1:",
    round(
        verification[
            "peak_gpu1_memory_gb"
        ],
        2,
    ),
    "GB",
)


print(
    "\nEXPLANATION:"
)


print(
    verification[
        "explanation"
    ]
)


print(
    "\nEVIDENCE:"
)


for evidence in (
    verification[
        "evidence"
    ]
):

    print(
        f"frame="
        f"{evidence['frame_id']} | "
        f"time="
        f"{evidence['timestamp']:.1f}s | "
        f"shot="
        f"{evidence['shot_id']}"
    )


print(
    "=" * 70
)

In [ ]:
# ============================================================
# V9 - VERIFICATION RESULT FOR UI
# ============================================================


def get_verification_for_ui(
    verification,
    show_explanation=False,
):

    result = {

        "status":
            verification[
                "status"
            ],

        "confidence":
            verification[
                "confidence"
            ],

        "video_id":
            verification[
                "segment"
            ][
                "video_id"
            ],

        "start_time":
            verification[
                "segment"
            ][
                "start_time"
            ],

        "end_time":
            verification[
                "segment"
            ][
                "end_time"
            ],
    }


    if show_explanation:

        result[
            "explanation"
        ] = verification[
            "explanation"
        ]


        result[
            "evidence"
        ] = [

            {
                "frame_id":
                    x[
                        "frame_id"
                    ],

                "timestamp":
                    x[
                        "timestamp"
                    ],

                "shot_id":
                    x[
                        "shot_id"
                    ],
            }

            for x in (
                verification[
                    "evidence"
                ]
            )
        ]


    return result

In [ ]:
# ============================================================
# FASTAPI BACKEND — COMPLETE SINGLE CELL
# Search + feedback + neighbor frames + image mount + video streaming + ngrok.
# Replace the whole old backend/ngrok block with this one cell.
# ============================================================

!pip install -q -U fastapi uvicorn pyngrok nest_asyncio python-multipart pydantic
import mimetypes
import os
import re
import threading
from pathlib import Path
from time import perf_counter
from typing import Any, Iterator
from uuid import uuid4
import hashlib
import nest_asyncio
import pandas as pd
from PIL import Image, ImageOps
import uvicorn
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, Response, StreamingResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel, Field
from pyngrok import ngrok
from fastapi.middleware.gzip import GZipMiddleware
from concurrent.futures import ThreadPoolExecutor
import copy

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
DEFAULT_VIDEO_ROOT = Path("/kaggle/input/datasets/sscarecrow/aic2025")

VIDEO_EXTENSIONS = {
    ".mp4",
    ".m4v",
    ".mov",
    ".webm",
    ".mkv",
    ".avi",
}

# Lower number = preferred when duplicate video IDs exist.
VIDEO_EXTENSION_PRIORITY = {
    ".mp4": 0,
    ".m4v": 1,
    ".webm": 2,
    ".mov": 3,
    ".mkv": 4,
    ".avi": 5,
}

BROWSER_FRIENDLY_VIDEO_EXTENSIONS = {
    ".mp4",
    ".m4v",
    ".webm",
    ".mov",
}

VIDEO_ID_PATTERN = re.compile(
    r"(?<![A-Z0-9])(?P<video_id>L\d{2}_V\d{3})(?![A-Z0-9])",
    re.IGNORECASE,
)

STREAM_CHUNK_SIZE = 1024 * 1024

def candidate_video_roots() -> list[Path]:
    """
    Bỏ tính năng tự động quét. Trỏ thẳng đích danh vào thư mục chứa video.
    """
    return [Path("/kaggle/input/datasets/sscarecrow/aic2025/Batch"), Path("/kaggle/input/datasets/lamnguynx/aic2026-l25/video")]


def video_path_preference(path: Path) -> tuple[int, int, int]:
    """Prefer browser-friendly codecs/containers and then shorter paths."""
    extension_rank = VIDEO_EXTENSION_PRIORITY.get(
        path.suffix.lower(),
        99,
    )
    video_folder_bonus = 0 if "video" in str(path.parent).lower() else 1
    return (
        extension_rank,
        video_folder_bonus,
        len(str(path)),
    )


def build_video_index() -> tuple[dict[str, Path], list[Path]]:
    """
    Chỉ quét đúng file mp4, không dùng rglob("*") để tránh quét nhầm file ảnh .jpg
    """
    index: dict[str, Path] = {}
    scanned_roots: list[Path] = []

    for root in candidate_video_roots():
        if not root.is_dir():
            continue

        scanned_roots.append(root)

        # Đổi rglob("*") thành rglob("*.mp4") để bỏ qua ngay lập tức các thư mục ảnh
        for path in root.rglob("*.mp4"): 
            match = VIDEO_ID_PATTERN.search(path.stem)
            if match is None:
                continue

            video_id = match.group("video_id").upper()
            current = index.get(video_id)

            if current is None or video_path_preference(path) < video_path_preference(current):
                index[video_id] = path

    return index, scanned_roots


VIDEO_INDEX, VIDEO_ROOTS_SCANNED = build_video_index()


def rebuild_video_index() -> None:
    global VIDEO_INDEX, VIDEO_ROOTS_SCANNED
    VIDEO_INDEX, VIDEO_ROOTS_SCANNED = build_video_index()


def resolve_video(video_id: str) -> Path | None:
    normalized_video_id = video_id.strip().upper()

    if VIDEO_ID_PATTERN.fullmatch(normalized_video_id) is None:
        return None

    path = VIDEO_INDEX.get(normalized_video_id)
    return path if path is not None and path.is_file() else None

In [ ]:
from bisect import bisect_left
import time 
# ------------------------------------------------------------
# 2. HTTP RANGE STREAMING
# ------------------------------------------------------------

def parse_range_header(
    range_header: str,
    file_size: int,
) -> tuple[int, int]:
    """
    Parse one HTTP byte range.

    Supported:
        bytes=0-999
        bytes=1000-
        bytes=-500

    Multiple ranges are deliberately rejected because the HTML <video>
    element only needs normal single-range responses for seeking.
    """
    if file_size <= 0:
        raise ValueError("Video file is empty.")

    normalized = range_header.strip()

    if "," in normalized:
        raise ValueError("Multiple byte ranges are not supported.")

    match = re.fullmatch(
        r"bytes=(?P<start>\d*)-(?P<end>\d*)",
        normalized,
    )
    if match is None:
        raise ValueError("Invalid Range header.")

    start_raw = match.group("start")
    end_raw = match.group("end")

    if start_raw == "" and end_raw == "":
        raise ValueError("Empty Range header.")

    if start_raw == "":
        suffix_length = int(end_raw)
        if suffix_length <= 0:
            raise ValueError("Invalid suffix byte range.")

        start = max(file_size - suffix_length, 0)
        end = file_size - 1
    else:
        start = int(start_raw)
        end = int(end_raw) if end_raw else file_size - 1

    if start >= file_size:
        raise ValueError("Requested range starts after the file end.")

    end = min(end, file_size - 1)

    if start < 0 or end < start:
        raise ValueError("Requested range is not satisfiable.")

    return start, end


def iter_file_range(
    path: Path,
    start: int,
    end: int,
    chunk_size: int = STREAM_CHUNK_SIZE,
) -> Iterator[bytes]:
    with path.open("rb") as file:
        file.seek(start)
        remaining = end - start + 1

        while remaining > 0:
            chunk = file.read(min(chunk_size, remaining))
            if not chunk:
                break

            remaining -= len(chunk)
            yield chunk


def video_media_type(path: Path) -> str:
    guessed, _ = mimetypes.guess_type(path.name)

    if guessed:
        return guessed

    fallback_by_extension = {
        ".mkv": "video/x-matroska",
        ".m4v": "video/x-m4v",
    }
    return fallback_by_extension.get(
        path.suffix.lower(),
        "application/octet-stream",
    )


def stream_video_response(
    video_path: Path,
    request: Request,
):
    file_size = video_path.stat().st_size
    media_type = video_media_type(video_path)
    safe_filename = video_path.name.replace('"', "")
    range_header = request.headers.get("range")

    common_headers = {
        "Accept-Ranges": "bytes",
        "Cache-Control": "public, max-age=3600",
        "Content-Disposition": f'inline; filename="{safe_filename}"',
    }

    if request.method == "HEAD":
        return Response(
            status_code=200,
            media_type=media_type,
            headers={
                **common_headers,
                "Content-Length": str(file_size),
            },
        )

    if not range_header:
        return FileResponse(
            path=str(video_path),
            media_type=media_type,
            headers=common_headers,
        )

    try:
        start, end = parse_range_header(
            range_header,
            file_size,
        )
    except ValueError as error:
        raise HTTPException(
            status_code=416,
            detail=str(error),
            headers={
                "Accept-Ranges": "bytes",
                "Content-Range": f"bytes */{file_size}",
            },
        ) from error

    content_length = end - start + 1

    return StreamingResponse(
        iter_file_range(video_path, start, end),
        status_code=206,
        media_type=media_type,
        headers={
            **common_headers,
            "Content-Range": (
                f"bytes {start}-{end}/{file_size}"
            ),
            "Content-Length": str(content_length),
        },
    )


# ------------------------------------------------------------
# 3. REQUEST CONTRACT
# ------------------------------------------------------------

class SearchSceneRequest(BaseModel):
    scene_id: int = Field(default=1, ge=1)
    visual_query: str = ""
    ocr_query: str = ""
    asr_query: str = ""


class SearchOptions(BaseModel):
    top_k_per_modality: int = Field(
        default=100,
        ge=1,
        le=1000,
    )
    display_result_count: int = Field(
        default=20,
        ge=1,
        le=500,
    )
    use_query_rewrite: bool = True
    visual_models: list[VisualSearchModel] = Field(
        default_factory=lambda: list(DEFAULT_VISUAL_MODELS)
    )

    # When enabled, /api/v1/search returns every result together
    # with neighboring frames in the SAME JSON response.
    include_neighbors: bool = False
    # Default behavior: initial search returns center frames immediately;
    # neighbors are fetched only when a frame is opened/clicked.
    lazy_neighbors: bool = True
    neighbor_radius: int = Field(
        default=3,
        ge=1,
        le=50,
    )


class SearchRequest(BaseModel):
    """
    Compatible with both:
      1) old notebook request:
         {"query": "...", "top_k": 100}

      2) current React request:
         {
           "task": "task_1",
           "scenes": [...],
           "options": {...}
         }
    """
    query: str | None = None
    top_k: int | None = Field(
        default=None,
        ge=1,
        le=1000,
    )

    task: str = "task_1"
    feedback: str = ""
    scenes: list[SearchSceneRequest] = Field(
        default_factory=list
    )
    options: SearchOptions = Field(
        default_factory=SearchOptions
    )

    def _join_scene_field(self, field_name: str) -> str:
        values: list[str] = []

        for scene in self.scenes:
            value = str(getattr(scene, field_name, "")).strip()
            if value:
                values.append(value)

        return " ".join(values).strip()

    def resolved_visual_query(self) -> str:
        # Legacy {"query": "..."} remains a visual query.
        if self.query and self.query.strip():
            return self.query.strip()

        return self._join_scene_field("visual_query")

    def resolved_ocr_query(self) -> str:
        # OCR query must stay in its original language; do not translate it.
        if self.query is not None and not self.scenes:
            return ""

        return self._join_scene_field("ocr_query")

    def resolved_asr_query(self) -> str:
        if self.query is not None and not self.scenes:
            return ""

        return self._join_scene_field("asr_query")

    def resolved_query(self) -> str:
        """
        Full intent for logging/query reconstruction.
        Retrieval modalities are still executed separately.
        """
        parts = [
            self.resolved_visual_query(),
            self.resolved_ocr_query(),
            self.resolved_asr_query(),
        ]

        return " ".join(
            part for part in parts if part
        ).strip()

    def resolved_top_k(self) -> int:
        return self.top_k or self.options.top_k_per_modality

    def resolved_result_count(self) -> int:
        if self.query is not None and not self.scenes:
            return 20

        return self.options.display_result_count



class QueryRewriteRequest(BaseModel):
    original_query: str = Field(min_length=1)
    feedback_history: list[dict[str, Any]] = Field(
        default_factory=list
    )
    previous_state: dict[str, Any] | None = None


class OCRSearchRequest(BaseModel):
    query: str = Field(min_length=1)
    top_k: int = Field(default=100, ge=1, le=1000)
    display_result_count: int = Field(
        default=20,
        ge=1,
        le=500,
    )


class ASRSearchRequest(BaseModel):
    query: str = Field(min_length=1)
    top_k: int = Field(default=100, ge=1, le=1000)
    display_result_count: int = Field(
        default=20,
        ge=1,
        le=500,
    )


class FeedbackSearchRequest(BaseModel):
    search_id: str = Field(min_length=1)
    selected_frame_ref: str = Field(min_length=1)
    feedback: str = Field(min_length=1)
    top_k: int = Field(default=100, ge=1, le=1000)
    display_result_count: int = Field(
        default=20,
        ge=1,
        le=500,
    )


class NeighborBatchRequest(BaseModel):
    """
    Fallback endpoint contract. The main frontend flow should prefer
    neighbors already embedded in POST /api/v1/search.
    """
    frame_refs: list[str]
    neighbor_radius: int = Field(
        default=3,
        ge=1,
        le=50,
    )


FRAME_REFERENCE_CACHE: dict[str, str] = {}
FRAME_PATH_REFERENCE_CACHE: dict[str, str] = {}
SEARCH_SESSION_CACHE: dict[str, dict[str, Any]] = {}

# One entry stores the already-sorted frame list for one video directory.
# Key format: VIDEO_ID|/absolute/frame/directory
FRAME_DIRECTORY_INDEX_CACHE: dict[str, dict[str, Any]] = {}
FRAME_DIRECTORY_SCAN_COUNT: dict[str, int] = {}

MAX_FRAME_REFERENCE_CACHE = 20000
MAX_SEARCH_SESSION_CACHE = 100
MAX_FRAME_DIRECTORY_INDEX_CACHE = 512

# Expensive GPU/CPU retrieval is serialized across API requests so two
# browser clicks cannot concurrently fight for VRAM/model state.
RETRIEVAL_EXECUTION_LOCK = threading.Lock()

# Force the fast UX even when an older frontend still sends
# include_neighbors=true. Set AIC_FORCE_LAZY_NEIGHBORS=0 to restore legacy.
FORCE_LAZY_NEIGHBORS = os.getenv(
    "AIC_FORCE_LAZY_NEIGHBORS",
    "1",
).strip().lower() not in {"0", "false", "no"}

THUMBNAIL_CACHE_ROOT = Path(
    os.getenv(
        "AIC_THUMBNAIL_CACHE",
        "/kaggle/working/aic_thumbnail_cache",
    )
)
THUMBNAIL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
THUMBNAIL_DEFAULT_SIZE = int(
    os.getenv("AIC_THUMBNAIL_SIZE", "480")
)
THUMBNAIL_DEFAULT_QUALITY = int(
    os.getenv("AIC_THUMBNAIL_QUALITY", "82")
)

FRAME_IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
}

NEIGHBOR_FRAME_PATTERN = re.compile(
    r"(?P<video_id>L\d{2}_V\d{3})"
    r"_t(?P<timestamp>\d+(?:\.\d+)?)"
    r"_f(?P<frame_idx>\d+)"
    r"_s(?P<shot_id>\d+)",
    re.IGNORECASE,
)


def trim_in_memory_cache(
    cache: dict,
    max_size: int,
) -> None:
    while len(cache) > max_size:
        oldest_key = next(iter(cache))
        cache.pop(oldest_key, None)


def canonical_frame_path(
    local_image_path: str | Path,
) -> str:
    path = Path(local_image_path)
    try:
        return str(path.resolve(strict=True))
    except (FileNotFoundError, OSError):
        return str(path.resolve(strict=False))


def trim_frame_reference_cache() -> None:
    while len(FRAME_REFERENCE_CACHE) > MAX_FRAME_REFERENCE_CACHE:
        oldest_ref = next(iter(FRAME_REFERENCE_CACHE))
        oldest_path = FRAME_REFERENCE_CACHE.pop(oldest_ref, None)

        if oldest_path is not None:
            canonical = canonical_frame_path(oldest_path)
            if FRAME_PATH_REFERENCE_CACHE.get(canonical) == oldest_ref:
                FRAME_PATH_REFERENCE_CACHE.pop(canonical, None)


def register_frame_reference(
    local_image_path: str | Path,
) -> str:
    """
    Return one stable in-memory reference per local frame path.

    Reusing the same frame_ref prevents the center frame and its neighbor
    copy from receiving different references in one response.
    """
    canonical = canonical_frame_path(local_image_path)
    existing_ref = FRAME_PATH_REFERENCE_CACHE.get(canonical)

    if (
        existing_ref is not None
        and FRAME_REFERENCE_CACHE.get(existing_ref) == canonical
    ):
        return existing_ref

    frame_ref = uuid4().hex
    FRAME_REFERENCE_CACHE[frame_ref] = canonical
    FRAME_PATH_REFERENCE_CACHE[canonical] = frame_ref
    trim_frame_reference_cache()
    return frame_ref


# ------------------------------------------------------------
# 4. FASTAPI APP
# ------------------------------------------------------------

app = FastAPI(
    title="AIC Retrieval API",
    version="4.0.0",
)

_extra_frontend_origins = [
    origin.strip()
    for origin in os.getenv("FRONTEND_ORIGINS", "").split(",")
    if origin.strip()
]
FRONTEND_ORIGINS = list(
    dict.fromkeys(
        [
            "http://localhost:5173",
            "http://127.0.0.1:5173",
            *_extra_frontend_origins,
        ]
    )
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=FRONTEND_ORIGINS,
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
    expose_headers=[
        "Accept-Ranges",
        "Content-Range",
        "Content-Length",
    ],
)

# Compress JSON/search metadata over ngrok. JPEG/video payloads are already
# compressed and are served separately.
app.add_middleware(
    GZipMiddleware,
    minimum_size=1024,
    compresslevel=4,
)

if not KAGGLE_INPUT_ROOT.is_dir():
    raise RuntimeError(
        "Missing /kaggle/input. Run this backend inside Kaggle."
    )

app.mount(
    "/images",
    StaticFiles(directory=str(KAGGLE_INPUT_ROOT)),
    name="images",
)


def image_public_url(
    request: Request,
    local_path: str | Path,
) -> str | None:
    """
    Return an absolute URL only when the frame is an existing file
    inside /kaggle/input. This prevents bad paths and traversal.
    """
    candidate = Path(local_path)

    try:
        resolved_candidate = candidate.resolve(strict=True)
        resolved_root = KAGGLE_INPUT_ROOT.resolve(strict=True)
        relative = resolved_candidate.relative_to(resolved_root)
    except (FileNotFoundError, ValueError, OSError):
        return None

    return str(
        request.url_for(
            "images",
            path=relative.as_posix(),
        )
    )


def format_timestamp_display(seconds: float) -> str:
    """Tự xử lý format thời gian ngay tại đây"""
    total_seconds = int(seconds)
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    
    if hours > 0:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"




def _thumbnail_cache_file(
    local_path: str | Path,
    max_size: int,
    quality: int,
) -> Path:
    source = Path(local_path)
    stat = source.stat()
    fingerprint = (
        f"{canonical_frame_path(source)}|"
        f"{stat.st_size}|{stat.st_mtime_ns}|"
        f"{max_size}|{quality}"
    )
    digest = hashlib.sha1(
        fingerprint.encode("utf-8")
    ).hexdigest()
    return THUMBNAIL_CACHE_ROOT / f"{digest}.jpg"


def ensure_thumbnail(
    local_path: str | Path,
    max_size: int = THUMBNAIL_DEFAULT_SIZE,
    quality: int = THUMBNAIL_DEFAULT_QUALITY,
) -> Path:
    """
    Create a compact cached JPEG for result grids.

    This happens only when the browser requests a thumbnail, after the
    search JSON has already returned.
    """
    max_size = max(160, min(int(max_size), 1280))
    quality = max(55, min(int(quality), 95))

    source = Path(local_path)
    if not source.is_file():
        raise FileNotFoundError(source)

    cached = _thumbnail_cache_file(
        source,
        max_size=max_size,
        quality=quality,
    )
    if cached.is_file():
        return cached

    tmp = cached.with_name(
        f"{cached.stem}.{threading.get_ident()}.tmp.jpg"
    )
    with Image.open(source) as image:
        image = ImageOps.exif_transpose(image)
        if image.mode != "RGB":
            image = image.convert("RGB")
        image.thumbnail(
            (max_size, max_size),
            Image.Resampling.LANCZOS,
        )
        image.save(
            tmp,
            format="JPEG",
            quality=quality,
            progressive=True,
            optimize=False,
        )

    if cached.is_file():
        try:
            tmp.unlink(missing_ok=True)
        except Exception:
            pass
    else:
        tmp.replace(cached)
    return cached


def thumbnail_public_url(
    request: Request,
    frame_ref: str,
) -> str:
    return str(
        request.url_for(
            "frame_thumbnail",
            frame_ref=frame_ref,
        )
    )


def parse_frame_file_metadata(
    frame_path: str | Path,
    expected_video_id: str | None = None,
) -> dict[str, Any] | None:
    """
    Parse metadata directly from the AutoShot frame filename.

    Example:
      L21_V002_t00005.600_f00000168_s000001.jpg
    """
    path = Path(frame_path)
    match = NEIGHBOR_FRAME_PATTERN.search(path.name)

    if match is None:
        return None

    video_id = match.group("video_id").upper()

    if (
        expected_video_id is not None
        and video_id != expected_video_id.strip().upper()
    ):
        return None

    return {
        "local_image_path": canonical_frame_path(path),
        "video_id": video_id,
        "frame_index": int(match.group("frame_idx")),
        "timestamp_seconds": float(match.group("timestamp")),
        "shot_id": int(match.group("shot_id")),
    }


def frame_directory_cache_key(
    video_id: str,
    directory: str | Path,
) -> str:
    normalized_video_id = video_id.strip().upper()
    canonical_directory = canonical_frame_path(directory)
    return f"{normalized_video_id}|{canonical_directory}"


def get_cached_frame_directory_index(
    center_path: str | Path,
    video_id: str,
) -> tuple[str, dict[str, Any]]:
    """
    Scan one frame directory only once, then reuse its sorted frame index.

    Twenty results belonging to the same video/directory share this cache
    entry instead of globbing the directory twenty times.
    """
    center = Path(center_path)
    directory = center.parent
    cache_key = frame_directory_cache_key(video_id, directory)

    cached = FRAME_DIRECTORY_INDEX_CACHE.get(cache_key)
    if cached is not None:
        cached["cache_hits"] += 1
        return cache_key, cached

    if not directory.is_dir():
        raise FileNotFoundError(
            f"Frame directory was not found: {directory}"
        )

    records: list[dict[str, Any]] = []

    # iterdir() performs one directory scan. Results are cached below.
    for candidate in directory.iterdir():
        if (
            not candidate.is_file()
            or candidate.suffix.lower() not in FRAME_IMAGE_EXTENSIONS
        ):
            continue

        metadata = parse_frame_file_metadata(
            candidate,
            expected_video_id=video_id,
        )
        if metadata is not None:
            records.append(metadata)

    records.sort(
        key=lambda item: (
            item["frame_index"],
            item["timestamp_seconds"],
            item["local_image_path"],
        )
    )

    path_to_position = {
        item["local_image_path"]: index
        for index, item in enumerate(records)
    }
    frame_indices = [
        item["frame_index"]
        for item in records
    ]

    cache_entry = {
        "video_id": video_id.strip().upper(),
        "directory": canonical_frame_path(directory),
        "records": records,
        "frame_indices": frame_indices,
        "path_to_position": path_to_position,
        "cache_hits": 0,
    }

    FRAME_DIRECTORY_INDEX_CACHE[cache_key] = cache_entry
    FRAME_DIRECTORY_SCAN_COUNT[cache_key] = (
        FRAME_DIRECTORY_SCAN_COUNT.get(cache_key, 0) + 1
    )
    trim_in_memory_cache(
        FRAME_DIRECTORY_INDEX_CACHE,
        MAX_FRAME_DIRECTORY_INDEX_CACHE,
    )

    return cache_key, cache_entry


def find_center_position(
    cache_entry: dict[str, Any],
    center_path: str | Path,
    frame_index: int,
) -> int | None:
    canonical_center = canonical_frame_path(center_path)
    direct_position = cache_entry["path_to_position"].get(
        canonical_center
    )

    if direct_position is not None:
        return int(direct_position)

    # Fallback to frame_index when the same path has a different mount prefix.
    frame_indices = cache_entry["frame_indices"]
    position = bisect_left(frame_indices, int(frame_index))

    if (
        position < len(frame_indices)
        and frame_indices[position] == int(frame_index)
    ):
        return position

    return None


def resolve_kaggle_output_metadata(
    local_image_path: str | Path,
) -> dict[str, str | None]:
    """
    Resolve the Kaggle notebook Output page that supplied a frame.

    Expected mounted path:
    /kaggle/input/notebooks/<owner>/<notebook-slug>/<relative-path>
    """
    normalized_path = str(local_image_path).replace("\\", "/")
    notebook_prefix = "/kaggle/input/notebooks/"

    if not normalized_path.startswith(notebook_prefix):
        return {
            "kaggle_output_url": None,
            "kaggle_relative_path": None,
        }

    remainder = normalized_path[len(notebook_prefix):]
    parts = remainder.split("/", 2)

    if len(parts) < 2:
        return {
            "kaggle_output_url": None,
            "kaggle_relative_path": None,
        }

    owner = parts[0].strip()
    notebook_slug = parts[1].strip()

    if not owner or not notebook_slug:
        return {
            "kaggle_output_url": None,
            "kaggle_relative_path": None,
        }

    relative_path = (
        parts[2].strip()
        if len(parts) >= 3 and parts[2].strip()
        else None
    )

    return {
        "kaggle_output_url": (
            f"https://www.kaggle.com/code/"
            f"{owner}/{notebook_slug}/output"
        ),
        "kaggle_relative_path": relative_path,
    }


def serialize_neighbor_record(
    metadata: dict[str, Any],
    center_path: str | Path,
    request: Request,
) -> dict[str, Any] | None:
    local_image_path = metadata["local_image_path"]
    image_url = image_public_url(
        request,
        local_image_path,
    )

    if image_url is None:
        return None

    frame_ref = register_frame_reference(local_image_path)
    timestamp_seconds = float(
        metadata["timestamp_seconds"]
    )
    kaggle_source = resolve_kaggle_output_metadata(
        local_image_path
    )

    return {
        "frame_ref": frame_ref,
        "video_id": str(metadata["video_id"]).upper(),
        "frame_index": int(metadata["frame_index"]),
        "timestamp_seconds": timestamp_seconds,
        "timestamp_display": format_timestamp_display(
            timestamp_seconds
        ),
        "image_url": image_url,
        **kaggle_source,
        "shot_id": (
            int(metadata["shot_id"])
            if metadata.get("shot_id") is not None
            else None
        ),
        "is_center": (
            canonical_frame_path(local_image_path)
            == canonical_frame_path(center_path)
        ),
    }


def get_neighbor_frames_for_center(
    center_path: str | Path,
    video_id: str,
    frame_index: int,
    neighbor_radius: int,
    request: Request,
) -> tuple[list[dict[str, Any]], set[str], str | None]:
    """
    Return up to neighbor_radius frames before and after the center.

    The center itself is included and identified by is_center=True.
    """
    cache_key, cache_entry = get_cached_frame_directory_index(
        center_path=center_path,
        video_id=video_id,
    )

    center_position = find_center_position(
        cache_entry=cache_entry,
        center_path=center_path,
        frame_index=frame_index,
    )

    if center_position is None:
        return [], set(), cache_key

    records = cache_entry["records"]
    start = max(0, center_position - neighbor_radius)
    stop = min(
        len(records),
        center_position + neighbor_radius + 1,
    )

    neighbors: list[dict[str, Any]] = []
    neighbor_refs: set[str] = set()

    for metadata in records[start:stop]:
        serialized = serialize_neighbor_record(
            metadata=metadata,
            center_path=center_path,
            request=request,
        )
        if serialized is None:
            continue

        neighbors.append(serialized)
        neighbor_refs.add(serialized["frame_ref"])

    neighbors.sort(key=lambda item: item["frame_index"])
    return neighbors, neighbor_refs, cache_key


def attach_neighbors_to_results_batch(
    results_json: list[dict[str, Any]],
    center_paths: list[str],
    request: Request,
    neighbor_radius: int,
) -> tuple[set[str], dict[str, Any]]:
    """
    Batch-enrich all search results inside one backend operation.

    The frontend still makes exactly one POST /api/v1/search request.
    Internally, unique video directories are indexed once and reused.
    """
    all_neighbor_refs: set[str] = set()
    used_cache_keys: set[str] = set()
    center_count = min(len(results_json), len(center_paths))

    for result, center_path in zip(
        results_json,
        center_paths,
    ):
        neighbors, neighbor_refs, cache_key = (
            get_neighbor_frames_for_center(
                center_path=center_path,
                video_id=result["video_id"],
                frame_index=result["frame_index"],
                neighbor_radius=neighbor_radius,
                request=request,
            )
        )
        result["neighbors"] = neighbors
        all_neighbor_refs.update(neighbor_refs)

        if cache_key is not None:
            used_cache_keys.add(cache_key)

    return all_neighbor_refs, {
        "included": True,
        "neighbor_radius": neighbor_radius,
        "center_count": center_count,
        "unique_frame_directories": len(used_cache_keys),
        "directory_scan_counts": {
            cache_key: FRAME_DIRECTORY_SCAN_COUNT.get(
                cache_key,
                0,
            )
            for cache_key in sorted(used_cache_keys)
        },
    }


def attach_neighbor_refs_to_matching_sessions(
    center_refs: set[str],
    neighbor_refs: set[str],
) -> None:
    """
    Let fallback batch-neighbor results participate in feedback.

    Any search session containing one of the requested center refs is
    expanded with the newly registered neighbor refs.
    """
    if not center_refs or not neighbor_refs:
        return

    for session in SEARCH_SESSION_CACHE.values():
        current_refs = session.get("current_frame_refs", set())
        if current_refs.intersection(center_refs):
            current_refs.update(neighbor_refs)


def serialize_result_dataframe(
    result_df: pd.DataFrame,
    request: Request,
    include_neighbors: bool = False,
    neighbor_radius: int = 3,
) -> tuple[
    list[dict],
    list[dict[str, str]],
    set[str],
    dict[str, Any],
]:
    """
    Convert internal DataFrame rows to the React API contract.

    When include_neighbors=True, all center results and their neighbor
    metadata are produced in this same serialization pass and returned
    by the same POST /api/v1/search response.
    """
    warnings: list[dict[str, str]] = []
    results_json: list[dict] = []
    frame_refs: set[str] = set()
    center_paths: list[str] = []

    neighbor_summary: dict[str, Any] = {
        "included": include_neighbors,
        "neighbor_radius": neighbor_radius,
        "center_count": 0,
        "unique_frame_directories": 0,
        "directory_scan_counts": {},
    }

    if result_df is None or result_df.empty:
        return (
            results_json,
            warnings,
            frame_refs,
            neighbor_summary,
        )

    missing_frame_count = 0
    missing_video_ids: set[str] = set()

    for _, row in result_df.iterrows():
        local_image_path = str(row["local_image_path"])
        frame_url = image_public_url(
            request,
            local_image_path,
        )

        if frame_url is None:
            missing_frame_count += 1
            continue

        frame_ref = register_frame_reference(
            local_image_path
        )
        thumbnail_url = thumbnail_public_url(
            request,
            frame_ref,
        )

        kaggle_source = resolve_kaggle_output_metadata(
            local_image_path
        )

        video_id = str(row["video_id_fixed"]).upper()
        timestamp_seconds = float(row["timestamp_fixed"])
        frame_index = int(row["frame_idx_fixed"])
        video_path = resolve_video(video_id)
        video_available = video_path is not None

        if not video_available:
            missing_video_ids.add(video_id)

        video_url = (
            str(
                request.url_for(
                    "stream_video",
                    video_id=video_id,
                )
            )
            if video_available
            else None
        )

        frame_refs.add(frame_ref)

        rank = len(results_json) + 1

        asr_start_value = row.get("asr_start_time")
        asr_start_time = (
            float(asr_start_value)
            if asr_start_value is not None
            and pd.notna(asr_start_value)
            else None
        )
        playback_time_seconds = (
            asr_start_time
            if asr_start_time is not None
            else timestamp_seconds
        )

        timestamp_display = format_timestamp_display(
            playback_time_seconds
        )
        score = float(row["rrf_score"])

        fusion_sources = row.get("fusion_sources", [])
        if isinstance(fusion_sources, np.ndarray):
            fusion_sources = fusion_sources.tolist()
        elif not isinstance(fusion_sources, list):
            fusion_sources = (
                [str(fusion_sources)]
                if pd.notna(fusion_sources)
                else []
            )

        shot_value = row.get("shot_id_fixed")
        shot_id = (
            int(shot_value)
            if pd.notna(shot_value)
            else None
        )

        results_json.append(
            {
                "rank": rank,
                "final_rank": rank,
                "video_id": video_id,
                "frame_index": frame_index,
                "frame_idx": frame_index,
                "frame_ref": frame_ref,
                "frame_timestamp_seconds": timestamp_seconds,
                "timestamp_seconds": playback_time_seconds,
                "playback_time_seconds": playback_time_seconds,
                "timestamp_display": timestamp_display,
                "time": timestamp_display,
                "shot_id": shot_id,
                "thumbnail_url": thumbnail_url,
                "frame_url": frame_url,
                "image_url": frame_url,
                "neighbors_url": str(
                    request.url_for(
                        "get_neighbors_one",
                        frame_ref=frame_ref,
                    )
                ),
                **kaggle_source,
                "video_url": video_url,
                "video_available": video_available,
                "score": score,
                "fusion_sources": fusion_sources,
                "matched_scene": 1,
                "neighbors": [],
                "ocr_text": (
                    str(row.get("ocr_text"))
                    if pd.notna(row.get("ocr_text"))
                    else None
                ),
                "ocr_text_recall": (
                    str(row.get("ocr_text_recall"))
                    if pd.notna(row.get("ocr_text_recall"))
                    else None
                ),
                "ocr_snippet": (
                    str(row.get("ocr_snippet"))
                    if pd.notna(row.get("ocr_snippet"))
                    else None
                ),
                "ocr_search_mode": (
                    str(row.get("ocr_search_mode"))
                    if pd.notna(row.get("ocr_search_mode"))
                    else None
                ),
                "asr_text": (
                    str(row.get("asr_text"))
                    if pd.notna(row.get("asr_text"))
                    else None
                ),
                "asr_segment_index": (
                    int(row.get("asr_segment_index"))
                    if pd.notna(row.get("asr_segment_index"))
                    else None
                ),
                "asr_start_time": asr_start_time,
                "asr_end_time": (
                    float(row.get("asr_end_time"))
                    if pd.notna(row.get("asr_end_time"))
                    else None
                ),
                "asr_start_frame": (
                    int(row.get("asr_start_frame"))
                    if pd.notna(row.get("asr_start_frame"))
                    else None
                ),
                "asr_end_frame": (
                    int(row.get("asr_end_frame"))
                    if pd.notna(row.get("asr_end_frame"))
                    else None
                ),
                "asr_score": (
                    float(row.get("asr_score"))
                    if pd.notna(row.get("asr_score"))
                    else None
                ),
                "modality_scores": {
                    "visual": (
                        score
                        if any(
                            source.startswith("text_")
                            or source == "cir_bge"
                            for source in fusion_sources
                        )
                        else None
                    ),
                    "ocr": (
                        score
                        if "ocr_sqlite" in fusion_sources
                        else None
                    ),
                    "asr": (
                        float(row.get("asr_score"))
                        if "asr_hybrid" in fusion_sources
                        and pd.notna(row.get("asr_score"))
                        else (
                            score
                            if "asr_hybrid" in fusion_sources
                            else None
                        )
                    ),
                },
            }
        )
        center_paths.append(local_image_path)

    if include_neighbors and results_json:
        try:
            neighbor_refs, neighbor_summary = (
                attach_neighbors_to_results_batch(
                    results_json=results_json,
                    center_paths=center_paths,
                    request=request,
                    neighbor_radius=neighbor_radius,
                )
            )
            frame_refs.update(neighbor_refs)
        except Exception as error:
            warnings.append(
                {
                    "module": "neighbors",
                    "code": "NEIGHBOR_BATCH_FAILED",
                    "message": (
                        "Search results were returned, but neighbor "
                        f"enrichment failed: {error}"
                    ),
                }
            )

    if missing_frame_count:
        warnings.append(
            {
                "module": "frames",
                "code": "FRAME_PATH_NOT_FOUND",
                "message": (
                    f"Skipped {missing_frame_count} frame(s) "
                    "because the local path was missing or outside "
                    "/kaggle/input."
                ),
            }
        )

    if missing_video_ids:
        preview = ", ".join(
            sorted(missing_video_ids)[:10]
        )
        suffix = (
            "..."
            if len(missing_video_ids) > 10
            else ""
        )
        warnings.append(
            {
                "module": "videos",
                "code": "VIDEO_NOT_FOUND",
                "message": (
                    "No original video was indexed for: "
                    f"{preview}{suffix}."
                ),
            }
        )

    return (
        results_json,
        warnings,
        frame_refs,
        neighbor_summary,
    )


@app.post("/api/v1/query/reconstruct")
def reconstruct_query_api(
    req: QueryRewriteRequest,
):
    """
    Compatibility endpoint only.

    Query reconstruction has been removed from the current retrieval pipeline.
    Return the original query unchanged so older clients do not crash.
    """
    original_query = req.original_query.strip()

    return {
        "status": "success",
        "query_rewrite_applied": False,
        "query_state": {
            "original_query": original_query,
            "standalone_visual_query_en": original_query,
            "rewrite_applied": False,
        },
    }


@app.get("/api/v1/health")
@app.get("/api/health")
async def health():
    return {
        "status": "ok",
        "service": "aic-retrieval-kaggle",
        "indexed_videos": len(VIDEO_INDEX),
        "video_roots_scanned": [
            str(path)
            for path in VIDEO_ROOTS_SCANNED
        ],
        "sample_video_ids": sorted(VIDEO_INDEX)[:10],
        "ocr": get_ocr_database_stats(),
        "asr": get_asr_stats(),
        "neighbor_cache": {
            "indexed_directories": len(
                FRAME_DIRECTORY_INDEX_CACHE
            ),
            "registered_frame_refs": len(
                FRAME_REFERENCE_CACHE
            ),
            "total_directory_scans": sum(
                FRAME_DIRECTORY_SCAN_COUNT.values()
            ),
            "lazy_neighbors_forced": FORCE_LAZY_NEIGHBORS,
        },
        "visual_search": {
            "parallel": VISUAL_DB_PARALLEL,
            "workers": VISUAL_DB_WORKERS,
            "profile": VISUAL_SEARCH_PROFILE,
            "last_timing": _LAST_VISUAL_TIMINGS,
        },
        "thumbnail_cache": {
            "root": str(THUMBNAIL_CACHE_ROOT),
            "default_size": THUMBNAIL_DEFAULT_SIZE,
            "default_quality": THUMBNAIL_DEFAULT_QUALITY,
        },
    }


@app.post("/api/v1/videos/reindex")
def reindex_videos():
    rebuild_video_index()

    return {
        "status": "ok",
        "indexed_videos": len(VIDEO_INDEX),
        "video_roots_scanned": [
            str(path)
            for path in VIDEO_ROOTS_SCANNED
        ],
    }


@app.get("/api/v1/videos/{video_id}/info")
@app.get("/api/videos/{video_id}/info")
async def get_video_info(video_id: str):
    normalized_video_id = video_id.strip().upper()
    video_path = resolve_video(normalized_video_id)

    if video_path is None:
        raise HTTPException(
            status_code=404,
            detail=(
                f"Video {normalized_video_id} was not found. "
                "Check that the dataset is mounted at "
                "/kaggle/input/aic2025 or set AIC_VIDEO_ROOT, "
                "then call POST /api/v1/videos/reindex."
            ),
        )

    return {
        "video_id": normalized_video_id,
        "filename": video_path.name,
        "extension": video_path.suffix.lower(),
        "size_bytes": video_path.stat().st_size,
        "browser_friendly": (
            video_path.suffix.lower()
            in BROWSER_FRIENDLY_VIDEO_EXTENSIONS
        ),
        # Do not expose the server's absolute local filesystem path.
        "video_url": f"/api/v1/videos/{normalized_video_id}/stream",
    }


@app.api_route(
    "/api/v1/videos/{video_id}/stream",
    methods=["GET", "HEAD"],
    name="stream_video",
)
async def stream_video(
    video_id: str,
    request: Request,
):
    normalized_video_id = video_id.strip().upper()
    video_path = resolve_video(normalized_video_id)

    if video_path is None:
        raise HTTPException(
            status_code=404,
            detail=(
                f"Video {normalized_video_id} was not found. "
                "Check AIC_VIDEO_ROOT and the original video filename."
            ),
        )

    return stream_video_response(
        video_path,
        request,
    )


# Legacy alias kept for the old notebook/UI contract.
@app.api_route(
    "/api/videos/{video_id}",
    methods=["GET", "HEAD"],
    name="stream_video_legacy",
)
async def stream_video_legacy(
    video_id: str,
    request: Request,
):
    normalized_video_id = video_id.strip().upper()
    video_path = resolve_video(normalized_video_id)

    if video_path is None:
        raise HTTPException(
            status_code=404,
            detail=f"Video {normalized_video_id} was not found.",
        )

    return stream_video_response(
        video_path,
        request,
    )


@app.get("/api/v1/videos/{video_id}/clip")
async def get_video_clip(
    video_id: str,
    center_time: float,
    before: float = 5.0,
    after: float = 5.0,
):
    """
    Compatibility endpoint for the existing VideoPlayerModal.
    The updated frontend below no longer needs this extra request,
    but keeping it avoids breaking older builds.
    """
    if center_time < 0 or before < 0 or after < 0:
        raise HTTPException(
            status_code=422,
            detail="center_time, before and after must be non-negative.",
        )

    normalized_video_id = video_id.strip().upper()
    video_path = resolve_video(normalized_video_id)
    video_available = video_path is not None

    return {
        "video_id": normalized_video_id,
        "center_time": center_time,
        "start_time": max(0.0, center_time - before),
        "end_time": center_time + after,
        "video_available": video_available,
        "video_url": (
            f"/api/v1/videos/{normalized_video_id}/stream"
            if video_available
            else None
        ),
        "poster_url": None,
    }



# ------------------------------------------------------------
# 5. LAZY THUMBNAIL + SINGLE NEIGHBOR ENDPOINTS
# ------------------------------------------------------------

@app.get(
    "/api/v1/frames/{frame_ref}/thumbnail",
    name="frame_thumbnail",
)
def frame_thumbnail(
    frame_ref: str,
    max_size: int = THUMBNAIL_DEFAULT_SIZE,
    quality: int = THUMBNAIL_DEFAULT_QUALITY,
):
    local_path = FRAME_REFERENCE_CACHE.get(frame_ref)
    if not local_path:
        raise HTTPException(
            status_code=404,
            detail="Frame reference expired or was not registered.",
        )

    try:
        thumbnail_path = ensure_thumbnail(
            local_path,
            max_size=max_size,
            quality=quality,
        )
    except FileNotFoundError as error:
        raise HTTPException(
            status_code=404,
            detail=str(error),
        ) from error
    except Exception as error:
        raise HTTPException(
            status_code=500,
            detail=f"Thumbnail generation failed: {error}",
        ) from error

    return FileResponse(
        thumbnail_path,
        media_type="image/jpeg",
        headers={
            "Cache-Control": "public, max-age=86400, immutable",
        },
    )


@app.get(
    "/api/v1/frames/{frame_ref}/neighbors",
    name="get_neighbors_one",
)
def get_neighbors_one(
    frame_ref: str,
    request: Request,
    radius: int = 3,
):
    """
    Lazy neighbor lookup for one clicked/opened center frame.
    The first access indexes only that frame's directory; later accesses
    to the same video directory reuse the in-memory cache.
    """
    radius = max(1, min(int(radius), 50))
    center_path = FRAME_REFERENCE_CACHE.get(frame_ref)

    if not center_path:
        raise HTTPException(
            status_code=404,
            detail="Frame reference expired or was not registered.",
        )

    metadata = parse_frame_file_metadata(center_path)
    if metadata is None:
        raise HTTPException(
            status_code=422,
            detail="Could not parse frame metadata from filename.",
        )

    try:
        neighbors, neighbor_refs, cache_key = (
            get_neighbor_frames_for_center(
                center_path=center_path,
                video_id=metadata["video_id"],
                frame_index=metadata["frame_index"],
                neighbor_radius=radius,
                request=request,
            )
        )
    except Exception as error:
        raise HTTPException(
            status_code=500,
            detail=f"Neighbor lookup failed: {error}",
        ) from error

    attach_neighbor_refs_to_matching_sessions(
        center_refs={frame_ref},
        neighbor_refs=neighbor_refs,
    )

    return {
        "status": "success",
        "frame_ref": frame_ref,
        "video_id": metadata["video_id"],
        "frame_index": metadata["frame_index"],
        "neighbor_radius": radius,
        "neighbor_count": len(neighbors),
        "cache_key": cache_key,
        "directory_scan_count": (
            FRAME_DIRECTORY_SCAN_COUNT.get(cache_key, 0)
            if cache_key
            else 0
        ),
        "neighbors": neighbors,
        "data": neighbors,
    }


# ------------------------------------------------------------
# 5. BATCH NEIGHBOR FRAME ENDPOINT (FALLBACK)
# ------------------------------------------------------------

@app.post("/api/v1/frames/neighbors/batch")
def get_neighbors_batch(
    req: NeighborBatchRequest,
    request: Request,
):
    """
    Batch fallback for clients that want neighbors for several frames.
    The preferred low-latency UI uses GET /frames/{frame_ref}/neighbors
    only when the user opens a result.
    """
    if not req.frame_refs:
        raise HTTPException(
            status_code=422,
            detail="frame_refs must contain at least one frame_ref.",
        )

    if len(req.frame_refs) > 100:
        raise HTTPException(
            status_code=422,
            detail="A maximum of 100 frame_ref values is allowed.",
        )

    # Preserve request order while avoiding duplicate work.
    ordered_refs = list(dict.fromkeys(
        frame_ref.strip()
        for frame_ref in req.frame_refs
        if frame_ref and frame_ref.strip()
    ))

    if not ordered_refs:
        raise HTTPException(
            status_code=422,
            detail="frame_refs did not contain any valid value.",
        )

    response_items: list[dict[str, Any]] = []
    all_neighbor_refs: set[str] = set()
    valid_center_refs: set[str] = set()
    used_cache_keys: set[str] = set()

    for center_ref in ordered_refs:
        center_path = FRAME_REFERENCE_CACHE.get(center_ref)

        if not center_path:
            response_items.append(
                {
                    "frame_ref": center_ref,
                    "status": "not_found",
                    "error": (
                        "Frame reference expired or was not registered."
                    ),
                    "neighbors": [],
                }
            )
            continue

        metadata = parse_frame_file_metadata(center_path)

        if metadata is None:
            response_items.append(
                {
                    "frame_ref": center_ref,
                    "status": "invalid_frame_name",
                    "error": (
                        "Could not parse video_id/frame_index from "
                        "the registered frame filename."
                    ),
                    "neighbors": [],
                }
            )
            continue

        try:
            neighbors, neighbor_refs, cache_key = (
                get_neighbor_frames_for_center(
                    center_path=center_path,
                    video_id=metadata["video_id"],
                    frame_index=metadata["frame_index"],
                    neighbor_radius=req.neighbor_radius,
                    request=request,
                )
            )
        except Exception as error:
            response_items.append(
                {
                    "frame_ref": center_ref,
                    "status": "error",
                    "error": str(error),
                    "neighbors": [],
                }
            )
            continue

        valid_center_refs.add(center_ref)
        all_neighbor_refs.update(neighbor_refs)

        if cache_key is not None:
            used_cache_keys.add(cache_key)

        response_items.append(
            {
                "frame_ref": center_ref,
                "status": "success",
                "video_id": metadata["video_id"],
                "frame_index": metadata["frame_index"],
                "neighbor_count": len(neighbors),
                "neighbors": neighbors,
            }
        )

    attach_neighbor_refs_to_matching_sessions(
        center_refs=valid_center_refs,
        neighbor_refs=all_neighbor_refs,
    )

    payload = {
        "status": "success",
        "requested_count": len(ordered_refs),
        "successful_count": sum(
            item["status"] == "success"
            for item in response_items
        ),
        "neighbor_radius": req.neighbor_radius,
        "unique_frame_directories": len(used_cache_keys),
        "directory_scan_counts": {
            cache_key: FRAME_DIRECTORY_SCAN_COUNT.get(
                cache_key,
                0,
            )
            for cache_key in sorted(used_cache_keys)
        },
        "results": response_items,
        "data": response_items,
    }
    return payload


# ------------------------------------------------------------
# 6. OCR TEST ENDPOINT
# ------------------------------------------------------------

@app.get("/api/v1/ocr/stats")
async def ocr_stats():
    return get_ocr_database_stats()


@app.post("/api/v1/search/ocr")
def search_ocr_only(
    req: OCRSearchRequest,
    request: Request,
):
    started_at = perf_counter()

    try:
        with RETRIEVAL_EXECUTION_LOCK:
            ocr_results = search_ocr_sqlite(
                req.query,
                top_k=req.top_k,
            )

        fused = compute_weighted_rrf(
            {"ocr_sqlite": ocr_results},
            weights={"ocr_sqlite": 1.75},
            k=60,
        )

        final_results = finalize_fused_results(
            fused,
            max_results=req.display_result_count,
        )
    except Exception as error:
        raise HTTPException(
            status_code=500,
            detail=f"OCR retrieval failed: {error}",
        ) from error

    results_json, warnings, _, _ = (
        serialize_result_dataframe(
            final_results,
            request,
        )
    )

    return {
        "status": "success",
        "query": normalize_ocr_query(req.query),
        "processing_time_ms": round(
            (perf_counter() - started_at) * 1000
        ),
        "total_results": len(results_json),
        "ocr": get_ocr_database_stats(),
        "warnings": warnings,
        "results": results_json,
        "data": results_json,
    }


# ------------------------------------------------------------
# 7. ASR TEST ENDPOINT
# ------------------------------------------------------------

@app.get("/api/v1/asr/stats")
async def asr_stats():
    return get_asr_stats()


@app.post("/api/v1/search/asr")
def search_asr_only(
    req: ASRSearchRequest,
    request: Request,
):
    started_at = perf_counter()

    try:
        with RETRIEVAL_EXECUTION_LOCK:
            asr_results = search_asr_dataframe(
                req.query,
                top_k=req.top_k,
            )

        fused = compute_weighted_rrf(
            {"asr_hybrid": asr_results},
            weights=FUSION_WEIGHTS,
            k=60,
        )

        final_results = finalize_fused_results(
            fused,
            max_results=req.display_result_count,
        )
    except Exception as error:
        raise HTTPException(
            status_code=500,
            detail=f"ASR retrieval failed: {error}",
        ) from error

    results_json, warnings, _, _ = (
        serialize_result_dataframe(
            final_results,
            request,
        )
    )

    return {
        "status": "success",
        "query": normalize_asr_query(req.query),
        "processing_time_ms": round(
            (perf_counter() - started_at) * 1000
        ),
        "total_results": len(results_json),
        "asr": get_asr_stats(),
        "warnings": warnings,
        "results": results_json,
        "data": results_json,
    }


# ------------------------------------------------------------
# 8. MULTIMODAL SEARCH ENDPOINT
# ------------------------------------------------------------

@app.post(
    "/api/v1/search",
    name="search_video_v1",
)
@app.post(
    "/api/search",
    name="search_video_legacy",
)
def search_video(
    req: SearchRequest,
    request: Request,
):
    """
    Multimodal search WITHOUT query reconstruction.

    Uses run_multimodal_search(), which is the active retrieval helper
    defined earlier in this notebook.

    A lightweight search session is still kept because the temporal
    localization / verification endpoints need search_id + frame_ref.
    """
    started_at = perf_counter()

    original_query = req.resolved_query()
    visual_query = req.resolved_visual_query()
    ocr_query = req.resolved_ocr_query()
    asr_query = req.resolved_asr_query()

    if not original_query:
        raise HTTPException(
            status_code=422,
            detail=(
                "Provide either query or at least one non-empty "
                "visual_query, ocr_query or asr_query."
            ),
        )

    try:
        query_mode = detect_query_mode(
            visual_query=visual_query,
            ocr_query=ocr_query,
            asr_query=asr_query,
        )
    except ValueError as error:
        raise HTTPException(
            status_code=422,
            detail=str(error),
        ) from error

    visual_top_k = req.resolved_top_k()
    ocr_top_k = OCR_DEFAULT_TOP_K
    asr_top_k = ASR_CFG.max_rerank_candidates
    display_result_count = req.resolved_result_count()
    visual_models = list(req.options.visual_models)

    print(f"\n[API] Full query: {original_query!r}")
    print(f"[API] Visual query: {visual_query!r}")
    print(f"[API] OCR query: {ocr_query!r}")
    print(f"[API] ASR query: {asr_query!r}")
    print(f"[API] Query mode: {query_mode!r}")
    print(f"[API] Visual models: {visual_models!r}")
    print(
        "[API] Query reconstruction: disabled "
        f"(client_requested={req.options.use_query_rewrite})"
    )

    lazy_neighbors = bool(
        FORCE_LAZY_NEIGHBORS
        or req.options.lazy_neighbors
    )

    include_neighbors_now = bool(
        req.options.include_neighbors
        and not lazy_neighbors
    )

    print(
        f"[API] candidate_k="
        f"visual:{visual_top_k}, "
        f"ocr:{ocr_top_k}, "
        f"asr:{asr_top_k}, "
        f"display_result_count={display_result_count}, "
        f"include_neighbors_requested={req.options.include_neighbors}, "
        f"include_neighbors_effective={include_neighbors_now}, "
        f"lazy_neighbors={lazy_neighbors}, "
        f"neighbor_radius={req.options.neighbor_radius}"
    )

    retrieval_started_at = perf_counter()

    try:
        with RETRIEVAL_EXECUTION_LOCK:
            search_meta, final_results = run_multimodal_search(
                visual_query=visual_query,
                ocr_query=ocr_query,
                asr_query=asr_query,
                query_mode=query_mode,
                visual_models=visual_models,
                visual_top_k=visual_top_k,
                ocr_top_k=ocr_top_k,
                asr_top_k=asr_top_k,
                max_results=display_result_count,
            )

    except ValueError as error:
        raise HTTPException(
            status_code=422,
            detail=str(error),
        ) from error

    except Exception as error:
        raise HTTPException(
            status_code=500,
            detail=f"Initial retrieval failed: {error}",
        ) from error

    retrieval_time_ms = round(
        (perf_counter() - retrieval_started_at) * 1000.0,
        2,
    )

    serialization_started_at = perf_counter()

    (
        results_json,
        warnings,
        frame_refs,
        neighbor_summary,
    ) = serialize_result_dataframe(
        final_results,
        request,
        include_neighbors=include_neighbors_now,
        neighbor_radius=req.options.neighbor_radius,
    )

    serialization_time_ms = round(
        (perf_counter() - serialization_started_at) * 1000.0,
        2,
    )

    # --------------------------------------------------------
    # IMPORTANT FOR TEMPORAL LOCALIZATION / VERIFICATION
    # --------------------------------------------------------
    search_id = str(uuid4())

    SEARCH_SESSION_CACHE[search_id] = {
        "original_query": original_query,
        "visual_query": visual_query,
        "ocr_query": ocr_query,
        "asr_query": asr_query,
        "query_mode": query_mode,
        "visual_models": visual_models,
        "visual_top_k": visual_top_k,
        "ocr_top_k": ocr_top_k,
        "asr_top_k": asr_top_k,
        "current_frame_refs": set(frame_refs),
        "task": req.task,
        "feedback_history": [],
    }

    trim_in_memory_cache(
        SEARCH_SESSION_CACHE,
        MAX_SEARCH_SESSION_CACHE,
    )

    processing_time_ms = round(
        (perf_counter() - started_at) * 1000
    )

    pipeline_timings = search_meta.get(
        "timings",
        {},
    )

    # Frontend compatibility: keep these fields, but explicitly
    # report that no rewrite/reconstruction was applied.
    compatibility_query_state = {
        "original_query": original_query,
        "standalone_visual_query_en": visual_query,
        "rewrite_applied": False,
    }

    payload = {
        "status": "success",
        "search_id": search_id,
        "task": req.task,
        "iteration": 0,
        "processing_time_ms": processing_time_ms,

        "timings": {
            "retrieval_ms": retrieval_time_ms,
            "serialization_ms": serialization_time_ms,
            "pipeline": pipeline_timings,
            "visual": pipeline_timings.get(
                "visual_detail",
                {},
            ),
            "total_ms": processing_time_ms,
        },

        "original_query": original_query,
        "query_mode": query_mode,
        "visual_query": visual_query,
        "ocr_query": ocr_query,
        "asr_query": asr_query,

        "reconstructed_query": visual_query,
        "query_rewrite_applied": False,
        "query_state": compatibility_query_state,

        "total_results": len(results_json),
        "indexed_videos": len(VIDEO_INDEX),

        "neighbors_included": include_neighbors_now,
        "neighbors_lazy": lazy_neighbors,
        "neighbor_radius": req.options.neighbor_radius,
        "neighbor_summary": neighbor_summary,

        "warnings": warnings,
        "results": results_json,
        "data": results_json,
    }

    print(
        f"[API TIMING] retrieval={retrieval_time_ms:.1f}ms | "
        f"serialize={serialization_time_ms:.1f}ms | "
        f"total={processing_time_ms}ms"
    )

    print(
        f"[API] Returned {len(results_json)} result(s); "
        f"search_id={search_id}."
    )

    return payload


@app.post("/api/v1/search/feedback")
@app.post("/api/search/feedback")
def feedback_search(
    req: FeedbackSearchRequest,
    request: Request,
):
    """
    Legacy endpoint retained for an explicit response.

    The current pipeline is:
        search -> temporal localization -> selective verification

    The old feedback/query-reconstruction retrieval module is not active.
    """
    raise HTTPException(
        status_code=410,
        detail=(
            "Legacy feedback/query-reconstruction retrieval is disabled "
            "in this notebook. Use the normal search results and the "
            "temporal localization / verification endpoints instead."
        ),
    )


# ============================================================
# 9. ORIGINAL-INTENT ASYNC VERIFICATION PIPELINE
#
# Design goals:
#   1) Search stays interactive.
#   2) User explicitly supplies ORIGINAL USER INTENT.
#   3) Selected candidates enter an asynchronous pending queue.
#   4) Evidence preparation runs separately from Qwen inference.
#   5) Temporal visual evidence + OCR + ASR are gathered automatically.
#   6) GPU 0 is used only briefly for local temporal scoring.
#   7) GPU 1 runs exactly one Qwen verifier at a time to avoid OOM.
#
# Pipeline:
#   submit -> pending
#          -> localizing (GPU0, short)
#          -> collecting_metadata (CPU/SQLite)
#          -> queued_for_vlm
#          -> verifying (GPU1)
#          -> completed
# ============================================================

from concurrent.futures import ThreadPoolExecutor
from bisect import bisect_left
import copy


# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

MAX_TEMPORAL_SEGMENT_CACHE = 192
MAX_VERIFICATION_JOB_CACHE = 256

DEFAULT_VERIFY_MAX_FRAMES = 3
MAX_VERIFY_MAX_FRAMES = 5

# One evidence worker is the safest default for interactive search:
# it pipelines the NEXT candidate while GPU 1 verifies the current one,
# but does not create a backlog of GPU-0 localization workers competing
# with new search requests. Raise AIC_VERIFY_EVIDENCE_WORKERS only after
# profiling on a larger GPU host.
VERIFICATION_EVIDENCE_WORKERS = max(
    1,
    min(
        int(
            os.getenv(
                "AIC_VERIFY_EVIDENCE_WORKERS",
                "1",
            )
        ),
        2,
    ),
)

# Qwen verifier is isolated on GPU 1.
VERIFICATION_EXECUTION_LOCK = threading.Lock()

TEMPORAL_SEGMENT_CACHE: dict[
    str,
    dict[str, Any],
] = {}

TEMPORAL_SEGMENT_CACHE_LOCK = (
    threading.Lock()
)

VERIFICATION_JOB_CACHE: dict[
    str,
    dict[str, Any],
] = {}

VERIFICATION_JOB_CACHE_LOCK = (
    threading.Lock()
)

# Prevent duplicate clicks from launching the same expensive job.
VERIFICATION_DEDUP_CACHE: dict[
    str,
    str,
] = {}

# Lazy temporal transcript index.
_ASR_TEMPORAL_INDEX_LOCK = (
    threading.Lock()
)

_ASR_TEMPORAL_INDEX_SIGNATURE = None

_ASR_TEMPORAL_INDEX: dict[
    str,
    dict[str, Any],
] = {}

# engine_v2 uses one read-only SQLite connection. Serialize only the
# very short metadata queries so two evidence workers never share a
# connection operation at exactly the same time.
_OCR_EVIDENCE_LOCK = threading.Lock()


# ------------------------------------------------------------
# RESTART-SAFE EXECUTORS
# ------------------------------------------------------------

for _executor_name in (
    "_AIC_VERIFICATION_EXECUTOR",
    "_AIC_EVIDENCE_EXECUTOR",
    "_AIC_VLM_EXECUTOR",
):

    _old_executor = globals().get(
        _executor_name
    )

    if _old_executor is not None:

        try:
            _old_executor.shutdown(
                wait=False,
                cancel_futures=True,
            )
        except Exception:
            pass


_AIC_EVIDENCE_EXECUTOR = (
    ThreadPoolExecutor(
        max_workers=
            VERIFICATION_EVIDENCE_WORKERS,

        thread_name_prefix=
            "aic-evidence",
    )
)


_AIC_VLM_EXECUTOR = (
    ThreadPoolExecutor(
        max_workers=1,

        thread_name_prefix=
            "aic-vlm-verify",
    )
)


# ============================================================
# REQUEST CONTRACTS
# ============================================================

class TemporalLocalizeRequest(
    BaseModel
):

    search_id: str = Field(
        min_length=1
    )

    frame_ref: str = Field(
        min_length=1
    )

    # Optional explicit localization query.
    # If omitted, backend reuses the search visual query.
    query_override: str | None = None

    radius_frames: int = Field(
        default=24,
        ge=4,
        le=64,
    )

    max_seconds_from_anchor: float = Field(
        default=24.0,
        ge=2.0,
        le=60.0,
    )

    max_shots_each_side: int = Field(
        default=3,
        ge=0,
        le=10,
    )

    max_duration: float = Field(
        default=20.0,
        ge=2.0,
        le=60.0,
    )


class VerificationSubmitRequest(
    TemporalLocalizeRequest
):

    # This field is intentionally separate from the search query.
    original_query: str = Field(
        min_length=3,
        max_length=4000,
    )

    max_frames: int = Field(
        default=
            DEFAULT_VERIFY_MAX_FRAMES,

        ge=2,
        le=MAX_VERIFY_MAX_FRAMES,
    )

    include_ocr: bool = True
    include_asr: bool = True


class VerificationBatchSubmitRequest(
    BaseModel
):

    search_id: str = Field(
        min_length=1
    )

    original_query: str = Field(
        min_length=3,
        max_length=4000,
    )

    frame_refs: list[str] = Field(
        min_length=1,
        max_length=20,
    )

    max_frames: int = Field(
        default=
            DEFAULT_VERIFY_MAX_FRAMES,

        ge=2,
        le=MAX_VERIFY_MAX_FRAMES,
    )

    include_ocr: bool = True
    include_asr: bool = True


# ============================================================
# RUNTIME CHECKS
# ============================================================

def ensure_temporal_localizer_ready():

    fn = globals().get(
        "localize_adaptive_segment_v2"
    )

    if not callable(fn):

        raise HTTPException(
            status_code=503,
            detail=(
                "Temporal localizer is not loaded. "
                "Run the temporal localization cells "
                "before starting the backend."
            ),
        )


def ensure_vlm_verifier_ready():

    fn = globals().get(
        "verify_segment_vlm_local"
    )

    if not callable(fn):

        raise HTTPException(
            status_code=503,
            detail=(
                "verify_segment_vlm_local() "
                "is not loaded."
            ),
        )

    if globals().get(
        "qwen_vlm"
    ) is None:

        raise HTTPException(
            status_code=503,
            detail=(
                "Qwen verifier is not loaded."
            ),
        )


# ============================================================
# SEARCH / ANCHOR CONTEXT
# ============================================================

def _session_search_query(
    session: dict[str, Any],
) -> str:

    visual = str(
        session.get(
            "visual_query",
            "",
        )
        or ""
    ).strip()

    ocr = str(
        session.get(
            "ocr_query",
            "",
        )
        or ""
    ).strip()

    asr = str(
        session.get(
            "asr_query",
            "",
        )
        or ""
    ).strip()

    parts = []

    if visual:
        parts.append(
            f"Visual: {visual}"
        )

    if ocr:
        parts.append(
            f"OCR: {ocr}"
        )

    if asr:
        parts.append(
            f"ASR: {asr}"
        )

    if parts:
        return " | ".join(
            parts
        )

    return str(
        session.get(
            "original_query",
            "",
        )
        or ""
    ).strip()


def _localization_query(
    session: dict[str, Any],
    original_query: str,
    query_override: str | None = None,
) -> str:
    """
    Evidence localization stays search-conditioned by default.

    Rationale:
      q_search was explicitly refined to retrieve this candidate, so it is
      the better cue for identifying the candidate's local event region.
      q_original is preserved unchanged and is used ONLY as the semantic
      verification target.

    This separation prevents the original-query retrieval weakness from
    being reintroduced during temporal localization.
    """

    if (
        query_override
        and query_override.strip()
    ):
        return (
            query_override.strip()
        )

    visual_query = str(
        session.get(
            "visual_query",
            "",
        )
        or ""
    ).strip()

    if visual_query:
        return visual_query

    search_query = str(
        session.get(
            "original_query",
            "",
        )
        or ""
    ).strip()

    if search_query:
        return search_query

    return str(
        original_query or ""
    ).strip()


def resolve_anchor_context(
    search_id: str,
    frame_ref: str,
) -> tuple[
    dict[str, Any],
    str,
]:

    session = SEARCH_SESSION_CACHE.get(
        search_id
    )

    if session is None:

        raise HTTPException(
            status_code=404,
            detail=(
                "Search session was not found. "
                "Run a new search."
            ),
        )

    if (
        frame_ref
        not in session.get(
            "current_frame_refs",
            set(),
        )
    ):

        raise HTTPException(
            status_code=400,
            detail=(
                "frame_ref does not belong "
                "to this search session."
            ),
        )

    anchor_path = (
        FRAME_REFERENCE_CACHE.get(
            frame_ref
        )
    )

    if not anchor_path:

        raise HTTPException(
            status_code=404,
            detail=(
                "Frame reference expired "
                "or was not registered."
            ),
        )

    if not Path(
        anchor_path
    ).is_file():

        raise HTTPException(
            status_code=404,
            detail=(
                "Anchor frame file "
                "does not exist."
            ),
        )

    return (
        session,
        anchor_path,
    )


# ============================================================
# JSON HELPERS
# ============================================================

def _to_json_value(value):

    if value is None:
        return None

    if isinstance(
        value,
        (
            str,
            int,
            float,
            bool,
        ),
    ):
        return value

    if hasattr(
        value,
        "item",
    ):

        try:
            return value.item()
        except Exception:
            pass

    if isinstance(
        value,
        dict,
    ):

        return {
            str(k):
                _to_json_value(v)

            for k, v
            in value.items()
        }

    if isinstance(
        value,
        (list, tuple),
    ):

        return [
            _to_json_value(v)
            for v in value
        ]

    return str(value)


def _first_frame_score(
    frame,
) -> float | None:

    for field in (
        "visual_score_smooth",
        "consensus_smooth",
        "visual_score",
        "score",
    ):

        value = getattr(
            frame,
            field,
            None,
        )

        if value is None:
            continue

        try:
            return float(
                value
            )
        except Exception:
            continue

    return None


# ============================================================
# SEGMENT SERIALIZATION
# ============================================================

def serialize_temporal_segment_internal(
    segment,
    expansion: dict[str, Any],
) -> dict[str, Any]:

    frames_json = []

    for frame in segment.frames:

        image_path = str(
            frame.image_path
        )

        frame_ref = (
            register_frame_reference(
                image_path
            )
        )

        frames_json.append(
            {
                "frame_ref":
                    frame_ref,

                "timestamp_seconds":
                    float(
                        frame.timestamp
                    ),

                "shot_id":
                    (
                        int(
                            frame.shot_id
                        )
                        if getattr(
                            frame,
                            "shot_id",
                            None,
                        )
                        is not None
                        else None
                    ),

                "is_anchor":
                    bool(
                        getattr(
                            frame,
                            "is_anchor",
                            False,
                        )
                    ),

                "score":
                    _first_frame_score(
                        frame
                    ),
            }
        )

    frames_json.sort(
        key=lambda item:
            item[
                "timestamp_seconds"
            ]
    )

    return {
        "video_id":
            str(
                segment.video_id
            ),

        "anchor_timestamp":
            float(
                segment.anchor_timestamp
            ),

        "start_time":
            float(
                segment.start_time
            ),

        "end_time":
            float(
                segment.end_time
            ),

        "duration":
            float(
                segment.end_time
                - segment.start_time
            ),

        "anchor_shot":
            _to_json_value(
                expansion.get(
                    "anchor_shot"
                )
            ),

        "left_shot":
            _to_json_value(
                expansion.get(
                    "left_shot"
                )
            ),

        "right_shot":
            _to_json_value(
                expansion.get(
                    "right_shot"
                )
            ),

        "threshold":
            _to_json_value(
                expansion.get(
                    "threshold"
                )
            ),

        "frame_count":
            len(
                frames_json
            ),

        "frames":
            frames_json,
    }


def publicize_temporal_segment(
    segment_payload: dict[str, Any],
    request: Request,
) -> dict[str, Any]:

    payload = copy.deepcopy(
        segment_payload
    )

    video_id = payload[
        "video_id"
    ]

    video_path = resolve_video(
        video_id
    )

    payload[
        "video_available"
    ] = (
        video_path is not None
    )

    payload[
        "video_url"
    ] = (
        str(
            request.url_for(
                "stream_video",
                video_id=video_id,
            )
        )
        if video_path is not None
        else None
    )

    for frame in payload[
        "frames"
    ]:

        frame_ref = frame[
            "frame_ref"
        ]

        local_path = (
            FRAME_REFERENCE_CACHE.get(
                frame_ref
            )
        )

        frame[
            "thumbnail_url"
        ] = thumbnail_public_url(
            request,
            frame_ref,
        )

        frame[
            "image_url"
        ] = (
            image_public_url(
                request,
                local_path,
            )
            if local_path
            else None
        )

    return payload


# ============================================================
# TEMPORAL CACHE
# ============================================================

def temporal_cache_key(
    search_id: str,
    frame_ref: str,
    query: str,
    radius_frames: int,
    max_seconds_from_anchor: float,
    max_shots_each_side: int,
    max_duration: float,
    models: tuple[str, ...],
) -> str:

    query_hash = (
        hashlib.sha1(
            query.encode(
                "utf-8"
            )
        )
        .hexdigest()[:12]
    )

    model_key = ",".join(
        models
    )

    return (
        f"{search_id}|"
        f"{frame_ref}|"
        f"{query_hash}|"
        f"{radius_frames}|"
        f"{max_seconds_from_anchor}|"
        f"{max_shots_each_side}|"
        f"{max_duration}|"
        f"{model_key}"
    )


def _resolve_localization_models(
    session: dict[str, Any],
) -> tuple[str, ...]:

    configured = (
        session.get(
            "visual_models"
        )
        or []
    )

    valid = [
        str(model)
        for model in configured
        if str(model)
        in globals().get(
            "MODEL_CONFIG",
            {},
        )
    ]

    if valid:
        return tuple(
            dict.fromkeys(
                valid
            )
        )

    fallback = [
        model
        for model in (
            "metaclip",
            "perception",
            "bge",
            "beit3",
        )
        if model
        in globals().get(
            "MODEL_CONFIG",
            {},
        )
    ]

    return tuple(
        fallback
    )


def get_or_localize_temporal_segment(
    *,
    search_id: str,
    frame_ref: str,
    anchor_path: str,
    query: str,
    session: dict[str, Any],
    radius_frames: int,
    max_seconds_from_anchor: float,
    max_shots_each_side: int,
    max_duration: float,
):

    ensure_temporal_localizer_ready()

    models = (
        _resolve_localization_models(
            session
        )
    )

    cache_key = temporal_cache_key(
        search_id,
        frame_ref,
        query,
        radius_frames,
        max_seconds_from_anchor,
        max_shots_each_side,
        max_duration,
        models,
    )

    with TEMPORAL_SEGMENT_CACHE_LOCK:

        cached = (
            TEMPORAL_SEGMENT_CACHE.get(
                cache_key
            )
        )

    if cached is not None:

        return (
            cached,
            True,
        )

    # Short GPU0 stage.
    # It shares the retrieval models, so serialize only this part
    # with normal search to prevent model-state / VRAM contention.
    with RETRIEVAL_EXECUTION_LOCK:

        segment, scores, expansion = (
            localize_adaptive_segment_v2(
                anchor_path=
                    anchor_path,

                query=
                    query,

                radius_frames=
                    radius_frames,

                max_seconds_from_anchor=
                    max_seconds_from_anchor,

                max_shots_each_side=
                    max_shots_each_side,

                max_duration=
                    max_duration,

                models=
                    models,
            )
        )

    payload = (
        serialize_temporal_segment_internal(
            segment,
            expansion,
        )
    )

    entry = {
        "segment":
            segment,

        "expansion":
            expansion,

        "payload":
            payload,

        "models":
            models,
    }

    with TEMPORAL_SEGMENT_CACHE_LOCK:

        TEMPORAL_SEGMENT_CACHE[
            cache_key
        ] = entry

        trim_in_memory_cache(
            TEMPORAL_SEGMENT_CACHE,
            MAX_TEMPORAL_SEGMENT_CACHE,
        )

    return (
        entry,
        False,
    )


# ============================================================
# FAST OCR EVIDENCE LOOKUP
#
# No OCR model is run here.
# We reuse the existing OCR SQLite metadata generated offline.
# ============================================================

def _normalize_evidence_text(
    value: Any,
) -> str:

    return " ".join(
        str(
            value
            if value is not None
            else ""
        ).split()
    ).strip()


def collect_ocr_segment_evidence(
    segment,
    *,
    max_items: int = 10,
) -> list[dict[str, Any]]:
    """
    Resolve OCR metadata for frames already present in the localized segment.

    Fast path:
      one SQLite query using exact frame basenames.

    Fallback:
      a small video/time range query if the exact-name query returns nothing.
    """

    engine = globals().get(
        "engine_v2"
    )

    connection = getattr(
        engine,
        "connection",
        None,
    )

    if connection is None:
        return []

    frame_names = list(
        dict.fromkeys(
            Path(
                str(
                    frame.image_path
                )
            ).name

            for frame
            in segment.frames
        )
    )

    if not frame_names:
        return []

    rows = []

    try:
        for start in range(
            0,
            len(frame_names),
            300,
        ):

            part = frame_names[
                start:start + 300
            ]

            placeholders = ",".join(
                "?"
                for _ in part
            )

            with _OCR_EVIDENCE_LOCK:
                rows.extend(
                    connection.execute(
                        f"""
                        SELECT
                            frame,
                            video_id,
                            timestamp_sec,
                            frame_index,
                            shot_id,
                            frame_text_index,
                            frame_text_recall
                        FROM frames
                        WHERE frame IN ({placeholders})
                        """,
                        part,
                    ).fetchall()
                )

    except Exception as error:

        print(
            "[VERIFY OCR] exact-frame lookup failed:",
            repr(error),
        )

        rows = []

    # Fallback if the OCR DB uses a different frame-name representation.
    if not rows:

        try:
            with _OCR_EVIDENCE_LOCK:
                rows = (
                    connection.execute(
                        """
                        SELECT
                            frame,
                            video_id,
                            timestamp_sec,
                            frame_index,
                            shot_id,
                            frame_text_index,
                            frame_text_recall
                        FROM frames
                        WHERE video_id = ?
                          AND timestamp_sec >= ?
                          AND timestamp_sec <= ?
                        ORDER BY timestamp_sec
                        LIMIT 96
                        """,
                        (
                            str(
                                segment.video_id
                            ),
                            float(
                                segment.start_time
                            )
                            - 0.6,
                            float(
                                segment.end_time
                            )
                            + 0.6,
                        ),
                    ).fetchall()
                )

        except Exception as error:

            print(
                "[VERIFY OCR] temporal fallback failed:",
                repr(error),
            )

            return []

    anchor_name = Path(
        str(
            segment.anchor_path
        )
    ).name

    anchor_time = float(
        segment.anchor_timestamp
    )

    evidence = []
    seen_texts = set()

    for row in rows:

        primary = (
            _normalize_evidence_text(
                row[
                    "frame_text_index"
                ]
            )
        )

        recall = (
            _normalize_evidence_text(
                row[
                    "frame_text_recall"
                ]
            )
        )

        text = (
            primary
            or recall
        )

        if not text:
            continue

        normalized_key = (
            text.casefold()
        )

        if normalized_key in seen_texts:
            continue

        seen_texts.add(
            normalized_key
        )

        timestamp = float(
            row[
                "timestamp_sec"
            ]
            or 0.0
        )

        frame_name = str(
            row[
                "frame"
            ]
            or ""
        )

        evidence.append(
            {
                "text":
                    text,

                "text_recall":
                    recall
                    if recall
                    and recall != text
                    else None,

                "timestamp":
                    timestamp,

                "frame_index":
                    (
                        int(
                            row[
                                "frame_index"
                            ]
                        )
                        if row[
                            "frame_index"
                        ]
                        is not None
                        else None
                    ),

                "shot_id":
                    (
                        int(
                            row[
                                "shot_id"
                            ]
                        )
                        if row[
                            "shot_id"
                        ]
                        is not None
                        else None
                    ),

                "frame":
                    frame_name,

                "is_anchor":
                    (
                        Path(
                            frame_name
                        ).name
                        == anchor_name
                        or abs(
                            timestamp
                            - anchor_time
                        )
                        <= 0.45
                    ),
            }
        )

    # Anchor OCR first, then chronological evidence.
    evidence.sort(
        key=lambda item: (
            0
            if item[
                "is_anchor"
            ]
            else 1,
            item[
                "timestamp"
            ],
        )
    )

    return evidence[
        :max_items
    ]


# ============================================================
# FAST ASR TEMPORAL LOOKUP
#
# No BGE-M3 / reranker inference is run here.
# We only read the already loaded transcript segments.
# ============================================================

def _ensure_asr_temporal_index():

    global _ASR_TEMPORAL_INDEX
    global _ASR_TEMPORAL_INDEX_SIGNATURE

    segments = globals().get(
        "ASR_SEGMENTS"
    )

    if not isinstance(
        segments,
        list,
    ):
        return

    signature = (
        id(
            segments
        ),
        len(
            segments
        ),
    )

    if (
        _ASR_TEMPORAL_INDEX_SIGNATURE
        == signature
    ):
        return

    with _ASR_TEMPORAL_INDEX_LOCK:

        if (
            _ASR_TEMPORAL_INDEX_SIGNATURE
            == signature
        ):
            return

        by_video: dict[
            str,
            list[Any],
        ] = {}

        for segment in segments:

            video_id = str(
                getattr(
                    segment,
                    "video_id",
                    "",
                )
            ).upper()

            if not video_id:
                continue

            by_video.setdefault(
                video_id,
                [],
            ).append(
                segment
            )

        new_index = {}

        for (
            video_id,
            video_segments,
        ) in by_video.items():

            ordered = sorted(
                video_segments,
                key=lambda item:
                    (
                        float(
                            getattr(
                                item,
                                "start_time",
                                0.0,
                            )
                        ),
                        float(
                            getattr(
                                item,
                                "end_time",
                                0.0,
                            )
                        ),
                    ),
            )

            new_index[
                video_id
            ] = {
                "segments":
                    ordered,

                "starts":
                    [
                        float(
                            getattr(
                                item,
                                "start_time",
                                0.0,
                            )
                        )
                        for item
                        in ordered
                    ],
            }

        _ASR_TEMPORAL_INDEX = (
            new_index
        )

        _ASR_TEMPORAL_INDEX_SIGNATURE = (
            signature
        )


def collect_asr_segment_evidence(
    segment,
    *,
    margin_seconds: float = 2.5,
    max_items: int = 8,
) -> list[dict[str, Any]]:

    _ensure_asr_temporal_index()

    video_id = str(
        segment.video_id
    ).upper()

    entry = (
        _ASR_TEMPORAL_INDEX.get(
            video_id
        )
    )

    if not entry:
        return []

    window_start = (
        float(
            segment.start_time
        )
        - margin_seconds
    )

    window_end = (
        float(
            segment.end_time
        )
        + margin_seconds
    )

    starts = entry[
        "starts"
    ]

    segments = entry[
        "segments"
    ]

    # Start slightly before the first candidate start so a transcript
    # that begins earlier but overlaps the window is still captured.
    position = max(
        0,
        bisect_left(
            starts,
            window_start,
        )
        - 3,
    )

    evidence = []
    seen = set()

    for transcript in (
        segments[
            position:
        ]
    ):

        start_time = float(
            getattr(
                transcript,
                "start_time",
                0.0,
            )
        )

        if (
            start_time
            > window_end
        ):
            break

        end_time = float(
            getattr(
                transcript,
                "end_time",
                start_time,
            )
        )

        if (
            end_time
            < window_start
        ):
            continue

        text = (
            _normalize_evidence_text(
                getattr(
                    transcript,
                    "text",
                    "",
                )
            )
        )

        if not text:
            continue

        key = (
            round(
                start_time,
                2,
            ),
            round(
                end_time,
                2,
            ),
            text.casefold(),
        )

        if key in seen:
            continue

        seen.add(
            key
        )

        evidence.append(
            {
                "start_time":
                    start_time,

                "end_time":
                    end_time,

                "text":
                    text,

                "segment_index":
                    int(
                        getattr(
                            transcript,
                            "index",
                            -1,
                        )
                    ),
            }
        )

        if (
            len(
                evidence
            )
            >= max_items
        ):
            break

    return evidence


def collect_multimodal_verification_evidence(
    segment,
    *,
    include_ocr: bool,
    include_asr: bool,
) -> dict[str, Any]:

    started_at = (
        perf_counter()
    )

    ocr_evidence = (
        collect_ocr_segment_evidence(
            segment
        )
        if include_ocr
        else []
    )

    asr_evidence = (
        collect_asr_segment_evidence(
            segment
        )
        if include_asr
        else []
    )

    return {
        "ocr":
            ocr_evidence,

        "asr":
            asr_evidence,

        "collection_time_ms":
            round(
                (
                    perf_counter()
                    - started_at
                )
                * 1000.0,
                2,
            ),
    }


# ============================================================
# ENDPOINT:
# TEMPORAL LOCALIZATION ONLY
#
# This remains useful for diagnostics. It is NOT required before
# submitting a verification job.
# ============================================================

@app.post(
    "/api/v1/temporal/localize",
    name="localize_temporal_candidate",
)
def localize_temporal_candidate(
    req: TemporalLocalizeRequest,
    request: Request,
):

    started_at = perf_counter()

    (
        session,
        anchor_path,
    ) = resolve_anchor_context(
        search_id=
            req.search_id,

        frame_ref=
            req.frame_ref,
    )

    # Diagnostic endpoint has no separate original intent field.
    # Use override first, then current visual search query.
    query = (
        req.query_override.strip()
        if (
            req.query_override
            and req.query_override.strip()
        )
        else str(
            session.get(
                "visual_query",
                "",
            )
            or session.get(
                "original_query",
                "",
            )
        ).strip()
    )

    try:

        entry, cache_hit = (
            get_or_localize_temporal_segment(
                search_id=
                    req.search_id,

                frame_ref=
                    req.frame_ref,

                anchor_path=
                    anchor_path,

                query=
                    query,

                session=
                    session,

                radius_frames=
                    req.radius_frames,

                max_seconds_from_anchor=
                    req.max_seconds_from_anchor,

                max_shots_each_side=
                    req.max_shots_each_side,

                max_duration=
                    req.max_duration,
            )
        )

    except HTTPException:
        raise

    except Exception as error:

        raise HTTPException(
            status_code=500,
            detail=(
                "Temporal localization "
                f"failed: {error}"
            ),
        ) from error

    return {
        "status":
            "success",

        "search_id":
            req.search_id,

        "frame_ref":
            req.frame_ref,

        "query":
            query,

        "cache_hit":
            cache_hit,

        "processing_time_ms":
            round(
                (
                    perf_counter()
                    - started_at
                )
                * 1000.0,
                2,
            ),

        "segment":
            publicize_temporal_segment(
                entry[
                    "payload"
                ],
                request,
            ),
    }


# ============================================================
# JOB HELPERS
# ============================================================

def update_verification_job(
    job_id: str,
    **updates,
):

    with VERIFICATION_JOB_CACHE_LOCK:

        job = (
            VERIFICATION_JOB_CACHE.get(
                job_id
            )
        )

        if job is not None:

            job.update(
                updates
            )


def _verification_dedup_key(
    *,
    search_id: str,
    frame_ref: str,
    original_query: str,
    max_frames: int,
    include_ocr: bool,
    include_asr: bool,
) -> str:

    normalized_original = (
        " ".join(
            original_query.split()
        )
        .casefold()
    )

    payload = (
        f"{search_id}|"
        f"{frame_ref}|"
        f"{normalized_original}|"
        f"{max_frames}|"
        f"{int(include_ocr)}|"
        f"{int(include_asr)}"
    )

    return (
        hashlib.sha1(
            payload.encode(
                "utf-8"
            )
        ).hexdigest()
    )


def _queue_vlm_stage(
    *,
    job_id: str,
    segment,
    original_query: str,
    search_query: str,
    evidence_context: dict[str, Any],
    max_frames: int,
):

    update_verification_job(
        job_id,
        status="pending",
        stage="queued_for_vlm",
        evidence_ready_at=
            time.time(),
    )

    _AIC_VLM_EXECUTOR.submit(
        run_vlm_verification_stage,

        job_id=
            job_id,

        segment=
            segment,

        original_query=
            original_query,

        search_query=
            search_query,

        evidence_context=
            evidence_context,

        max_frames=
            max_frames,
    )


def prepare_verification_evidence_stage(
    *,
    job_id: str,
    search_id: str,
    frame_ref: str,
    anchor_path: str,
    session: dict[str, Any],
    original_query: str,
    localization_query: str,
    radius_frames: int,
    max_seconds_from_anchor: float,
    max_shots_each_side: int,
    max_duration: float,
    max_frames: int,
    include_ocr: bool,
    include_asr: bool,
):

    try:

        update_verification_job(
            job_id,
            status="running",
            stage="localizing",
            started_at=
                time.time(),
        )

        entry, cache_hit = (
            get_or_localize_temporal_segment(
                search_id=
                    search_id,

                frame_ref=
                    frame_ref,

                anchor_path=
                    anchor_path,

                query=
                    localization_query,

                session=
                    session,

                radius_frames=
                    radius_frames,

                max_seconds_from_anchor=
                    max_seconds_from_anchor,

                max_shots_each_side=
                    max_shots_each_side,

                max_duration=
                    max_duration,
            )
        )

        segment = entry[
            "segment"
        ]

        update_verification_job(
            job_id,
            stage=
                "collecting_metadata",

            temporal_cache_hit=
                cache_hit,

            segment=
                entry[
                    "payload"
                ],

            localization_models=
                list(
                    entry.get(
                        "models",
                        [],
                    )
                ),
        )

        evidence_context = (
            collect_multimodal_verification_evidence(
                segment,

                include_ocr=
                    include_ocr,

                include_asr=
                    include_asr,
            )
        )

        update_verification_job(
            job_id,
            evidence_context=
                evidence_context,
        )

        _queue_vlm_stage(
            job_id=
                job_id,

            segment=
                segment,

            original_query=
                original_query,

            search_query=
                _session_search_query(
                    session
                ),

            evidence_context=
                evidence_context,

            max_frames=
                max_frames,
        )

    except Exception as error:

        update_verification_job(
            job_id,
            status="failed",
            stage="failed",
            error=(
                f"{type(error).__name__}: "
                f"{error}"
            ),
            finished_at=
                time.time(),
        )


def run_vlm_verification_stage(
    *,
    job_id: str,
    segment,
    original_query: str,
    search_query: str,
    evidence_context: dict[str, Any],
    max_frames: int,
):

    update_verification_job(
        job_id,
        status="running",
        stage="verifying",
        vlm_started_at=
            time.time(),
    )

    try:

        with VERIFICATION_EXECUTION_LOCK:

            verification = (
                verify_segment_vlm_local(
                    original_query=
                        original_query,

                    search_query=
                        search_query,

                    segment=
                        segment,

                    max_frames=
                        max_frames,

                    ocr_evidence=
                        evidence_context.get(
                            "ocr",
                            [],
                        ),

                    asr_evidence=
                        evidence_context.get(
                            "asr",
                            [],
                        ),
                )
            )

        update_verification_job(
            job_id,
            status="completed",
            stage="completed",
            verification=
                verification,
            finished_at=
                time.time(),
        )

    except Exception as error:

        try:
            import gc

            gc.collect()

            if (
                "torch"
                in globals()
            ):
                torch.cuda.empty_cache()

        except Exception:
            pass

        update_verification_job(
            job_id,
            status="failed",
            stage="failed",
            error=(
                f"{type(error).__name__}: "
                f"{error}"
            ),
            finished_at=
                time.time(),
        )


def _submit_verification_job(
    *,
    req: VerificationSubmitRequest,
) -> dict[str, Any]:

    ensure_temporal_localizer_ready()
    ensure_vlm_verifier_ready()

    (
        session,
        anchor_path,
    ) = resolve_anchor_context(
        search_id=
            req.search_id,

        frame_ref=
            req.frame_ref,
    )

    original_query = (
        " ".join(
            req.original_query.split()
        ).strip()
    )

    localization_query = (
        _localization_query(
            session=
                session,

            original_query=
                original_query,

            query_override=
                req.query_override,
        )
    )

    dedup_key = (
        _verification_dedup_key(
            search_id=
                req.search_id,

            frame_ref=
                req.frame_ref,

            original_query=
                original_query,

            max_frames=
                req.max_frames,

            include_ocr=
                req.include_ocr,

            include_asr=
                req.include_asr,
        )
    )

    with VERIFICATION_JOB_CACHE_LOCK:

        existing_job_id = (
            VERIFICATION_DEDUP_CACHE.get(
                dedup_key
            )
        )

        existing_job = (
            VERIFICATION_JOB_CACHE.get(
                existing_job_id
            )
            if existing_job_id
            else None
        )

        if (
            existing_job
            and existing_job.get(
                "status"
            )
            in {
                "pending",
                "running",
                "completed",
            }
        ):

            return {
                "job":
                    copy.deepcopy(
                        existing_job
                    ),

                "deduplicated":
                    True,
            }

    job_id = (
        uuid4().hex
    )

    job = {
        "job_id":
            job_id,

        "status":
            "pending",

        "stage":
            "queued",

        "search_id":
            req.search_id,

        "frame_ref":
            req.frame_ref,

        "original_query":
            original_query,

        "search_query":
            _session_search_query(
                session
            ),

        "localization_query":
            localization_query,

        "max_frames":
            req.max_frames,

        "include_ocr":
            req.include_ocr,

        "include_asr":
            req.include_asr,

        "submitted_at":
            time.time(),

        "started_at":
            None,

        "evidence_ready_at":
            None,

        "vlm_started_at":
            None,

        "finished_at":
            None,

        "temporal_cache_hit":
            None,

        "localization_models":
            None,

        "segment":
            None,

        "evidence_context":
            None,

        "verification":
            None,

        "error":
            None,

        "dedup_key":
            dedup_key,
    }

    with VERIFICATION_JOB_CACHE_LOCK:

        VERIFICATION_JOB_CACHE[
            job_id
        ] = job

        VERIFICATION_DEDUP_CACHE[
            dedup_key
        ] = job_id

        trim_in_memory_cache(
            VERIFICATION_JOB_CACHE,
            MAX_VERIFICATION_JOB_CACHE,
        )

    # Return immediately. Evidence work continues on background threads.
    _AIC_EVIDENCE_EXECUTOR.submit(
        prepare_verification_evidence_stage,

        job_id=
            job_id,

        search_id=
            req.search_id,

        frame_ref=
            req.frame_ref,

        anchor_path=
            anchor_path,

        session=
            copy.deepcopy(
                session
            ),

        original_query=
            original_query,

        localization_query=
            localization_query,

        radius_frames=
            req.radius_frames,

        max_seconds_from_anchor=
            req.max_seconds_from_anchor,

        max_shots_each_side=
            req.max_shots_each_side,

        max_duration=
            req.max_duration,

        max_frames=
            req.max_frames,

        include_ocr=
            req.include_ocr,

        include_asr=
            req.include_asr,
    )

    return {
        "job":
            copy.deepcopy(
                job
            ),

        "deduplicated":
            False,
    }


# ============================================================
# ENDPOINT:
# SUBMIT ONE VERIFICATION JOB
# ============================================================

@app.post(
    "/api/v1/verification/submit",
    status_code=202,
    name="submit_verification",
)
def submit_verification(
    req: VerificationSubmitRequest,
    request: Request,
):

    submitted = (
        _submit_verification_job(
            req=req
        )
    )

    job = submitted[
        "job"
    ]

    return {
        "status":
            job[
                "status"
            ],

        "job_id":
            job[
                "job_id"
            ],

        "search_id":
            job[
                "search_id"
            ],

        "frame_ref":
            job[
                "frame_ref"
            ],

        "original_query":
            job[
                "original_query"
            ],

        "search_query":
            job[
                "search_query"
            ],

        "stage":
            job[
                "stage"
            ],

        "deduplicated":
            submitted[
                "deduplicated"
            ],

        "poll_url":
            str(
                request.url_for(
                    "get_verification_job",
                    job_id=
                        job[
                            "job_id"
                        ],
                )
            ),
    }


# ============================================================
# ENDPOINT:
# SUBMIT MULTIPLE CANDIDATES IN ONE CALL
# ============================================================

@app.post(
    "/api/v1/verification/submit-batch",
    status_code=202,
    name="submit_verification_batch",
)
def submit_verification_batch(
    req: VerificationBatchSubmitRequest,
    request: Request,
):

    jobs = []

    for frame_ref in list(
        dict.fromkeys(
            value.strip()
            for value in req.frame_refs
            if value.strip()
        )
    ):

        one_req = (
            VerificationSubmitRequest(
                search_id=
                    req.search_id,

                frame_ref=
                    frame_ref,

                original_query=
                    req.original_query,

                max_frames=
                    req.max_frames,

                include_ocr=
                    req.include_ocr,

                include_asr=
                    req.include_asr,
            )
        )

        try:

            submitted = (
                _submit_verification_job(
                    req=one_req
                )
            )

            job = submitted[
                "job"
            ]

            jobs.append(
                {
                    "frame_ref":
                        frame_ref,

                    "job_id":
                        job[
                            "job_id"
                        ],

                    "status":
                        job[
                            "status"
                        ],

                    "stage":
                        job[
                            "stage"
                        ],

                    "deduplicated":
                        submitted[
                            "deduplicated"
                        ],

                    "poll_url":
                        str(
                            request.url_for(
                                "get_verification_job",
                                job_id=
                                    job[
                                        "job_id"
                                    ],
                            )
                        ),
                }
            )

        except HTTPException as error:

            jobs.append(
                {
                    "frame_ref":
                        frame_ref,

                    "status":
                        "failed",

                    "error":
                        str(
                            error.detail
                        ),
                }
            )

    return {
        "status":
            "accepted",

        "search_id":
            req.search_id,

        "original_query":
            req.original_query,

        "jobs":
            jobs,
    }


# ============================================================
# PUBLIC SERIALIZERS
# ============================================================

def publicize_evidence_context(
    evidence_context: dict[str, Any]
    | None,
) -> dict[str, Any] | None:

    if not evidence_context:
        return None

    return {
        "ocr":
            copy.deepcopy(
                evidence_context.get(
                    "ocr",
                    [],
                )
            ),

        "asr":
            copy.deepcopy(
                evidence_context.get(
                    "asr",
                    [],
                )
            ),

        "collection_time_ms":
            evidence_context.get(
                "collection_time_ms"
            ),
    }


def publicize_verification(
    verification: dict[str, Any],
    request: Request,
    include_explanation: bool,
) -> dict[str, Any]:

    source = verification

    result = {
        "status":
            source.get(
                "status",
                "UNKNOWN",
            ),

        "confidence":
            source.get(
                "confidence",
                0.0,
            ),

        "original_query":
            source.get(
                "original_query",
                "",
            ),

        "search_query":
            source.get(
                "search_query",
                "",
            ),

        "claims":
            copy.deepcopy(
                source.get(
                    "claims",
                    [],
                )
            ),

        "modalities_used":
            copy.deepcopy(
                source.get(
                    "modalities_used",
                    {},
                )
            ),

        "ocr_evidence":
            copy.deepcopy(
                source.get(
                    "ocr_evidence",
                    [],
                )
            ),

        "asr_evidence":
            copy.deepcopy(
                source.get(
                    "asr_evidence",
                    [],
                )
            ),

        "num_frames_sent":
            source.get(
                "num_frames_sent",
                0,
            ),

        "latency_ms":
            source.get(
                "latency_ms",
            ),

        "peak_gpu1_memory_gb":
            source.get(
                "peak_gpu1_memory_gb",
            ),

        "evidence":
            [],
    }

    segment = source.get(
        "segment",
        {}
    )

    if segment:

        video_id = str(
            segment.get(
                "video_id",
                "",
            )
        )

        result[
            "segment"
        ] = {
            "video_id":
                video_id,

            "start_time":
                segment.get(
                    "start_time"
                ),

            "end_time":
                segment.get(
                    "end_time"
                ),

            "anchor_time":
                segment.get(
                    "anchor_time"
                ),

            "video_url":
                (
                    str(
                        request.url_for(
                            "stream_video",
                            video_id=
                                video_id,
                        )
                    )
                    if (
                        video_id
                        and resolve_video(
                            video_id
                        )
                        is not None
                    )
                    else None
                ),
        }

    for evidence in (
        source.get(
            "evidence",
            []
        )
        or []
    ):

        evidence_item = {
            "frame_id":
                evidence.get(
                    "frame_id"
                ),

            "timestamp":
                evidence.get(
                    "timestamp"
                ),

            "shot_id":
                evidence.get(
                    "shot_id"
                ),

            "score":
                evidence.get(
                    "score"
                ),
        }

        image_path = (
            evidence.get(
                "image_path"
            )
        )

        if image_path:

            frame_ref = (
                register_frame_reference(
                    image_path
                )
            )

            evidence_item[
                "frame_ref"
            ] = frame_ref

            evidence_item[
                "thumbnail_url"
            ] = (
                thumbnail_public_url(
                    request,
                    frame_ref,
                )
            )

            evidence_item[
                "image_url"
            ] = (
                image_public_url(
                    request,
                    image_path,
                )
            )

        result[
            "evidence"
        ].append(
            evidence_item
        )

    result[
        "has_explanation"
    ] = bool(
        source.get(
            "explanation",
            "",
        )
    )

    if include_explanation:

        result[
            "explanation"
        ] = source.get(
            "explanation",
            "",
        )

    return result


# ============================================================
# ENDPOINT:
# POLL JOB STATUS
# ============================================================

class VerificationStatusBatchRequest(
    BaseModel
):

    job_ids: list[str] = Field(
        min_length=1,
        max_length=50,
    )

    include_explanation: bool = False


def _publicize_verification_job(
    *,
    job_id: str,
    request: Request,
    include_explanation: bool,
) -> dict[str, Any]:

    with VERIFICATION_JOB_CACHE_LOCK:

        cached = (
            VERIFICATION_JOB_CACHE.get(
                job_id
            )
        )

        if cached is None:

            raise HTTPException(
                status_code=404,
                detail=(
                    "Verification job "
                    "was not found or expired."
                ),
            )

        job = copy.deepcopy(
            cached
        )

    response = {
        "job_id":
            job_id,

        "status":
            job[
                "status"
            ],

        "stage":
            job.get(
                "stage"
            ),

        "search_id":
            job.get(
                "search_id"
            ),

        "frame_ref":
            job.get(
                "frame_ref"
            ),

        "original_query":
            job.get(
                "original_query"
            ),

        "search_query":
            job.get(
                "search_query"
            ),

        "localization_query":
            job.get(
                "localization_query"
            ),

        "max_frames":
            job.get(
                "max_frames"
            ),

        "include_ocr":
            job.get(
                "include_ocr"
            ),

        "include_asr":
            job.get(
                "include_asr"
            ),

        "temporal_cache_hit":
            job.get(
                "temporal_cache_hit"
            ),

        "localization_models":
            job.get(
                "localization_models"
            ),

        "submitted_at":
            job.get(
                "submitted_at"
            ),

        "started_at":
            job.get(
                "started_at"
            ),

        "evidence_ready_at":
            job.get(
                "evidence_ready_at"
            ),

        "vlm_started_at":
            job.get(
                "vlm_started_at"
            ),

        "finished_at":
            job.get(
                "finished_at"
            ),

        "error":
            job.get(
                "error"
            ),

        "evidence_context":
            publicize_evidence_context(
                job.get(
                    "evidence_context"
                )
            ),
    }

    if job.get(
        "segment"
    ):

        response[
            "segment"
        ] = (
            publicize_temporal_segment(
                job[
                    "segment"
                ],
                request,
            )
        )

    if (
        job[
            "status"
        ]
        == "completed"
        and job.get(
            "verification"
        )
    ):

        response[
            "verification"
        ] = publicize_verification(
            job[
                "verification"
            ],
            request,
            include_explanation=
                include_explanation,
        )

    return response


@app.get(
    "/api/v1/verification/{job_id}",
    name="get_verification_job",
)
def get_verification_job(
    job_id: str,
    request: Request,
    include_explanation: bool = False,
):

    return (
        _publicize_verification_job(
            job_id=
                job_id,

            request=
                request,

            include_explanation=
                include_explanation,
        )
    )


# One request polls every active verification job.
# This avoids N HTTP requests per second when many candidates are queued.
@app.post(
    "/api/v1/verification/status-batch",
    name="get_verification_jobs_batch",
)
def get_verification_jobs_batch(
    req: VerificationStatusBatchRequest,
    request: Request,
):

    jobs = []

    for job_id in list(
        dict.fromkeys(
            value.strip()
            for value in req.job_ids
            if value.strip()
        )
    ):

        try:

            jobs.append(
                _publicize_verification_job(
                    job_id=
                        job_id,

                    request=
                        request,

                    include_explanation=
                        req.include_explanation,
                )
            )

        except HTTPException as error:

            jobs.append(
                {
                    "job_id":
                        job_id,

                    "status":
                        "failed",

                    "stage":
                        "missing",

                    "error":
                        str(
                            error.detail
                        ),
                }
            )

    return {
        "status":
            "success",

        "jobs":
            jobs,
    }


# ============================================================
# OPTIONAL:
# DELETE/CANCEL A WAITING JOB
#
# Python futures already running cannot be safely killed.
# This endpoint removes completed/failed jobs from the UI cache.
# ============================================================

@app.delete(
    "/api/v1/verification/{job_id}",
    name="delete_verification_job",
)
def delete_verification_job(
    job_id: str,
):

    with VERIFICATION_JOB_CACHE_LOCK:

        job = (
            VERIFICATION_JOB_CACHE.get(
                job_id
            )
        )

        if job is None:

            raise HTTPException(
                status_code=404,
                detail=(
                    "Verification job "
                    "was not found."
                ),
            )

        if job.get(
            "status"
        ) in {
            "pending",
            "running",
        }:

            raise HTTPException(
                status_code=409,
                detail=(
                    "A running verification job "
                    "cannot be deleted safely."
                ),
            )

        removed = (
            VERIFICATION_JOB_CACHE.pop(
                job_id,
                None,
            )
        )

        dedup_key = (
            removed.get(
                "dedup_key"
            )
            if removed
            else None
        )

        if (
            dedup_key
            and VERIFICATION_DEDUP_CACHE.get(
                dedup_key
            )
            == job_id
        ):

            VERIFICATION_DEDUP_CACHE.pop(
                dedup_key,
                None,
            )

    return {
        "status":
            "deleted",

        "job_id":
            job_id,
    }


# ------------------------------------------------------------
# 9. NGROK + UVICORN
# ------------------------------------------------------------

def read_kaggle_secret(name: str) -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient

        value = UserSecretsClient().get_secret(name)
        return value.strip() if value else None
    except Exception:
        value = os.getenv(name)
        return value.strip() if value else None


print("\n" + "=" * 72)
print(
    "VIDEO ROOTS SCANNED:",
    [str(path) for path in VIDEO_ROOTS_SCANNED],
)
print("INDEXED VIDEOS:", len(VIDEO_INDEX))
print("SAMPLE VIDEO IDS:", sorted(VIDEO_INDEX)[:10])
print("ASR:", get_asr_stats())

sample_id = "L21_V002"
sample_path = resolve_video(sample_id)
print(
    f"SAMPLE {sample_id}:",
    str(sample_path) if sample_path else "NOT FOUND",
)
print("=" * 72 + "\n")


SERVER_PORT = int(os.getenv("AIC_API_PORT", "8000"))

# Stop the previous notebook-local server before starting a new one.
_previous_server = globals().get("_AIC_UVICORN_SERVER")
if _previous_server is not None:
    try:
        _previous_server.should_exit = True
        time.sleep(1.0)
    except Exception:
        pass

nest_asyncio.apply()
_server_config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=SERVER_PORT,
    proxy_headers=True,
    forwarded_allow_ips="*",
    log_level="info",
)
_AIC_UVICORN_SERVER = uvicorn.Server(_server_config)
_AIC_UVICORN_THREAD = threading.Thread(
    target=_AIC_UVICORN_SERVER.run,
    name="aic-fastapi",
    daemon=True,
)
_AIC_UVICORN_THREAD.start()
time.sleep(2.0)

print(
    f"Local API: http://127.0.0.1:{SERVER_PORT}/api/v1/health"
)


NGROK_AUTH_TOKEN = ""

NGROK_DOMAIN = os.getenv("NGROK_DOMAIN", "")

# Kaggle Secrets fallback. Never hard-code tunnel credentials in notebooks.
if not NGROK_AUTH_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        NGROK_AUTH_TOKEN = (
            UserSecretsClient()
            .get_secret("NGROK_AUTH_TOKEN")
            .strip()
        )
    except Exception:
        NGROK_AUTH_TOKEN = ""

if NGROK_AUTH_TOKEN:
    try:
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
        ngrok.kill()

        connect_kwargs = {"addr": SERVER_PORT}
        if NGROK_DOMAIN:
            connect_kwargs["domain"] = NGROK_DOMAIN

        tunnel = ngrok.connect(**connect_kwargs)
        public_url = tunnel.public_url

        print("\n" + "=" * 72)
        print("PUBLIC API:", public_url)
        print("FRONTEND .env:")
        print(
            f"VITE_API_BASE_URL={public_url}/api/v1"
        )
        print(
            "HEALTH:",
            f"{public_url}/api/v1/health",
        )
        print("=" * 72 + "\n")
    except Exception as error:
        print(
            "Ngrok could not start. "
            f"The local API is still running: {error}"
        )
else:
    print(
        "NGROK_AUTH_TOKEN is missing. Add it in Kaggle Secrets "
        "to expose the API publicly. NGROK_DOMAIN is optional."
    )

In [ ]:
# ============================================================
# HOT FIX - RESTORE RETRIEVAL DEVICE
# ============================================================

import torch

RETRIEVAL_DEVICE = (
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

device = RETRIEVAL_DEVICE

print(
    "Retrieval device:",
    RETRIEVAL_DEVICE
)

In [ ]:
# ============================================================
# TEXT ENCODERS - FIXED DEVICE
# ============================================================

@torch.no_grad()
def encode_bge_text(query):

    q_emb = bge_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0]

    return q_emb.astype(
        "float32"
    )


@torch.no_grad()
def encode_metaclip_text(query):

    tokens = (
        metaclip_tokenizer(
            [query]
        )
        .to(
            RETRIEVAL_DEVICE
        )
    )

    features = F.normalize(
        metaclip_model.encode_text(
            tokens
        ),
        dim=-1,
    )

    return (
        features[0]
        .detach()
        .cpu()
        .numpy()
        .astype("float32")
    )


@torch.no_grad()
def encode_beit3_text(
    query,
    max_len=64,
):

    ids = sp.encode(
        query,
        out_type=int,
    )

    ids = (
        [BOS_ID]
        + ids[:max_len - 2]
        + [EOS_ID]
    )

    ids = (
        ids
        + [PAD_ID]
        * (
            max_len
            - len(ids)
        )
    )

    input_ids = torch.tensor(
        [ids],
        dtype=torch.long,
        device=RETRIEVAL_DEVICE,
    )

    padding_mask = (
        input_ids
        .eq(PAD_ID)
    )

    output = beit3_model(
        image=None,
        text_description=input_ids,
        padding_mask=padding_mask,
        only_infer=True,
    )

    text_emb = (
        output[1]
        if isinstance(
            output,
            tuple,
        )
        else output.get(
            "language_cls",
            output,
        )
    )

    return (
        F.normalize(
            text_emb,
            dim=-1,
        )[0]
        .detach()
        .cpu()
        .numpy()
        .astype("float32")
    )


@torch.inference_mode()
def encode_perception_text(
    query,
):

    text = (
        perception_tokenizer(
            [query]
        )
        .to(
            RETRIEVAL_DEVICE
        )
    )

    if hasattr(
        perception_model,
        "encode_text",
    ):

        features = (
            perception_model
            .encode_text(
                text
            )
        )

        if isinstance(
            features,
            tuple,
        ):
            features = (
                features[0]
            )

    else:

        h = w = (
            perception_model.image_size
            if isinstance(
                perception_model.image_size,
                int,
            )
            else perception_model.image_size[0]
        )

        dummy_image = torch.zeros(
            (
                1,
                3,
                h,
                w,
            ),
            device=
                RETRIEVAL_DEVICE,
        )

        _, features, _ = (
            perception_model(
                dummy_image,
                text,
            )
        )

    return (
        F.normalize(
            features,
            dim=-1,
        )[0]
        .detach()
        .cpu()
        .numpy()
        .astype("float32")
    )


print(
    "Text encoders fixed."
)